In [ ]:
import os, re, numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 140)


candidates = [
    "/kaggle/input/amazonml/student_resource",
    "/kaggle/input/amazonml",
    "/kaggle/input/student_resource",
    "/kaggle/working/student_resource",   
]
base_path = None
for c in candidates:
    if os.path.exists(os.path.join(c, "dataset", "train.csv")):
        base_path = c; break
assert base_path is not None, "Couldn't find dataset folder — adjust candidates."
print("BASE:", base_path)


In [ ]:
train = pd.read_csv(f"{base_path}/dataset/train.csv")
test  = pd.read_csv(f"{base_path}/dataset/test.csv")

print("train shape:", train.shape)
print("test  shape:",  test.shape)
print("train cols:",   train.columns.tolist())
print("test  cols:",   test.columns.tolist())

# id/target integrity
assert train['sample_id'].is_unique and test['sample_id'].is_unique
assert 'price' in train.columns and 'price' not in test.columns

# nulls
display(train.isna().mean().sort_values(ascending=False).head(10))
display(test.isna().mean().sort_values(ascending=False).head(10))


In [ ]:
train['price'] = train['price'].astype(float)
print(train['price'].describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))
print("Any non-positive prices? ->", (train['price'] <= 0).any())

# quick view of log-scale
train['log_price'] = np.log(train['price'].clip(lower=0.01))
print(train[['price','log_price']].describe())


In [ ]:


import re, collections
tc = train['catalog_content'].astype(str)


tc_norm = tc.str.replace(r'\s+', ' ', regex=True)


NUM   = r"\d+(?:[.,]\d+)?"
UNIT  = r"(?:kg|g|gm|gms|mg|lb|lbs|pound(?:s)?|l|lt|liter|litre|ml|oz|fl\s*oz|ounce(?:s)?|gal|gallon(?:s)?)"
PCS   = r"(?:pc|pcs|piece(?:s)?|count|ct)"
PK    = r"(?:pack|pk|pkt|case|bundle)"

PATTERNS = collections.OrderedDict([
    ("pack_of_N",       rf"\b(?:{PK})\s*(?:of)?\s*{NUM}\b"),                 # pack of 6, pk 6, case 24
    ("set_of_N",        rf"\bset\s*(?:of)?\s*{NUM}\b"),                      # set of 3
    ("N_pcs",           rf"\b{NUM}\s*{PCS}\b"),                              # 12 pcs, 24 ct, 6 count
    ("N_x_qty",         rf"\b{NUM}\s*[x×]\s*{NUM}\s*{UNIT}\b"),              # 2 x 150 ml, 8x150ml
    ("qty_x_N",         rf"\b{NUM}\s*{UNIT}\s*[x×]\s*{NUM}\b"),              # 150 ml x 2
    ("qty_unit",        rf"\b{NUM}\s*{UNIT}\b"),                             # 500 g, 12 oz, 0.5 l, 1,5 kg
    ("value_unit_pair", rf"\bvalue\s*:\s*{NUM}\b[\s\S]{0,40}?\bunit\s*:\s*[a-z\s\.]+"),  # Value: ... Unit: ...
])

print("\n EXPANDED UNIT/IPQ HIT RATES")
compiled = {name: re.compile(pat, flags=re.I) for name, pat in PATTERNS.items()}
for name, creg in compiled.items():
    rate = tc_norm.str.contains(creg, regex=True).mean()*100
    print(f"{name:16s} -> {rate:6.2f}%")

print("\nEXAMPLES PER PATTERN (up to 5")
def show_examples(creg, k=5):
    hits = train[tc_norm.str.contains(creg, regex=True)][['catalog_content','price']].head(k)
    if len(hits)==0:
        print("  (none)")
    else:
        display(hits)

for name, creg in compiled.items():
    print(f"\n-- {name} --")
    show_examples(creg, k=5)

mask = tc_norm.map(lambda s: bool(creg.search(s)))
rate = mask.mean()*100


Building a Baseline Model:

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge



In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    out = np.zeros_like(denom)
    m = denom != 0
    out[m] = np.abs(y_true[m] - y_pred[m]) / denom[m]
    return out.mean()*100



In [ ]:
def clean_text(s):
    s = str(s).lower()
    s = re.sub(r'https?://\S+',' ', s)
    s = s.replace('|',' ').replace('/',' ')
    s = re.sub(r'[^a-z0-9\.\-\+\%\s]',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return s



In [ ]:
train['text'] = train['catalog_content'].fillna('').map(clean_text)
test['text']  = test['catalog_content'].fillna('').map(clean_text)

In [ ]:
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(1,3),
                        max_features=250_000, min_df=3)

In [ ]:
X_tr = tfidf.fit_transform(train['text'])  
X_te = tfidf.transform(test['text'])

In [ ]:
y = train['price'].values.astype(float)
y_log = np.log(np.clip(y, 0.01, None))



In [ ]:

bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_log = np.zeros(len(train))
te_log  = np.zeros(len(test))



In [ ]:
for f,(tr,va) in enumerate(skf.split(X_tr, bins)):
    m = Ridge(alpha=1.0, solver="lsqr", tol=1e-3, random_state=42)
    m.fit(X_tr[tr], y_log[tr])
    oof_log[va] = m.predict(X_tr[va])
    te_log += m.predict(X_te)/skf.n_splits
    fold_smape = smape(y[va], np.exp(oof_log[va]).clip(0.01))
    print(f"Fold {f} SMAPE: {fold_smape:.3f}%")




In [ ]:
cv = smape(y, np.exp(oof_log).clip(0.01))
print("CV SMAPE:", round(cv,3), "%")



In [ ]:
import numpy as np, pandas as pd

pred_oof = np.exp(oof_log).clip(0.01)
den = (np.abs(train['price']) + np.abs(pred_oof)) / 2.0
train['smape_err'] = (np.abs(train['price'] - pred_oof) / den) * 100

# by price decile
train['price_decile'] = pd.qcut(train['price'], 10, labels=False, duplicates='drop')
print(train.groupby('price_decile')['smape_err'].mean().round(2))

# quick view of worst 20 errors
display(train[['catalog_content','price','smape_err']].sort_values('smape_err', ascending=False).head(20))


In [ ]:

import re, numpy as np, pandas as pd

# Conversions
GRAM_PER_OUNCE = 28.349523125
ML_PER_FLOZ    = 29.5735295625

MASS_UNITS = {
    'g': 1.0, 'gram': 1.0, 'grams': 1.0, 'gm': 1.0,
    'kg': 1000.0,
    'oz': GRAM_PER_OUNCE, 'ounce': GRAM_PER_OUNCE, 'ounces': GRAM_PER_OUNCE,
}
VOL_UNITS  = {
    'ml': 1.0,
    'l': 1000.0, 'lt': 1000.0, 'ltr': 1000.0, 'liter': 1000.0, 'litre': 1000.0,
    'fl oz': ML_PER_FLOZ, 'floz': ML_PER_FLOZ, 'fluid ounce': ML_PER_FLOZ, 'fluid ounces': ML_PER_FLOZ,
}

ITEM_WORDS = r'(?:bottles?|cans?|bags?|pouches?|bars?|cups?|sachets?|packets?|boxes?|tubes?|jars?)'

def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = s.replace('×', 'x')
    s = re.sub(r'fluid\s*oz', 'fl oz', s)
    s = re.sub(r'fl\.?\s*oz\.?', 'fl oz', s)
    s = re.sub(r'(\d)([a-z])', r'\1 \2', s)  # "16.9oz" -> "16.9 oz"
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def parse_pack_counts(s: str) -> dict:
    pack = 1
    case = 0

    # "pack of 20", "pack of20", "set of 6"
    m = re.search(r'(?:pack|set)\s*of\s*(\d+)', s)
    if m: pack = max(pack, int(m.group(1)))

    # "55 pcs", "55 pieces"
    m = re.search(r'(\d+)\s*(?:pcs|piece|pieces?)\b', s)
    if m: pack = max(pack, int(m.group(1)))

    # "288 per case", "288 / case"
    m = re.search(r'(\d+)\s*(?:per\s*case|/ ?case)\b', s)
    if m: case = max(case, int(m.group(1)))

    # "<N> bottles per case"
    m = re.search(r'(\d+)\s*' + ITEM_WORDS + r'\s*(?:per\s*case|/ ?case)\b', s)
    if m: case = max(case, int(m.group(1)))

    # "pallet of 84 cases" (rare but present)
    m = re.search(r'pallet\s*of\s*(\d+)\s*cases', s)
    if m:
        pallet_cases = int(m.group(1))
        case = max(case, pallet_cases)  # keep at least that many per pallet mention

    return {'pack_count': pack, 'case_count': case}

def parse_quantity(s: str) -> dict:
    
    qty_each_mass_g = np.nan
    qty_each_vol_ml = np.nan
    total_units = np.nan

    # Pattern A: "N x Q unit"  e.g., "24 x 16.9 oz", "2x250 ml"
    m = re.search(r'(\d+)\s*x\s*(\d+\.?\d*)\s*(fl oz|oz|ounce|ounces|kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre)\b', s)
    if m:
        n = int(m.group(1))
        val = float(m.group(2))
        unit = m.group(3)
        total_units = float(n)
        unit_norm = unit
        if unit_norm in VOL_UNITS:
            qty_each_vol_ml = val * VOL_UNITS[unit_norm]
        elif unit_norm in MASS_UNITS:
            qty_each_mass_g = val * MASS_UNITS[unit_norm]

    # Pattern B: single quantity "Q unit" (keep the first if none found yet)
    if np.isnan(qty_each_mass_g) and np.isnan(qty_each_vol_ml):
        m = re.search(r'(\d+\.?\d*)\s*(fl oz|oz|ounce|ounces|kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre)\b', s)
        if m:
            val = float(m.group(1))
            unit = m.group(2)
            if unit in VOL_UNITS:
                qty_each_vol_ml = val * VOL_UNITS[unit]
            elif unit in MASS_UNITS:
                qty_each_mass_g = val * MASS_UNITS[unit]

    # Pattern C: "<N> <item_word> of <Q unit>" or "<N> bottles 16.9 oz"
    if np.isnan(total_units):
        m = re.search(r'(\d+)\s*' + ITEM_WORDS + r'\b', s)
        if m:
            total_units = float(m.group(1))
            # try to find adjacent size
            m2 = re.search(r'(\d+\.?\d*)\s*(fl oz|oz|ounce|ounces|kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre)\b', s)
            if m2 and np.isnan(qty_each_mass_g) and np.isnan(qty_each_vol_ml):
                val = float(m2.group(1)); unit = m2.group(2)
                if unit in VOL_UNITS:
                    qty_each_vol_ml = val * VOL_UNITS[unit]
                elif unit in MASS_UNITS:
                    qty_each_mass_g = val * MASS_UNITS[unit]

    return {
        'qty_each_mass_g': qty_each_mass_g,
        'qty_each_vol_ml': qty_each_vol_ml,
        'units_mentioned': total_units
    }

def extract_brand(title: str) -> str:
    t = str(title).strip()
    first = re.split(r'[-|:]', t)[0]
    toks = re.findall(r'[A-Za-z0-9]+', first)
    if not toks: return 'unknown'
    b = toks[0] if len(toks[0]) > 2 else (' '.join(toks[:2]) if len(toks)>1 else toks[0])
    return b.lower()

# Build a single frame for train+test for consistent encoding
df_all = pd.concat([
    train[['sample_id','catalog_content']].assign(is_train=1),
    test[['sample_id','catalog_content']].assign(is_train=0)
], ignore_index=True)

packs, cases, mass_each, vol_each, units_any, brand, len_chars, len_words, num_digits = [],[],[],[],[],[],[],[],[]

for s in df_all['catalog_content'].fillna(''):
    ss = normalize_text(s)
    pc = parse_pack_counts(ss)
    qt = parse_quantity(ss)
    packs.append(pc['pack_count'])
    cases.append(pc['case_count'])
    mass_each.append(qt['qty_each_mass_g'])
    vol_each.append(qt['qty_each_vol_ml'])
    units_any.append(qt['units_mentioned'])
    brand.append(extract_brand(s))
    len_chars.append(len(ss))
    len_words.append(len(ss.split()))
    num_digits.append(sum(ch.isdigit() for ch in ss))

FE = pd.DataFrame({
    'pack_count': packs,
    'case_count': cases,
    'qty_each_mass_g': mass_each,
    'qty_each_vol_ml': vol_each,
    'units_mentioned': units_any,
    'len_chars': len_chars,
    'len_words': len_words,
    'num_digits': num_digits,
    'brand': brand
})

# One-hot top-K brands
K = 200
top_brands = FE['brand'].value_counts().head(K).index
FE['brand_top'] = np.where(FE['brand'].isin(top_brands), FE['brand'], 'other')
brand_oh = pd.get_dummies(FE['brand_top'], prefix='brand', dtype=np.uint8)

FE_NUM = FE.drop(columns=['brand','brand_top']).fillna(0)
FE_ALL = pd.concat([FE_NUM, brand_oh], axis=1).astype(float)

FE_TR = FE_ALL[df_all['is_train']==1].reset_index(drop=True)
FE_TE = FE_ALL[df_all['is_train']==0].reset_index(drop=True)

print("Feature shapes ->", FE_TR.shape, FE_TE.shape)

# ==== Coverage summary (how many rows we successfully parsed) ====
def pct(x): return f"{100.0*np.mean(x):.1f}%"

cov = pd.Series({
    'has_mass_each_g': pct(~np.isnan(FE['qty_each_mass_g'])),
    'has_vol_each_ml': pct(~np.isnan(FE['qty_each_vol_ml'])),
    'has_any_units_mentioned': pct(~np.isnan(FE['units_mentioned'])),
    'pack_count>1': pct(FE['pack_count'].values>1),
    'case_count>0': pct(FE['case_count'].values>0),
})
print(cov)

# Show 8 random rows where quantities were detected (sanity check)
hits_idx = np.where((~np.isnan(FE['qty_each_mass_g'])) | (~np.isnan(FE['qty_each_vol_ml'])) | (FE['pack_count']>1) | (FE['case_count']>0))[0]
sample_hits = np.random.RandomState(42).choice(hits_idx, size=min(8, len(hits_idx)), replace=False) if len(hits_idx)>0 else []
display(pd.concat([df_all.loc[sample_hits, ['catalog_content']], FE.loc[sample_hits, ['pack_count','case_count','qty_each_mass_g','qty_each_vol_ml','units_mentioned','brand']]], axis=1))


In [ ]:
import re, numpy as np, pandas as pd

GRAM_PER_OUNCE = 28.349523125
ML_PER_FLOZ    = 29.5735295625

MASS_UNITS = {
    'g': 1.0, 'gram': 1.0, 'grams': 1.0, 'gm': 1.0,
    'kg': 1000.0,
    'oz': GRAM_PER_OUNCE, 'ounce': GRAM_PER_OUNCE, 'ounces': GRAM_PER_OUNCE
}
VOL_UNITS  = {
    'ml': 1.0,
    'l': 1000.0, 'lt': 1000.0, 'ltr': 1000.0, 'liter': 1000.0, 'litre': 1000.0,
    'fl oz': ML_PER_FLOZ, 'floz': ML_PER_FLOZ, 'fluid ounce': ML_PER_FLOZ, 'fluid ounces': ML_PER_FLOZ
}
ITEM_WORDS = r'(?:bottles?|cans?|bags?|pouches?|bars?|cups?|sachets?|packets?|boxes?|tubes?|jars?)'

def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = s.replace('×', 'x')
    s = re.sub(r'fluid\s*oz', 'fl oz', s)
    s = re.sub(r'fl\.?\s*oz\.?', 'fl oz', s)
    s = re.sub(r'(\d)([a-z])', r'\1 \2', s)  # "16.9oz" -> "16.9 oz"
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# --- (1) Pack/case detectors ---
def parse_pack_counts(s: str) -> dict:
    pack, case = 1, 0
    m = re.search(r'(?:pack|set)\s*of\s*(\d+)', s)
    if m: pack = max(pack, int(m.group(1)))
    m = re.search(r'(\d+)\s*(?:pcs|piece|pieces?)\b', s)
    if m: pack = max(pack, int(m.group(1)))
    m = re.search(r'(\d+)\s*(?:per\s*case|/ ?case)\b', s)
    if m: case = max(case, int(m.group(1)))
    m = re.search(r'(\d+)\s*' + ITEM_WORDS + r'\s*(?:per\s*case|/ ?case)\b', s)
    if m: case = max(case, int(m.group(1)))
    m = re.search(r'pallet\s*of\s*(\d+)\s*cases', s)
    if m: case = max(case, int(m.group(1)))
    return {'pack_count': pack, 'case_count': case}

# --- (2) Quantity detectors from free text like "24 x 16.9 oz" or "500 g" ---
def parse_quantity_free_text(s: str) -> dict:
    qty_each_mass_g = np.nan
    qty_each_vol_ml = np.nan
    units_mentioned = np.nan

    m = re.search(r'(\d+)\s*x\s*(\d+\.?\d*)\s*(fl oz|oz|ounce|ounces|kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre)\b', s)
    if m:
        n = int(m.group(1)); val = float(m.group(2)); unit = m.group(3)
        units_mentioned = float(n)
        if unit in VOL_UNITS:
            qty_each_vol_ml = val * VOL_UNITS[unit]
        elif unit in MASS_UNITS:
            qty_each_mass_g = val * MASS_UNITS[unit]

    if np.isnan(qty_each_mass_g) and np.isnan(qty_each_vol_ml):
        m = re.search(r'(\d+\.?\d*)\s*(fl oz|oz|ounce|ounces|kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre)\b', s)
        if m:
            val = float(m.group(1)); unit = m.group(2)
            if unit in VOL_UNITS:
                qty_each_vol_ml = val * VOL_UNITS[unit]
            elif unit in MASS_UNITS:
                qty_each_mass_g = val * MASS_UNITS[unit]

    if np.isnan(units_mentioned):
        m = re.search(r'(\d+)\s*' + ITEM_WORDS + r'\b', s)
        if m: units_mentioned = float(m.group(1))

    return {'qty_each_mass_g': qty_each_mass_g, 'qty_each_vol_ml': qty_each_vol_ml,
            'units_mentioned': units_mentioned}


def parse_value_unit_block(raw: str) -> dict:
    s = normalize_text(raw)
    v = re.search(r'value\s*:\s*([0-9]+(?:\.[0-9]+)?)', s)
    u = re.search(r'unit\s*:\s*([a-z0-9 ]+)', s)
    val = float(v.group(1)) if v else np.nan
    unit = u.group(1).strip() if u else None
    out = {'vu_each_mass_g': np.nan, 'vu_each_vol_ml': np.nan, 'vu_units_count': np.nan}
    if unit:
        unit = unit.replace('.', '')
        if unit in VOL_UNITS:
            out['vu_each_vol_ml'] = val * VOL_UNITS[unit]
        elif unit in MASS_UNITS:
            out['vu_each_mass_g'] = val * MASS_UNITS[unit]
        elif unit in ('count', 'counts', 'ct'):
            out['vu_units_count'] = val
    return out

GENERIC_FIRST = {'item', 'the', 'a', 'an', 'pack', 'set'}
def extract_brand(raw: str) -> str:
    s = str(raw)
    m = re.search(r'item name\s*:\s*([^\n]+)', s, flags=re.I)
    seg = m.group(1) if m else s
    seg = re.split(r'[\|\-\:\(\,]', seg)[0]  # stop at first delimiter
    toks = re.findall(r"[A-Za-z0-9']+", seg)
    if not toks: return 'unknown'
    b = toks[0].lower()
    if b in GENERIC_FIRST and len(toks) > 1:
        b = toks[1].lower()
    return b


df_all = pd.concat([
    train[['sample_id','catalog_content']].assign(is_train=1),
    test[['sample_id','catalog_content']].assign(is_train=0)
], ignore_index=True)

packs, cases, mass_each, vol_each, units_any = [], [], [], [], []
vu_mass, vu_vol, vu_units = [], [], []
brand, len_chars, len_words, num_digits = [], [], [], []

for raw in df_all['catalog_content'].fillna(''):
    s = normalize_text(raw)
    pc = parse_pack_counts(s)
    qt = parse_quantity_free_text(s)
    vu = parse_value_unit_block(raw)

    packs.append(pc['pack_count'])
    cases.append(pc['case_count'])
    mass_each.append(qt['qty_each_mass_g'])
    vol_each.append(qt['qty_each_vol_ml'])
    units_any.append(qt['units_mentioned'])
    vu_mass.append(vu['vu_each_mass_g'])
    vu_vol.append(vu['vu_each_vol_ml'])
    vu_units.append(vu['vu_units_count'])

    brand.append(extract_brand(raw))
    len_chars.append(len(s))
    len_words.append(len(s.split()))
    num_digits.append(sum(ch.isdigit() for ch in s))

FE = pd.DataFrame({
    'pack_count': packs,
    'case_count': cases,
    'qty_each_mass_g_ft': mass_each,
    'qty_each_vol_ml_ft': vol_each,
    'units_mentioned_ft': units_any,
    'qty_each_mass_g_vu': vu_mass,
    'qty_each_vol_ml_vu': vu_vol,
    'units_mentioned_vu': vu_units,
    'len_chars': len_chars,
    'len_words': len_words,
    'num_digits': num_digits,
    'brand': brand
})


FE['qty_each_mass_g'] = FE[['qty_each_mass_g_vu','qty_each_mass_g_ft']].max(axis=1, skipna=True)
FE['qty_each_vol_ml'] = FE[['qty_each_vol_ml_vu','qty_each_vol_ml_ft']].max(axis=1, skipna=True)
FE['units_mentioned'] = FE[['units_mentioned_vu','units_mentioned_ft']].max(axis=1, skipna=True)


FE['eff_units'] = 1.0
FE.loc[FE['pack_count']>1, 'eff_units'] = FE.loc[FE['pack_count']>1, 'pack_count']
FE.loc[FE['units_mentioned'].notna(), 'eff_units'] = FE.loc[FE['units_mentioned'].notna(), 'units_mentioned']
FE.loc[FE['case_count']>0, 'eff_units'] = np.maximum(
    FE.loc[FE['case_count']>0, 'eff_units'], FE.loc[FE['case_count']>0, 'case_count']
)

# totals if both per-item size and eff_units exist
FE['total_mass_g'] = FE['qty_each_mass_g'] * FE['eff_units']
FE['total_vol_ml']  = FE['qty_each_vol_ml'] * FE['eff_units']

# boolean flags
FE['has_mass']  = FE['qty_each_mass_g'].notna().astype(int)
FE['has_vol']   = FE['qty_each_vol_ml'].notna().astype(int)
FE['has_units'] = FE['eff_units'].fillna(1).gt(1).astype(int)
FE['has_case']  = (FE['case_count']>0).astype(int)
FE['has_pack']  = (FE['pack_count']>1).astype(int)

# One-hot top brands (now that we extract real brand names)
K = 200
top_brands = FE['brand'].value_counts().head(K).index
FE['brand_top'] = np.where(FE['brand'].isin(top_brands), FE['brand'], 'other')
brand_oh = pd.get_dummies(FE['brand_top'], prefix='brand', dtype=np.uint8)

# numeric frame
FE_NUM = FE.drop(columns=[
    'brand','brand_top',
    'qty_each_mass_g_ft','qty_each_vol_ml_ft','units_mentioned_ft',
    'qty_each_mass_g_vu','qty_each_vol_ml_vu','units_mentioned_vu'
]).copy()

# clip extreme totals to reasonable ranges using TRAIN quantiles to avoid leakage
is_tr = df_all['is_train'].values.astype(bool)
for col in ['qty_each_mass_g','qty_each_vol_ml','eff_units','total_mass_g','total_vol_ml','len_chars','len_words','num_digits']:
    q99 = np.nanquantile(FE_NUM.loc[is_tr, col].values.astype(float), 0.99)
    FE_NUM[col] = np.clip(FE_NUM[col], None, q99)

FE_NUM = FE_NUM.fillna(0).astype(float)
FE_ALL = pd.concat([FE_NUM, brand_oh], axis=1)

FE_TR = FE_ALL[df_all['is_train']==1].reset_index(drop=True)
FE_TE = FE_ALL[df_all['is_train']==0].reset_index(drop=True)

print("New feature shapes ->", FE_TR.shape, FE_TE.shape)

# Coverage summary
def pct(x): return f"{100.0*np.mean(x):.1f}%"
cov = pd.Series({
    'qty_each_mass_g (any)': pct(FE['qty_each_mass_g'].notna()),
    'qty_each_vol_ml (any)': pct(FE['qty_each_vol_ml'].notna()),
    'eff_units>1'          : pct(FE['eff_units'].fillna(1).gt(1)),
    'has_case'             : pct(FE['case_count'].values>0),
    'brand != other'       : pct(FE['brand'].isin(top_brands)),
})
print(cov)

# Show a few rows (sanity)
idx = np.random.RandomState(7).choice(len(df_all), size=8, replace=False)
display(pd.concat([
    df_all.loc[idx, ['catalog_content']],
    FE.loc[idx, ['brand','pack_count','case_count','qty_each_mass_g','qty_each_vol_ml','eff_units','total_mass_g','total_vol_ml','has_pack','has_case']]
], axis=1))


In [ ]:
from scipy.sparse import save_npz, load_npz
import joblib, os
OUT = "/kaggle/working/tfidf_artifacts"
os.makedirs(OUT, exist_ok=True)

# If you used TfidfVectorizer
save_npz(f"{OUT}/X_tr.npz", X_tr)
save_npz(f"{OUT}/X_te.npz", X_te)
joblib.dump(tfidf, f"{OUT}/tfidf_vectorizer.joblib", compress=3)

# If you used Hashing + SVD
# np.save(... dense arrays) or:
import numpy as np
np.save(f"{OUT}/Xtr_svd.npy", Xtr)
np.save(f"{OUT}/Xte_svd.npy", Xte)
joblib.dump(svd, f"{OUT}/svd.joblib", compress=3)


In [ ]:
!pip -q install sentence-transformers==2.6.1
import os, math, gc, numpy as np, torch
from sentence_transformers import SentenceTransformer

assert 'df_all' in globals() and 'catalog_content' in df_all.columns and 'is_train' in df_all.columns

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
model.max_seq_length = 128  # speed boost, fine for this task

texts = df_all['catalog_content'].fillna('').astype(str).tolist()
N = len(texts); EMB_DIM = 384
SHARD = 10_000
BATCH = 1024 if device == 'cuda' else 128
save_dir = "/kaggle/working/text_emb_shards"
os.makedirs(save_dir, exist_ok=True)

def shard_path(i): return os.path.join(save_dir, f"emb_shard_{i:05d}.npy")

num_shards = math.ceil(N/SHARD)
print(f"Device={device} | rows={N} | shards={num_shards} | shard={SHARD} | batch={BATCH}")

# encode shards; skip ones already saved
for i in range(num_shards):
    fp = shard_path(i)
    if os.path.exists(fp):
        print(f"[skip] {fp}")
        continue
    s, e = i*SHARD, min(N, (i+1)*SHARD)
    emb = model.encode(texts[s:e], batch_size=BATCH, show_progress_bar=True, normalize_embeddings=True).astype('float32')
    np.save(fp, emb.astype('float16'))  # store compactly
    del emb; gc.collect(); print(f"[saved] {fp} ({e-s} rows)")

# stitch shards -> full arrays
parts = [np.load(shard_path(i), mmap_mode='r').astype('float32') for i in range(num_shards)]
EMB_ALL = np.vstack(parts)
is_tr = (df_all['is_train'].values == 1)
EMB_TR, EMB_TE = EMB_ALL[is_tr], EMB_ALL[~is_tr]

print("EMB_ALL shape:", EMB_ALL.shape)
print("Embedding shapes ->", EMB_TR.shape, EMB_TE.shape, "| device:", device)


/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Device=cuda | rows=150000 | shards=15 | shard=10000 | batch=1024
[skip] /kaggle/working/text_emb_shards/emb_shard_00000.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00001.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00002.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00003.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00004.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00005.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00006.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00007.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00008.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00009.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00010.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00011.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00012.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00013.npy
[skip] /kaggle/working/text_emb_shards/emb_shard_00014.npy
EMB_ALL shape: (150000, 384)
Embedding shapes -> (

In [ ]:
import os, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # quiet the warning

assert 'FE_TR' in globals() and 'FE_TE' in globals()
assert 'EMB_TR' in globals() and 'EMB_TE' in globals()
assert 'train' in globals() and 'test' in globals()

# Stack features with embeddings
X_tr = np.hstack([FE_TR, EMB_TR]).astype('float32')
X_te = np.hstack([FE_TE, EMB_TE]).astype('float32')

y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))

# Stratified CV on price bins
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100


def detect_tree_method():
    try:
        probe = xgb.XGBRegressor(
            n_estimators=1, max_depth=2, tree_method='gpu_hist',
            objective='reg:squarederror', random_state=42
        )
        probe.fit(X_tr[:200], y_log[:200], verbose=False)
        print("Using XGBoost tree_method = gpu_hist")
        return 'gpu_hist'
    except Exception as e:
        print("gpu_hist unavailable, falling back to hist. Reason:", str(e)[:120])
        return 'hist'

tree_method = detect_tree_method()

params = dict(
    n_estimators=4000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_lambda=1.0,
    objective='reg:squarederror',
    tree_method=tree_method,
    random_state=42
)

oof_l = np.zeros(len(train), dtype='float32')
te_l  = np.zeros(len(test),  dtype='float32')

for f,(tr_idx, va_idx) in enumerate(skf.split(X_tr, bins)):
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_tr[tr_idx], y_log[tr_idx],
        eval_set=[(X_tr[va_idx], y_log[va_idx])],
        verbose=False, early_stopping_rounds=200
    )
    oof_l[va_idx] = model.predict(X_tr[va_idx])
    te_l += model.predict(X_te) / skf.n_splits

    fold_smape = smape(y[va_idx], np.exp(oof_l[va_idx]).clip(0.01))
    print(f"Fold {f} SMAPE: {fold_smape:.3f}%")

cv_smape = smape(y, np.exp(oof_l).clip(0.01))
print(f"XGB + FE + MiniLM  |  CV SMAPE: {cv_smape:.3f}%")


sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_l).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_v2.csv", index=False)
print("Saved -> out/test_predictions_v2.csv")


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:43:06] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


Using XGBoost tree_method = gpu_hist


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:43:10] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:44:35] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarn

KeyboardInterrupt: 

In [ ]:

sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_l).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_v2.csv", index=False)
print("Saved -> out/test_predictions_v2.csv")


In [ ]:

import os, re, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge



In [ ]:

assert 'train' in globals() and 'test' in globals(), "Need train/test"
assert 'oof_l' in globals() and 'te_l' in globals(), "Run the XGB cell first"

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100

def clean_text(s):
    s = str(s).lower()
    s = re.sub(r'https?://\S+',' ', s)
    s = s.replace('|',' ').replace('/',' ')
    s = re.sub(r'[^a-z0-9\.\-\+\%\s]',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return s



In [ ]:

train['text_r'] = train['catalog_content'].fillna('').map(clean_text)
test ['text_r'] = test ['catalog_content'].fillna('').map(clean_text)

tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(1,3),
                        max_features=300_000, min_df=3)
X_tr_r = tfidf.fit_transform(train['text_r'])   
X_te_r = tfidf.transform(test['text_r'])

y      = train['price'].astype(float).values
y_log  = np.log(np.clip(y, 0.01, None))
bins   = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# —— small alpha grid
alphas = [0.5, 1.0, 2.0, 4.0, 8.0]
best = {'alpha': None, 'cv': 1e9, 'oof_log': None, 'te_log': None}



In [ ]:
for a in alphas:
    oof_log = np.zeros(len(train)); te_log = np.zeros(len(test))
    for tr, va in skf.split(X_tr_r, bins):
        # force a safe solver to avoid SciPy cg() issue
        m = Ridge(alpha=a, solver='lsqr', random_state=42)
        m.fit(X_tr_r[tr], y_log[tr])
        oof_log[va] = m.predict(X_tr_r[va])
        te_log += m.predict(X_te_r) / skf.n_splits
    cv = smape(y, np.exp(oof_log).clip(0.01))
    print(f"Ridge alpha={a:.2f} | CV SMAPE={cv:.3f}%")
    if cv < best['cv']:
        best = {'alpha': a, 'cv': cv, 'oof_log': oof_log.copy(), 'te_log': te_log.copy()}

ridge_oof_log = best['oof_log']; ridge_te_log = best['te_log']
print(f"Best Ridge alpha={best['alpha']} | CV={best['cv']:.3f}%")


In [ ]:

w_grid = np.linspace(0.0, 1.0, 21)  
best_w, best_cv = None, 1e9
for w in w_grid:
    pred = np.exp(w*ridge_oof_log + (1-w)*oof_l).clip(0.01)
    cv = smape(y, pred)
    if cv < best_cv:
        best_cv, best_w = cv, w

print(f"Blend weight on Ridge: w={best_w:.2f} | Blended CV SMAPE={best_cv:.3f}%")

# —— blended test prediction & save
te_blend_log = best_w*ridge_te_log + (1-best_w)*te_l
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_blend_log).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_blend.csv", index=False)
print("Saved -> out/test_predictions_blend.csv")


In [ ]:

import numpy as np, pandas as pd

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100


best_w, best_cv = None, 1e9
for w in w_grid:
    oof_pred_blend = np.exp(w*ridge_oof_log + (1-w)*oof_l).clip(0.01)
    cv = smape(y, oof_pred_blend)
    if cv < best_cv:
        best_cv, best_w = cv, w

print(f"Best blend weight on Ridge: w={best_w:.2f} | Blended CV SMAPE={best_cv:.3f}%")


try:
    dec = pd.qcut(y, 10, labels=False, duplicates='drop')
    oof_xgb  = np.exp(oof_l).clip(0.01)
    oof_rid  = np.exp(ridge_oof_log).clip(0.01)
    oof_blnd = np.exp(best_w*ridge_oof_log + (1-best_w)*oof_l).clip(0.01)
    def smape_s(y_true, y_pred): return smape(y_true, y_pred)
    tab = pd.DataFrame({
        'XGB':   pd.Series([smape_s(y[dec==d], oof_xgb [dec==d]) for d in sorted(dec.unique())]),
        'Ridge': pd.Series([smape_s(y[dec==d], oof_rid [dec==d]) for d in sorted(dec.unique())]),
        'Blend': pd.Series([smape_s(y[dec==d], oof_blnd[dec==d]) for d in sorted(dec.unique())]),
    })
    print("SMAPE by price decile (lower is better):")
    display(tab.round(2))
except Exception as e:
    pass


te_blend_log = best_w*ridge_te_log + (1-best_w)*te_l
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_blend_log).clip(0.01).astype(float)})
import os; os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_blend.csv", index=False)
print("Saved -> out/test_predictions_blend.csv")


In [ ]:
# === Tiny spec flags + XGB retrain (device='cuda') ===
!pip -q install xgboost==2.0.3
import os, re, gc, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

assert 'train' in globals() and 'test' in globals() and 'df_all' in globals()
assert 'EMB_TR' in globals() and 'EMB_TE' in globals()

# ---------- Helpers (same base as before) ----------
GRAM_PER_OUNCE = 28.349523125
ML_PER_FLOZ    = 29.5735295625
MASS_UNITS = {'g':1.0,'gram':1.0,'grams':1.0,'gm':1.0,'kg':1000.0,'oz':GRAM_PER_OUNCE,'ounce':GRAM_PER_OUNCE,'ounces':GRAM_PER_OUNCE}
VOL_UNITS  = {'ml':1.0,'l':1000.0,'lt':1000.0,'ltr':1000.0,'liter':1000.0,'litre':1000.0,
              'fl oz':ML_PER_FLOZ,'floz':ML_PER_FLOZ,'fluid ounce':ML_PER_FLOZ,'fluid ounces':ML_PER_FLOZ}
ITEM_WORDS = r'(?:bottles?|cans?|bags?|pouches?|bars?|cups?|sachets?|packets?|boxes?|tubes?|jars?)'
GENERIC_FIRST = {'item','the','a','an','pack','set'}

def normalize_text(s: str) -> str:
    s = str(s).lower().replace('×','x')
    s = re.sub(r'fluid\s*oz', 'fl oz', s)
    s = re.sub(r'fl\.?\s*oz\.?', 'fl oz', s)
    s = re.sub(r'(\d)([a-z])', r'\1 \2', s)
    s = re.sub(r'\s+',' ', s).strip()
    return s

def parse_pack_counts(s: str):
    pack, case = 1, 0
    m = re.search(r'(?:pack|set)\s*of\s*(\d+)', s);                   pack = max(pack, int(m.group(1))) if m else pack
    m = re.search(r'(\d+)\s*(?:pcs|piece|pieces?)\b', s);             pack = max(pack, int(m.group(1))) if m else pack
    m = re.search(r'(\d+)\s*(?:per\s*case|/ ?case)\b', s);            case = max(case, int(m.group(1))) if m else case
    m = re.search(r'(\d+)\s*'+ITEM_WORDS+r'\s*(?:per\s*case|/ ?case)\b', s); case = max(case, int(m.group(1))) if m else case
    m = re.search(r'pallet\s*of\s*(\d+)\s*cases', s);                 case = max(case, int(m.group(1))) if m else case
    return pack, case

def parse_quantity_free_text(s: str):
    qty_mass = np.nan; qty_vol = np.nan; units_any = np.nan
    m = re.search(r'(\d+)\s*x\s*(\d+\.?\d*)\s*(fl oz|oz|ounce|ounces|kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre)\b', s)
    if m:
        n, val, unit = int(m.group(1)), float(m.group(2)), m.group(3)
        units_any = float(n)
        qty_vol = val*VOL_UNITS[unit] if unit in VOL_UNITS else qty_vol
        qty_mass = val*MASS_UNITS[unit] if unit in MASS_UNITS else qty_mass
    if np.isnan(qty_mass) and np.isnan(qty_vol):
        m = re.search(r'(\d+\.?\d*)\s*(fl oz|oz|ounce|ounces|kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre)\b', s)
        if m:
            val, unit = float(m.group(1)), m.group(2)
            qty_vol = val*VOL_UNITS[unit] if unit in VOL_UNITS else qty_vol
            qty_mass = val*MASS_UNITS[unit] if unit in MASS_UNITS else qty_mass
    if np.isnan(units_any):
        m = re.search(r'(\d+)\s*'+ITEM_WORDS+r'\b', s)
        if m: units_any = float(m.group(1))
    return qty_mass, qty_vol, units_any

def parse_value_unit_block(raw: str):
    s = normalize_text(raw)
    v = re.search(r'value\s*:\s*([0-9]+(?:\.[0-9]+)?)', s)
    u = re.search(r'unit\s*:\s*([a-z0-9 ]+)', s)
    val = float(v.group(1)) if v else np.nan
    unit = u.group(1).strip().replace('.','') if u else None
    vu_mass = vu_vol = vu_units = np.nan
    if unit:
        if unit in VOL_UNITS: vu_vol = val*VOL_UNITS[unit]
        elif unit in MASS_UNITS: vu_mass = val*MASS_UNITS[unit]
        elif unit in ('count','counts','ct'): vu_units = val
    return vu_mass, vu_vol, vu_units

def extract_brand(raw: str) -> str:
    s = str(raw)
    m = re.search(r'item name\s*:\s*([^\n]+)', s, flags=re.I)
    seg = m.group(1) if m else s
    seg = re.split(r'[\|\-\:\(\,]', seg)[0]
    toks = re.findall(r"[A-Za-z0-9']+", seg)
    if not toks: return 'unknown'
    b = toks[0].lower()
    if b in GENERIC_FIRST and len(toks)>1: b = toks[1].lower()
    return b

# ---------- Build features + tiny spec flags ----------
packs, cases, qty_m_ft, qty_v_ft, units_ft = [],[],[],[],[]
qty_m_vu, qty_v_vu, units_vu = [],[],[]
brand, len_chars, len_words, num_digits = [],[],[],[]
has_gb, has_inch, has_w, has_typec = [], [], [], []

for raw in df_all['catalog_content'].fillna(''):
    s = normalize_text(raw)
    p, c = parse_pack_counts(s)
    qm, qv, un = parse_quantity_free_text(s)
    vm, vv, vu = parse_value_unit_block(raw)

    packs.append(p); cases.append(c)
    qty_m_ft.append(qm); qty_v_ft.append(qv); units_ft.append(un)
    qty_m_vu.append(vm); qty_v_vu.append(vv); units_vu.append(vu)
    brand.append(extract_brand(raw))
    len_chars.append(len(s)); len_words.append(len(s.split()))
    num_digits.append(sum(ch.isdigit() for ch in s))

    # tiny spec flags
    has_gb.append(bool(re.search(r'\b\d{1,4}\s*gb\b', s)))
    has_inch.append(bool(re.search(r'\b\d{1,2}(?:\.\d)?\s*(?:inch|inches|")\b', s)))
    has_w.append(bool(re.search(r'\b\d{1,4}\s*(?:w|watt|watts)\b', s)))
    has_typec.append(bool(re.search(r'\b(?:type[- ]?c|usb[- ]?c|usbc)\b', s)))

FE = pd.DataFrame({
    'pack_count': packs, 'case_count': cases,
    'qty_each_mass_g_ft': qty_m_ft, 'qty_each_vol_ml_ft': qty_v_ft, 'units_mentioned_ft': units_ft,
    'qty_each_mass_g_vu': qty_m_vu, 'qty_each_vol_ml_vu': qty_v_vu, 'units_mentioned_vu': units_vu,
    'len_chars': len_chars, 'len_words': len_words, 'num_digits': num_digits,
    'has_gb': has_gb, 'has_inch': has_inch, 'has_w': has_w, 'has_typec': has_typec,
    'brand': brand
})

FE['qty_each_mass_g'] = FE[['qty_each_mass_g_vu','qty_each_mass_g_ft']].max(axis=1, skipna=True)
FE['qty_each_vol_ml'] = FE[['qty_each_vol_ml_vu','qty_each_vol_ml_ft']].max(axis=1, skipna=True)
FE['units_mentioned'] = FE[['units_mentioned_vu','units_mentioned_ft']].max(axis=1, skipna=True)

FE['eff_units'] = 1.0
FE.loc[FE['pack_count']>1, 'eff_units'] = FE['pack_count']
FE.loc[FE['units_mentioned'].notna(), 'eff_units'] = FE['units_mentioned']
FE.loc[FE['case_count']>0, 'eff_units'] = np.maximum(FE['eff_units'], FE['case_count'])

FE['total_mass_g'] = FE['qty_each_mass_g'] * FE['eff_units']
FE['total_vol_ml']  = FE['qty_each_vol_ml'] * FE['eff_units']

FE['has_mass']  = FE['qty_each_mass_g'].notna().astype(int)
FE['has_vol']   = FE['qty_each_vol_ml'].notna().astype(int)
FE['has_units'] = FE['eff_units'].fillna(1).gt(1).astype(int)
FE['has_case']  = (FE['case_count']>0).astype(int)
FE['has_pack']  = (FE['pack_count']>1).astype(int)

# one-hot top brands
K = 200
top_brands = FE['brand'].value_counts().head(K).index
FE['brand_top'] = np.where(FE['brand'].isin(top_brands), FE['brand'], 'other')
brand_oh = pd.get_dummies(FE['brand_top'], prefix='brand', dtype=np.uint8)

# numeric frame w/ mild clipping (train-only for caps)
is_tr_mask = (df_all['is_train'].values==1)
FE_NUM = FE.drop(columns=[
    'brand','brand_top',
    'qty_each_mass_g_ft','qty_each_vol_ml_ft','units_mentioned_ft',
    'qty_each_mass_g_vu','qty_each_vol_ml_vu','units_mentioned_vu'
]).copy()

for col in ['qty_each_mass_g','qty_each_vol_ml','eff_units','total_mass_g','total_vol_ml',
            'len_chars','len_words','num_digits']:
    q99 = np.nanquantile(FE_NUM.loc[is_tr_mask, col], 0.99)
    FE_NUM[col] = np.clip(FE_NUM[col], None, q99)

FE_NUM = FE_NUM.fillna(0).astype('float32')
FE_ALL = pd.concat([FE_NUM, brand_oh.astype('float32')], axis=1)

FE_TR = FE_ALL[df_all['is_train']==1].reset_index(drop=True).values
FE_TE = FE_ALL[df_all['is_train']==0].reset_index(drop=True).values
print("FE shapes (with spec flags) ->", FE_TR.shape, FE_TE.shape)

# ---------- Stack with MiniLM embeddings ----------
X_tr = np.hstack([FE_TR, EMB_TR]).astype('float32')
X_te = np.hstack([FE_TE, EMB_TE]).astype('float32')
print("Model matrices ->", X_tr.shape, X_te.shape)

# ---------- Train (CUDA) ----------
y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100

params = dict(
    n_estimators=4000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_lambda=1.0,
    objective='reg:squarederror',
    tree_method='hist',
    device='cuda',
    random_state=42
)

oof_l2 = np.zeros(len(train), dtype='float32')
te_l2  = np.zeros(len(test),  dtype='float32')

for f,(tr_idx, va_idx) in enumerate(skf.split(X_tr, bins)):
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_tr[tr_idx], y_log[tr_idx],
        eval_set=[(X_tr[va_idx], y_log[va_idx])],
        verbose=False, early_stopping_rounds=200
    )
    oof_l2[va_idx] = model.predict(X_tr[va_idx])
    te_l2 += model.predict(X_te) / skf.n_splits
    print(f"Fold {f} SMAPE: {smape(y[va_idx], np.exp(oof_l2[va_idx]).clip(0.01)):.3f}%")

cv2 = smape(y, np.exp(oof_l2).clip(0.01))
print(f"XGB + FE + MiniLM + spec flags | CV SMAPE: {cv2:.3f}%")

# save
sub3 = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_l2).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub3.to_csv("out/test_predictions_v3.csv", index=False)
print("Saved -> out/test_predictions_v3.csv")


In [24]:
# === Stacking: add Ridge OOF (log) as a feature into XGBoost ===
import os, re, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

assert 'train' in globals() and 'test' in globals()
assert 'FE_TR' in globals() and 'FE_TE' in globals() and 'EMB_TR' in globals() and 'EMB_TE' in globals()

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100

def clean_text(s):
    s = str(s).lower()
    s = re.sub(r'https?://\S+',' ', s)
    s = s.replace('|',' ').replace('/',' ')
    s = re.sub(r'[^a-z0-9\.\-\+\%\s]',' ', s)
    s = re.sub(r'\s+',' ', s).strip()
    return s

# --- Ridge TF-IDF (char) OOF in LOG space (no leakage) ---
train['text_r'] = train['catalog_content'].fillna('').map(clean_text)
test ['text_r'] = test ['catalog_content'].fillna('').map(clean_text)

tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(1,3), max_features=300_000, min_df=3)
X_tr_r = tfidf.fit_transform(train['text_r'])   # fit on train only
X_te_r = tfidf.transform(test['text_r'])

y      = train['price'].astype(float).values
y_log  = np.log(np.clip(y, 0.01, None))
bins   = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# fixed alpha=0.5 (your best); safe solver to avoid SciPy cg() issue
ridge_oof_log = np.zeros(len(train)); ridge_te_log = np.zeros(len(test))
for tr, va in skf.split(X_tr_r, bins):
    m = Ridge(alpha=0.5, solver='lsqr', random_state=42)
    m.fit(X_tr_r[tr], y_log[tr])
    ridge_oof_log[va] = m.predict(X_tr_r[va])
    ridge_te_log += m.predict(X_te_r) / skf.n_splits

print("Ridge OOF CV SMAPE:", f"{smape(y, np.exp(ridge_oof_log).clip(0.01)):.3f}%")

# --- Stack Ridge OOF/TEST as one extra feature ---
X_tr = np.hstack([FE_TR, EMB_TR, ridge_oof_log.reshape(-1,1)]).astype('float32')
X_te = np.hstack([FE_TE, EMB_TE, ridge_te_log.reshape(-1,1)]).astype('float32')
print("Stacked shapes:", X_tr.shape, X_te.shape)

# --- XGBoost on log(price), CUDA, MAE metric (aligns better with SMAPE on log) ---
params = dict(
    n_estimators=4000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_lambda=1.0,
    objective='reg:squarederror',
    eval_metric='mae',    # on log target
    tree_method='hist',
    device='cuda',
    random_state=42
)

oof_l = np.zeros(len(train), dtype='float32')
te_l  = np.zeros(len(test),  dtype='float32')

for f,(tr_idx, va_idx) in enumerate(skf.split(X_tr, bins)):
    model = xgb.XGBRegressor(**params, early_stopping_rounds=200)
    model.fit(
        X_tr[tr_idx], y_log[tr_idx],
        eval_set=[(X_tr[va_idx], y_log[va_idx])],
        verbose=False
    )
    oof_l[va_idx] = model.predict(X_tr[va_idx])
    te_l += model.predict(X_te) / skf.n_splits
    print(f"Fold {f} SMAPE: {smape(y[va_idx], np.exp(oof_l[va_idx]).clip(0.01)):.3f}%")

cv = smape(y, np.exp(oof_l).clip(0.01))
print(f"STACKED XGB (FE + MiniLM + RidgeOOF) | CV SMAPE: {cv:.3f}%")

# Save
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_l).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_stack.csv", index=False)
print("Saved -> out/test_predictions_stack.csv")


Ridge OOF CV SMAPE: 58.497%
Stacked shapes: (75000, 602) (75000, 602)
Fold 0 SMAPE: 50.860%
Fold 1 SMAPE: 50.794%
Fold 2 SMAPE: 51.294%
Fold 3 SMAPE: 50.954%
Fold 4 SMAPE: 50.270%
STACKED XGB (FE + MiniLM + RidgeOOF) | CV SMAPE: 50.834%
Saved -> out/test_predictions_stack.csv


In [25]:
# === Train STACKED XGB (FE + MiniLM + Ridge-OOF as a feature) on CUDA ===
import os, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

# expects these from earlier cells:
#   train, test
#   FE_TR, FE_TE, EMB_TR, EMB_TE
#   ridge_oof_log, ridge_te_log   (Ridge on TF-IDF with alpha=0.5, solver='lsqr')

for name in ['train','test','FE_TR','FE_TE','EMB_TR','EMB_TE','ridge_oof_log','ridge_te_log']:
    assert name in globals(), f"Missing {name}. Run previous cells first."

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100

# stack features
X_tr = np.hstack([FE_TR, EMB_TR, ridge_oof_log.reshape(-1,1)]).astype('float32')
X_te = np.hstack([FE_TE, EMB_TE, ridge_te_log.reshape(-1,1)]).astype('float32')
print("Stacked train/test shapes:", X_tr.shape, X_te.shape)

y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# XGBoost params (CUDA), early stopping in constructor (no warnings)
params = dict(
    n_estimators=4000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_lambda=1.0,
    objective='reg:squarederror',
    eval_metric='mae',     # on log target; correlates with SMAPE
    tree_method='hist',
    device='cuda',
    random_state=42,
    early_stopping_rounds=200
)

oof_stack_l = np.zeros(len(train), dtype='float32')
te_stack_l  = np.zeros(len(test),  dtype='float32')

for f,(tr_idx, va_idx) in enumerate(skf.split(X_tr, bins)):
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_tr[tr_idx], y_log[tr_idx],
        eval_set=[(X_tr[va_idx], y_log[va_idx])],
        verbose=False
    )
    oof_stack_l[va_idx] = model.predict(X_tr[va_idx])
    te_stack_l += model.predict(X_te) / skf.n_splits
    fold_smape = smape(y[va_idx], np.exp(oof_stack_l[va_idx]).clip(0.01))
    print(f"Fold {f} SMAPE: {fold_smape:.3f}%")

cv = smape(y, np.exp(oof_stack_l).clip(0.01))
print(f"STACKED XGB CV SMAPE: {cv:.3f}%")

# save submission from stacked model
sub_stack = pd.DataFrame({
    'sample_id': test['sample_id'],
    'price': np.exp(te_stack_l).clip(0.01).astype(float)
})
os.makedirs("out", exist_ok=True)
sub_stack.to_csv("out/test_predictions_stack.csv", index=False)
print("Saved -> out/test_predictions_stack.csv")


Stacked train/test shapes: (75000, 602) (75000, 602)
Fold 0 SMAPE: 50.860%
Fold 1 SMAPE: 50.794%
Fold 2 SMAPE: 51.294%
Fold 3 SMAPE: 50.954%
Fold 4 SMAPE: 50.270%
STACKED XGB CV SMAPE: 50.834%
Saved -> out/test_predictions_stack.csv


In [26]:
# ==== Blend: STACKED (oof_stack_l) vs earlier XGB (oof_l / oof_l2) ====
import numpy as np, pandas as pd, os

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100

# Expect these from previous cells:
# y (train prices), test (for sample_id), oof_stack_l/te_stack_l (STACKED, log),
# oof_l/te_l (earlier XGB, log) and/or oof_l2/te_l2 (spec-flags XGB, log)

for name in ['y','test','oof_stack_l','te_stack_l']:
    assert name in globals(), f"Missing {name}. Run the stacked XGB cell first."

# pick best available base model to blend with
if 'oof_l' in globals():
    base_oof_log, base_te_log, base_name = oof_l, te_l, 'XGB FE+MiniLM'
elif 'oof_l2' in globals():
    base_oof_log, base_te_log, base_name = oof_l2, te_l2, 'XGB FE+MiniLM+spec'
else:
    raise AssertionError("No base XGB oof/test arrays found (oof_l/te_l or oof_l2/te_l2).")

print("Blending STACKED with:", base_name)

# grid over weight on STACKED (in LOG-SPACE)
w_grid = np.linspace(0.0, 1.0, 21)  # 0..1 step 0.05
best_w, best_cv = None, 1e9
for w in w_grid:
    oof_blend_price = np.exp(w*oof_stack_l + (1-w)*base_oof_log).clip(0.01)
    cv = smape(y, oof_blend_price)
    if cv < best_cv:
        best_cv, best_w = cv, w

print(f"Best weight on STACKED: w={best_w:.2f} | Blended CV SMAPE={best_cv:.3f}%")

# apply to test
te_blend_log = best_w*te_stack_l + (1-best_w)*base_te_log
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_blend_log).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_stack_blend.csv", index=False)
print("Saved -> out/test_predictions_stack_blend.csv")


Blending STACKED with: XGB FE+MiniLM
Best weight on STACKED: w=0.40 | Blended CV SMAPE=50.834%
Saved -> out/test_predictions_stack_blend.csv


In [ ]:
# === Brand-wise calibration on top of STACKED model (log-space residual bias) ===
import re, numpy as np, pandas as pd, os

for name in ['train','test','df_all','oof_stack_l','te_stack_l']:
    assert name in globals(), f"Missing {name}. Run stacked model cell first."

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100

# --- brand extractor (same logic we used earlier) ---
GENERIC_FIRST = {'item','the','a','an','pack','set'}
def extract_brand(raw: str) -> str:
    s = str(raw)
    m = re.search(r'item name\s*:\s*([^\n]+)', s, flags=re.I)
    seg = m.group(1) if m else s
    seg = re.split(r'[\|\-\:\(\,]', seg)[0]
    toks = re.findall(r"[A-Za-z0-9']+", seg)
    if not toks: return 'unknown'
    b = toks[0].lower()
    if b in GENERIC_FIRST and len(toks)>1: b = toks[1].lower()
    return b

# --- build brand_top for train/test consistently (top-200 from TRAIN) ---
brand_train = df_all.loc[df_all['is_train']==1, 'catalog_content'].fillna('').map(extract_brand).astype(str)
brand_test  = df_all.loc[df_all['is_train']==0, 'catalog_content'].fillna('').map(extract_brand).astype(str)

topK = 200
top_brands = brand_train.value_counts().head(topK).index
brand_top_tr = np.where(brand_train.isin(top_brands), brand_train, 'other')
brand_top_te = np.where(brand_test.isin(top_brands),  brand_test,  'other')

# --- compute log residuals on OOF, then per-brand bias with shrinkage ---
y      = train['price'].astype(float).values
y_log  = np.log(np.clip(y, 0.01, None))
resid  = y_log - oof_stack_l  # positive residual => model underpredicts

df_res = pd.DataFrame({'brand_top': brand_top_tr, 'resid': resid})
grp    = df_res.groupby('brand_top')['resid']
brand_mean = grp.mean()
brand_cnt  = grp.size()

# James–Stein-like shrink toward 0 to avoid overfitting small brands
LAMBDA = 200  # smoothing strength
bias = {}
for b in brand_mean.index:
    n = brand_cnt[b]
    mu = brand_mean[b]
    shrink = n / (n + LAMBDA)
    adj = shrink * mu
    bias[b] = float(np.clip(adj, -0.25, 0.25))  # clip ±0.25 in log (~±28% multiplier)

# map biases to train/test
bias_tr = np.vectorize(lambda b: bias.get(b, 0.0))(brand_top_tr)
bias_te = np.vectorize(lambda b: bias.get(b, 0.0))(brand_top_te)

# --- apply calibration in LOG space ---
oof_cal_l = oof_stack_l + bias_tr
te_cal_l  = te_stack_l  + bias_te

cv_before = smape(y, np.exp(oof_stack_l).clip(0.01))
cv_after  = smape(y, np.exp(oof_cal_l ).clip(0.01))
print(f"Brand-calibration CV: before={cv_before:.3f}%  ->  after={cv_after:.3f}%  (Δ={cv_after-cv_before:+.3f} pp)")

# --- save calibrated submission ---
sub_cal = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_cal_l).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub_cal.to_csv("out/test_predictions_stack_calibrated.csv", index=False)
print("Saved -> out/test_predictions_stack_calibrated.csv")


In [29]:
# === Targeted OpenCLIP image embeddings + fusion retrain (CUDA) ===
!pip -q install open-clip-torch==2.24.0 pillow==10.4.0 torchvision==0.18.1

import os, re, io, math, gc, time, numpy as np, pandas as pd, requests
from PIL import Image, ImageFile
import torch
import torchvision.transforms as T
import open_clip
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

# --- prereqs from earlier steps:
for name in ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','oof_stack_l','te_stack_l']:
    assert name in globals(), f"Missing {name}. Run previous cells first."



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
torchaudio 2.6.0+cu124 requires torch==2.6.0, but you have torch 2.3.1 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu12 25.6.0 requires rmm-cu12==25.6.*, but you have rmm-cu12 25.2.0 which is incompatible.


/usr/local/lib/python3.11/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
# ------------------ helpers ------------------
ImageFile.LOAD_TRUNCATED_IMAGES = True
UA = {"User-Agent": "Mozilla/5.0 (compatible; Kaggle/Images/1.0)"}
IMG_DIR = "/kaggle/working/images"; EMB_DIR = "/kaggle/working/img_emb"
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(EMB_DIR, exist_ok=True)

def safe_filename(sid): return f"{sid}.jpg"
def emb_path(sid): return os.path.join(EMB_DIR, f"{sid}.npy")
def img_path(sid): return os.path.join(IMG_DIR, safe_filename(sid))

def download_image(sample_id, url, timeout=10, retries=3, sleep=0.5):
    """Download to cache if not exists. Return local path or None."""
    fp = img_path(sample_id)
    if os.path.exists(fp): return fp
    if not isinstance(url, str) or not url.strip(): return None
    for k in range(retries):
        try:
            r = requests.get(url, headers=UA, timeout=timeout)
            if r.status_code == 200 and r.content:
                with open(fp, 'wb') as f:
                    f.write(r.content)
                return fp
        except Exception:
            pass
        time.sleep(sleep * (k+1))
    return None

def open_image(fp, size=224):
    """Open and RGB-convert; return PIL Image or None."""
    try:
        with Image.open(fp) as im:
            return im.convert('RGB')
    except Exception:
        return None

def build_quantity_mask(series):
    """Quick regex to detect presence of quantities; used to target image downloads."""
    s = series.fillna('').str.lower().str.replace('×','x')
    has_qty = s.str.contains(r'(\d+\s*x\s*\d+\.?\d*\s*(kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre|oz|fl oz))|(\d+\.?\d*\s*(kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre|oz|fl oz))', regex=True)
    return ~has_qty  # True where qty missing in text

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100


if 'image_link' not in df_all.columns:
    df_all = df_all.merge(
        pd.concat([
            train[['sample_id','image_link']].assign(is_train=1),
            test [['sample_id','image_link']].assign(is_train=0)
        ], ignore_index=True),
        on=['sample_id','is_train'],
        how='left'
    )

is_tr = (df_all['is_train'].values==1)
img_urls = df_all['image_link'].fillna('')


mask_missing_qty = build_quantity_mask(df_all['catalog_content'])


y = train['price'].astype(float).values
oof_price = np.exp(oof_stack_l).clip(0.01)
err = np.zeros(len(df_all)); err[:] = -1.0
err[:len(train)] = (np.abs(y - oof_price) / ((np.abs(y)+np.abs(oof_price))/2.0)) * 100


MAX_TRAIN_IMAGES = 20000
MAX_TEST_IMAGES  = 20000

tr_idx_all = np.where(is_tr)[0]
te_idx_all = np.where(~is_tr)[0]
tr_idx_missing = np.intersect1d(tr_idx_all, np.where(mask_missing_qty)[0])
te_idx_missing = np.intersect1d(te_idx_all, np.where(mask_missing_qty)[0])

# top error rows in train:
tr_top_err = tr_idx_all[np.argsort(-err[tr_idx_all])[:MAX_TRAIN_IMAGES]]

# union and cap
tr_targets = np.unique(np.concatenate([tr_idx_missing, tr_top_err]))[:MAX_TRAIN_IMAGES]
te_targets = te_idx_missing[:MAX_TEST_IMAGES]

print(f"Targeting images -> train: {len(tr_targets)} | test: {len(te_targets)}")

def batch_download(indices, max_workers=16):
    futs = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for idx in indices:
            sid = df_all.iloc[idx]['sample_id']
            url = img_urls.iloc[idx]
            futs[ex.submit(download_image, sid, url)] = sid
        done = 0
        for fut in as_completed(futs):
            _ = fut.result()
            done += 1
            if done % 1000 == 0:
                print(f"  downloaded {done}/{len(futs)}")
    print("Downloads done.")

print("Downloading train images...")
batch_download(tr_targets)
print("Downloading test images...")
batch_download(te_targets)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k', device=device)
model.eval()

to_encode = []
for idx in np.concatenate([tr_targets, te_targets]):
    sid = df_all.iloc[idx]['sample_id']
    if not os.path.exists(emb_path(sid)):
        to_encode.append(idx)

print(f"To encode (not cached): {len(to_encode)}")

# Simple dataloader-style loop
BATCH = 128 if device=='cuda' else 32
def encode_batch(idxs):
    ims, sids = [], []
    for idx in idxs:
        sid = df_all.iloc[idx]['sample_id']
        fp = img_path(sid)
        im = open_image(fp)
        if im is None:
            
            im = Image.new('RGB', (224,224), (0,0,0))
        ims.append(preprocess(im))
        sids.append(sid)
    with torch.no_grad():
        ims_t = torch.stack(ims).to(device)
        feat = model.encode_image(ims_t)
        feat = torch.nn.functional.normalize(feat, dim=1)
        feat = feat.float().cpu().numpy()
    for sid, vec in zip(sids, feat):
        np.save(emb_path(sid), vec.astype('float16'))

# encode in chunks
for i in range(0, len(to_encode), BATCH):
    encode_batch(to_encode[i:i+BATCH])
    if (i//BATCH) % 50 == 0:
        print(f"  encoded {min(i+BATCH, len(to_encode))}/{len(to_encode)}")



with torch.no_grad():
    dummy = model.encode_image(torch.zeros(1,3,224,224).to(device))
IMG_DIM = dummy.shape[1]
del dummy; gc.collect()

IMG_ALL = np.zeros((len(df_all), IMG_DIM), dtype='float32')
HAS_IMG = np.zeros(len(df_all), dtype='uint8')

for i in range(len(df_all)):
    sid = df_all.iloc[i]['sample_id']
    ep = emb_path(sid)
    if os.path.exists(ep):
        v = np.load(ep).astype('float32')
        IMG_ALL[i] = v
        HAS_IMG[i] = 1

IMG_TR = IMG_ALL[is_tr]
IMG_TE = IMG_ALL[~is_tr]
print("Image emb shapes:", IMG_TR.shape, IMG_TE.shape, "| filled rows:", HAS_IMG.sum())



has_img_tr = HAS_IMG[is_tr].reshape(-1,1).astype('float32')
has_img_te = HAS_IMG[~is_tr].reshape(-1,1).astype('float32')

X_tr = np.hstack([FE_TR, EMB_TR, oof_stack_l.reshape(-1,1), IMG_TR, has_img_tr]).astype('float32')
X_te = np.hstack([FE_TE, EMB_TE, te_stack_l.reshape(-1,1), IMG_TE, has_img_te]).astype('float32')

print("Final train/test shapes with images:", X_tr.shape, X_te.shape)

y_log = np.log(np.clip(y, 0.01, None))
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = dict(
    n_estimators=4000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    reg_lambda=1.0,
    objective='reg:squarederror',
    eval_metric='mae',
    tree_method='hist',
    device='cuda',
    random_state=42,
    early_stopping_rounds=200
)

oof_img_l = np.zeros(len(train), dtype='float32')
te_img_l  = np.zeros(len(test),  dtype='float32')

for f,(tr_idx, va_idx) in enumerate(skf.split(X_tr, bins)):
    model = xgb.XGBRegressor(**params)
    model.fit(
        X_tr[tr_idx], y_log[tr_idx],
        eval_set=[(X_tr[va_idx], y_log[va_idx])],
        verbose=False
    )
    oof_img_l[va_idx] = model.predict(X_tr[va_idx])
    te_img_l += model.predict(X_te) / skf.n_splits
    print(f"Fold {f} SMAPE:", f"{smape(y[va_idx], np.exp(oof_img_l[va_idx]).clip(0.01)):.3f}%")

cv = smape(y, np.exp(oof_img_l).clip(0.01))
print(f"STACKED + IMG XGB | CV SMAPE: {cv:.3f}%")

# save submission
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_img_l).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_stack_img.csv", index=False)
print("Saved -> out/test_predictions_stack_img.csv")


/tmp/ipykernel_227/1590457489.py:39: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_qty = s.str.contains(r'(\d+\s*x\s*\d+\.?\d*\s*(kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre|oz|fl oz))|(\d+\.?\d*\s*(kg|g|gm|gram|grams|ml|l|lt|ltr|liter|litre|oz|fl oz))', regex=True)


Targeting images -> train: 20000 | test: 20000
  downloaded 1000/20000
  downloaded 2000/20000
  downloaded 3000/20000
  downloaded 4000/20000
  downloaded 5000/20000
  downloaded 6000/20000
  downloaded 7000/20000
  downloaded 8000/20000
  downloaded 9000/20000
  downloaded 10000/20000
  downloaded 11000/20000
  downloaded 12000/20000
  downloaded 13000/20000
  downloaded 14000/20000
  downloaded 15000/20000
  downloaded 16000/20000
  downloaded 17000/20000
  downloaded 18000/20000
  downloaded 19000/20000
  downloaded 20000/20000
Downloads done.
  downloaded 1000/20000
  downloaded 2000/20000
  downloaded 3000/20000
  downloaded 4000/20000
  downloaded 5000/20000
  downloaded 6000/20000
  downloaded 7000/20000
  downloaded 8000/20000
  downloaded 9000/20000
  downloaded 10000/20000
  downloaded 11000/20000
  downloaded 12000/20000
  downloaded 13000/20000
  downloaded 14000/20000
  downloaded 15000/20000
  downloaded 16000/20000
  downloaded 17000/20000
  downloaded 18000/20000
  dow

open_clip_pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

To encode (not cached): 40000
  encoded 128/40000
  encoded 6528/40000
  encoded 12928/40000
  encoded 19328/40000
  encoded 25728/40000
  encoded 32128/40000
  encoded 38528/40000
Image emb shapes: (75000, 512) (75000, 512) | filled rows: 40000
Final train/test shapes with images: (75000, 1115) (75000, 1115)
Fold 0 SMAPE: 50.710%
Fold 1 SMAPE: 50.562%
Fold 2 SMAPE: 50.965%
Fold 3 SMAPE: 50.627%
Fold 4 SMAPE: 50.055%
STACKED + IMG XGB | CV SMAPE: 50.584%
Saved -> out/test_predictions_stack_img.csv


In [ ]:
import numpy as np, pandas as pd, os

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100

for name in ['y','test','oof_img_l','te_img_l','oof_stack_l','te_stack_l']:
    assert name in globals(), f"Missing {name}. Run previous cells first."

w_grid = np.linspace(0.0, 1.0, 21)  
best_w, best_cv = None, 1e9
for w in w_grid:
    oof_blend_price = np.exp(w*oof_img_l + (1-w)*oof_stack_l).clip(0.01)
    cv = smape(y, oof_blend_price)
    if cv < best_cv:
        best_cv, best_w = cv, w

print(f"Best weight on STACKED+IMG: w={best_w:.2f} | Blended CV SMAPE={best_cv:.3f}%")

te_blend_log = best_w*te_img_l + (1-best_w)*te_stack_l
sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_blend_log).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_stack_img_blend.csv", index=False)
print("Saved -> out/test_predictions_stack_img_blend.csv")


Best weight on STACKED+IMG: w=0.75 | Blended CV SMAPE=50.544%
Saved -> out/test_predictions_stack_img_blend.csv


In [ ]:

import numpy as np, pandas as pd, os

def smape(y_true, y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = denom!=0
    out = np.zeros_like(denom)
    out[m] = np.abs(y_true[m]-y_pred[m])/denom[m]
    return out.mean()*100


for name in ['y','test','oof_stack_l','te_stack_l','oof_img_l','te_img_l','df_all','HAS_IMG','mask_missing_qty']:
    assert name in globals(), f"Missing {name}"

is_tr = (df_all['is_train'].values==1)
mA_tr = (HAS_IMG[is_tr]==1) & (mask_missing_qty[is_tr])  
mB_tr = ~(mA_tr)                                         


w_grid = np.linspace(0, 1, 21)
def best_w(mask):
    best_w, best_cv = 0.0, 1e9
    for w in w_grid:
        oof_blend = np.exp(w*oof_img_l[mask] + (1-w)*oof_stack_l[mask]).clip(0.01)
        cv = smape(y[mask], oof_blend)
        if cv < best_cv:
            best_cv, best_w = cv, w
    return best_w, best_cv

wA, cvA = best_w(mA_tr)
wB, cvB = best_w(mB_tr)
print(f"Segment A (has_img & missing_qty): w={wA:.2f} | seg-CV={cvA:.3f}%  | size={mA_tr.sum()}")
print(f"Segment B (others):                 w={wB:.2f} | seg-CV={cvB:.3f}%  | size={mB_tr.sum()}")


oof_blend_full = np.exp(
    (wA*oof_img_l + (1-wA)*oof_stack_l) * mA_tr +
    (wB*oof_img_l + (1-wB)*oof_stack_l) * mB_tr
).clip(0.01)
cv_full = smape(y, oof_blend_full)
print(f"Segment-aware blended CV SMAPE: {cv_full:.3f}%")


mA_te = (HAS_IMG[~is_tr]==1) & (mask_missing_qty[~is_tr])
mB_te = ~(mA_te)

te_blend_log = np.zeros_like(te_stack_l)
te_blend_log[mA_te] = wA*te_img_l[mA_te] + (1-wA)*te_stack_l[mA_te]
te_blend_log[mB_te] = wB*te_img_l[mB_te] + (1-wB)*te_stack_l[mB_te]

sub = pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_blend_log).clip(0.01).astype(float)})
os.makedirs("out", exist_ok=True)
sub.to_csv("out/test_predictions_stack_img_segblend.csv", index=False)
print("Saved -> out/test_predictions_stack_img_segblend.csv")


Segment A (has_img & missing_qty): w=0.45 | seg-CV=52.545%  | size=13850
Segment B (others):                 w=1.00 | seg-CV=50.004%  | size=61150
Segment-aware blended CV SMAPE: 50.473%
Saved -> out/test_predictions_stack_img_segblend.csv


In [ ]:

!rm -rf /kaggle/working/images



# Check space
!du -sh /kaggle/working/* | sort -h || true
!df -h


17M	/kaggle/working/out
110M	/kaggle/working/text_emb_shards
158M	/kaggle/working/img_emb
649M	/kaggle/working/tfidf_artifacts
Filesystem                                                              Size  Used Avail Use% Mounted on
overlay                                                                 7.9T  6.4T  1.5T  81% /
tmpfs                                                                    64M     0   64M   0% /dev
shm                                                                      14G  4.0K   14G   1% /dev/shm
/dev/sdb1                                                               122G   36G   86G  30% /opt/bin
/dev/loop1                                                               20G  933M   19G   5% /kaggle/lib
192.168.5.2:/data/kagglesdsdata/datasets/8451073/13329800/ddfan3eq304r   93T   69T   25T  74% /kaggle/input/amazonml
/dev/mapper/snap                                                        7.9T  6.4T  1.5T  81% /etc/hosts
tmpfs                                  

In [36]:
!rm -rf /kaggle/working/img_emb

In [ ]:

import os, io, time, math, gc, socket, numpy as np, pandas as pd, requests, torch, open_clip, xgboost as xgb
from PIL import Image, ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    from urllib3.util import Retry  # older envs
from sklearn.model_selection import StratifiedKFold



In [ ]:

assert all(v in globals() for v in [
    'train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','oof_stack_l','te_stack_l','mask_missing_qty'
]), "Missing upstream vars. Run prior cells."


MM_PATH  = "/kaggle/working/img_emb_all.f16"
HAS_PATH = "/kaggle/working/img_has.npy"

os.system("rm -rf /kaggle/working/images /kaggle/working/img_emb")



0

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k', device=device)
model.eval()
with torch.no_grad():
    IMG_DIM = model.encode_image(torch.zeros(1,3,224,224, device=device)).shape[1]


N = len(df_all)
mode = 'r+' if os.path.exists(MM_PATH) else 'w+'
emb_mm = np.memmap(MM_PATH, dtype='float16', mode=mode, shape=(N, IMG_DIM))
has_img = np.load(HAS_PATH) if os.path.exists(HAS_PATH) else np.zeros(N, dtype=np.uint8)



In [ ]:

is_tr = (df_all['is_train'].values==1)
y = train['price'].astype(float).values
oof_price = np.exp(oof_stack_l).clip(0.01)

err = np.zeros(N); err[:] = -1.0
err[:len(train)] = (np.abs(y - oof_price) / ((np.abs(y)+np.abs(oof_price))/2.0)) * 100
img_urls = df_all['image_link'].fillna('')

need = np.where(has_img == 0)[0]
prio_missing_qty = need[mask_missing_qty[need]]
prio_high_err    = need[np.argsort(-err[need])]

TARGET_TRAIN = 40000   # tune if you want
TARGET_TEST  = 40000
tr_need = [i for i in prio_missing_qty if is_tr[i]] + [i for i in prio_high_err if is_tr[i]]
te_need = [i for i in prio_missing_qty if not is_tr[i]]
targets = np.unique(np.r_[tr_need[:TARGET_TRAIN], te_need[:TARGET_TEST]])

print(f"[plan] rows without image emb: {len(need)} | will encode now: {len(targets)}")
ImageFile.LOAD_TRUNCATED_IMAGES = True


UA = {"User-Agent": "Mozilla/5.0 (compatible; Kaggle/Images/1.0)"}
session = requests.Session()
retries = Retry(total=2, connect=2, read=2,
                status_forcelist=[429, 500, 502, 503, 504],
                backoff_factor=0.2, raise_on_status=False)
adapter = HTTPAdapter(max_retries=retries, pool_connections=64, pool_maxsize=64)
session.mount("http://", adapter); session.mount("https://", adapter)
socket.setdefaulttimeout(10)

def fetch_and_preprocess(idx):
    """Return (idx, tensor) or None."""
    url = img_urls.iloc[idx]
    if not isinstance(url, str) or not url:
        return None
    try:
        r = session.get(url, headers=UA, timeout=(5,10))
        if r.status_code != 200 or not r.content:
            return None
        with Image.open(io.BytesIO(r.content)) as im:
            im = im.convert("RGB")
        return (idx, preprocess(im))
    except Exception:
        return None



[plan] rows without image emb: 150000 | will encode now: 62498


In [ ]:

FETCH_CHUNK   = 2048    
FETCH_WORKERS = 32      
ENCODE_BATCH  = 128 if device=='cuda' else 32

t0 = time.time()
base_done = int(has_img.sum())
total_to_do = len(targets)
total_fetched = total_kept = total_encoded = 0

def pretty_eta(done, total, elapsed):
    rate = done / max(elapsed, 1e-6)
    rem  = total - done
    eta  = rem / max(rate, 1e-6)
    return f"{elapsed/60:.1f}m elapsed | {rate:.1f}/s | ETA {eta/60:.1f}m"



In [42]:
for start in range(0, total_to_do, FETCH_CHUNK):
    chunk = targets[start:start+FETCH_CHUNK]

    # 1) fetch concurrently
    idx_keep, tensors = [], []
    t_fetch0 = time.time()
    with ThreadPoolExecutor(max_workers=FETCH_WORKERS) as ex:
        futs = {ex.submit(fetch_and_preprocess, idx): idx for idx in chunk}
        for i, fut in enumerate(as_completed(futs), 1):
            res = fut.result()
            total_fetched += 1
            if res is not None:
                idx_keep.append(res[0]); tensors.append(res[1])
                total_kept += 1
            # live status during fetch (every ~200 completes)
            if i % 200 == 0:
                elapsed = time.time() - t0
                print(f"[fetch] {total_fetched}/{total_to_do} fetched | kept={total_kept} | {pretty_eta(total_encoded, total_to_do, elapsed)}")

    if not tensors:
        print("[fetch] no valid images in this chunk; continuing…")
        continue

    # 2) encode on GPU in mini-batches
    for b in range(0, len(tensors), ENCODE_BATCH):
        batch_t = torch.stack(tensors[b:b+ENCODE_BATCH]).to(device)
        with torch.no_grad():
            feat = model.encode_image(batch_t)
            feat = torch.nn.functional.normalize(feat, dim=1).float().cpu().numpy().astype('float16')
        idx_sub = idx_keep[b:b+ENCODE_BATCH]
        for j, idx in enumerate(idx_sub):
            emb_mm[idx, :IMG_DIM] = feat[j]
            has_img[idx] = 1
        total_encoded += len(idx_sub)

    # 3) persist periodically + live status
    emb_mm.flush(); np.save(HAS_PATH, has_img)
    elapsed = time.time() - t0
    print(f"[progress] encoded {total_encoded}/{total_to_do} (this wave kept {len(idx_keep)}) | {pretty_eta(total_encoded, total_to_do, elapsed)}")

print(f"[done] new vectors written: {total_encoded} | total rows with images now: {int(has_img.sum())}")



[fetch] 200/62498 fetched | kept=200 | 0.3m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 400/62498 fetched | kept=400 | 0.4m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 600/62498 fetched | kept=600 | 0.5m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 800/62498 fetched | kept=800 | 0.5m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 1000/62498 fetched | kept=1000 | 0.6m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 1200/62498 fetched | kept=1200 | 0.7m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 1400/62498 fetched | kept=1400 | 0.7m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 1600/62498 fetched | kept=1600 | 0.8m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 1800/62498 fetched | kept=1800 | 0.9m elapsed | 0.0/s | ETA 1041633333.3m
[fetch] 2000/62498 fetched | kept=2000 | 1.0m elapsed | 0.0/s | ETA 1041633333.3m
[progress] encoded 2048/62498 (this wave kept 2048) | 1.1m elapsed | 31.3/s | ETA 32.2m
[fetch] 2248/62498 fetched | kept=2248 | 1.2m elapsed | 29.3/s | ETA 34.4m
[fetch] 2448/62498 fetche

In [ ]:

IMG_TR = np.array(emb_mm[is_tr], dtype='float32')
IMG_TE = np.array(emb_mm[~is_tr], dtype='float32')
HAS_TR = has_img[is_tr].reshape(-1,1).astype('float32')
HAS_TE = has_img[~is_tr].reshape(-1,1).astype('float32')
print("Image emb shapes:", IMG_TR.shape, IMG_TE.shape, "| filled total:", int(has_img.sum()))


def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = d!=0
    out = np.zeros_like(d)
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

X_tr = np.hstack([FE_TR, EMB_TR, oof_stack_l.reshape(-1,1), IMG_TR, HAS_TR]).astype('float32')
X_te = np.hstack([FE_TE, EMB_TE, te_stack_l.reshape(-1,1), IMG_TE, HAS_TE]).astype('float32')
print("Final train/test shapes:", X_tr.shape, X_te.shape)

y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)
skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

params = dict(
    n_estimators=4000, max_depth=7, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=2, reg_lambda=1.0,
    objective='reg:squarederror', eval_metric='mae',
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200
)

oof_img2_l = np.zeros(len(train), dtype='float32')
te_img2_l  = np.zeros(len(test),  dtype='float32')
t_train0 = time.time()
for f,(tr_idx, va_idx) in enumerate(skf.split(X_tr, bins)):
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr[tr_idx], y_log[tr_idx], eval_set=[(X_tr[va_idx], y_log[va_idx])], verbose=False)
    oof_img2_l[va_idx] = m.predict(X_tr[va_idx])
    te_img2_l += m.predict(X_te) / skf.n_splits
    print(f"[fold {f}] SMAPE: {smape(y[va_idx], np.exp(oof_img2_l[va_idx]).clip(0.01)):.3f}% "
          f"| elapsed {(time.time()-t_train0)/60:.1f}m")

cv2 = smape(y, np.exp(oof_img2_l).clip(0.01))
print(f"STACKED + IMG (memmap, expanded) | CV SMAPE: {cv2:.3f}%")


out_path = "out/test_predictions_stack_img_expanded.csv"
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_img2_l).clip(0.01).astype(float)}).to_csv(out_path, index=False)
print(f"[saved] -> {out_path}")


Image emb shapes: (75000, 512) (75000, 512) | filled total: 62496
Final train/test shapes: (75000, 1115) (75000, 1115)
[fold 0] SMAPE: 50.083% | elapsed 0.5m
[fold 1] SMAPE: 50.221% | elapsed 0.9m
[fold 2] SMAPE: 50.579% | elapsed 1.4m
[fold 3] SMAPE: 50.035% | elapsed 1.9m
[fold 4] SMAPE: 49.633% | elapsed 2.4m
STACKED + IMG (memmap, expanded) | CV SMAPE: 50.110%
[saved] -> out/test_predictions_stack_img_expanded.csv


In [ ]:

import os, json, time, sys, hashlib, numpy as np, pandas as pd
import platform
try:
    import torch, xgboost as xgb
except Exception:
    torch, xgb = None, None


RUN_TAG = "stack_img_memmap_v1"   # <== change tag when you try new ideas
STAMP   = time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = f"/kaggle/working/runs/{STAMP}_{RUN_TAG}"
os.makedirs(RUN_DIR, exist_ok=True)
print("Run dir:", RUN_DIR)


def write_json(path, obj):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def md5_of_array(a: np.ndarray) -> str:
    m = hashlib.md5(); m.update(np.ascontiguousarray(a).tobytes()); return m.hexdigest()


assert 'train' in globals(), "need train df"
y = train['price'].astype(float).values
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = np.full(len(train), -1, dtype=np.int32)
dummyX = np.zeros((len(train), 1), dtype=np.float32)  
for k, (_, va) in enumerate(skf.split(dummyX, bins)):
    folds[va] = k
pd.DataFrame({'sample_id': train['sample_id'], 'fold': folds}).to_csv(f"{RUN_DIR}/folds.csv", index=False)


fe_cols = []
if 'FE_COLS' in globals():
    fe_cols = list(FE_COLS)
else:
    
    fe_cols = [f"fe_{i}" for i in range(FE_TR.shape[1])]

feature_meta = {
    "FE_TR_shape": list(FE_TR.shape),
    "FE_TE_shape": list(FE_TE.shape),
    "FE_cols_count": len(fe_cols),
    "EMB_TR_dim": int(EMB_TR.shape[1]),
    "EMB_TE_dim": int(EMB_TE.shape[1]),
}
write_json(f"{RUN_DIR}/feature_meta.json", feature_meta)
pd.Series(fe_cols).to_csv(f"{RUN_DIR}/fe_columns.csv", index=False, header=["feature"])


img_cov = {}
for n in ["IMG_TR","IMG_TE","HAS_TR","HAS_TE"]:
    if n in globals():
        arr = globals()[n]
        if n.startswith("HAS"):
            img_cov[n] = int(arr.sum())
        img_cov[n+"_shape"] = list(arr.shape)
if img_cov:
    write_json(f"{RUN_DIR}/image_coverage.json", img_cov)


xgb_params = {}
if 'params' in globals() and isinstance(params, dict):
    xgb_params = params.copy()
write_json(f"{RUN_DIR}/xgb_params.json", xgb_params)


ridge_info = {}
if 'best' in globals() and isinstance(best, dict) and 'alpha' in best:
    ridge_info = {"alpha": float(best['alpha'])}
write_json(f"{RUN_DIR}/ridge_info.json", ridge_info)


def save_oof(name, oof_log_arr):
    df = pd.DataFrame({
        "sample_id": train["sample_id"],
        "fold": folds,
        "oof_log": oof_log_arr,
        "oof_price": np.exp(oof_log_arr).clip(0.01)
    })
    df.to_csv(f"{RUN_DIR}/oof_{name}.csv", index=False)
    return {
        "name": name,
        "cv_smape": float(((np.abs(train['price']-df['oof_price']) /
                           ((np.abs(train['price'])+np.abs(df['oof_price']))/2.0)).mean()*100)),
        "oof_log_md5": md5_of_array(oof_log_arr.astype(np.float32))
    }

oof_summary = []

if 'oof_stack_l' in globals():
    oof_summary.append(save_oof("stack_only", oof_stack_l))
if 'oof_img2_l' in globals():  
    oof_summary.append(save_oof("stack_img_memmap", oof_img2_l))
if 'ridge_oof_log' in globals():
    oof_summary.append(save_oof("ridge_text", ridge_oof_log))

write_json(f"{RUN_DIR}/oof_summary.json", oof_summary)


sub_candidates = [
    "out/test_predictions_stack_img_expanded.csv",
    "out/test_predictions_stack_img_memmap.csv",
    "out/test_predictions_stack_img_segblend.csv",
    "out/test_predictions_stack_calibrated.csv",
    "out/test_predictions_stack.csv",
    "out/test_predictions_v2.csv",
]
sub_used = None
for p in sub_candidates:
    if os.path.exists(p):
        dst = os.path.join(RUN_DIR, os.path.basename(p))
        pd.read_csv(p).to_csv(dst, index=False)
        sub_used = dst
        break


manifest = {
    "run_tag": RUN_TAG,
    "timestamp": STAMP,
    "cv_scheme": {"n_splits": 5, "stratify": "price qcut(20)", "seed": 42},
    "metrics": {
        "cv_smape_stack_img_memmap": float(((np.abs(train['price']-np.exp(oof_img2_l).clip(0.01)) /
                                            ((np.abs(train['price'])+np.abs(np.exp(oof_img2_l).clip(0.01)))/2.0)).mean()*100)) if 'oof_img2_l' in globals() else None,
    },
    "shapes": {
        "train": [int(len(train))],
        "test":  [int(len(test))],
        "FE_TR": list(FE_TR.shape),
        "EMB_TR": list(EMB_TR.shape),
        "XGB_stack_img_memmap_exists": bool('oof_img2_l' in globals()),
    },
    "paths": {
        "run_dir": RUN_DIR,
        "submission": sub_used,
        "img_memmap": "/kaggle/working/img_emb_all.f16",
        "img_has": "/kaggle/working/img_has.npy",
    },
    "env": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": None if torch is None else torch.__version__,
        "xgboost": None if xgb is None else xgb.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    }
}
write_json(f"{RUN_DIR}/manifest.json", manifest)

with open("/kaggle/working/runs/LATEST.txt", "w") as f:
    f.write(RUN_DIR)

print("\nSaved snapshot.")
print("Files:")
for fn in sorted(os.listdir(RUN_DIR)):
    print(" -", fn)
print("\nLatest run recorded in: /kaggle/working/runs/LATEST.txt")


Run dir: /kaggle/working/runs/20251012-073513_stack_img_memmap_v1

Saved snapshot.
Files:
 - fe_columns.csv
 - feature_meta.json
 - folds.csv
 - image_coverage.json
 - manifest.json
 - oof_ridge_text.csv
 - oof_stack_img_memmap.csv
 - oof_stack_only.csv
 - oof_summary.json
 - ridge_info.json
 - test_predictions_stack_img_expanded.csv
 - xgb_params.json

Latest run recorded in: /kaggle/working/runs/LATEST.txt


In [ ]:

import os, json, numpy as np, pandas as pd

# --- utils
def smape(y_true, y_pred):
    d = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    out = np.zeros_like(d, dtype=float)
    m = d != 0
    out[m] = np.abs(y_true[m] - y_pred[m]) / d[m]
    return out.mean() * 100

def load_latest_run_dir():
    latest_ptr = "/kaggle/working/runs/LATEST.txt"
    assert os.path.exists(latest_ptr), "LATEST.txt not found; run the snapshot cell first."
    with open(latest_ptr) as f:
        rd = f.read().strip()
    assert os.path.isdir(rd), f"Run dir not found: {rd}"
    return rd


need_train = ['train', 'test', 'df_all', 'mask_missing_qty']
for n in need_train:
    assert n in globals(), f"Missing {n} in memory."

run_dir = load_latest_run_dir()


if 'oof_img2_l' in globals():
    oof_img_l = oof_img2_l.copy()
else:
    p = os.path.join(run_dir, "oof_stack_img_memmap.csv")
    df = pd.read_csv(p)
    oof_img_l = df['oof_log'].values.astype('float32')

# B) stack-only (no new images)
if 'oof_stack_l' in globals():
    oof_stack_base_l = oof_stack_l.copy()
else:
    p = os.path.join(run_dir, "oof_stack_only.csv")
    df = pd.read_csv(p)
    oof_stack_base_l = df['oof_log'].values.astype('float32')


if 'te_img2_l' in globals():
    te_img_l = te_img2_l.copy()
else:
    
    p = os.path.join(run_dir, "test_predictions_stack_img_expanded.csv")
    te_img_price = pd.read_csv(p)['price'].values
    te_img_l = np.log(np.clip(te_img_price, 0.01, None)).astype('float32')


if 'te_stack_l' in globals():
    te_stack_base_l = te_stack_l.copy()
else:
    
    p_guess = "/kaggle/working/out/test_predictions_stack.csv"
    assert os.path.exists(p_guess), "te_stack_l not in memory and base stack CSV not found."
    te_stack_price = pd.read_csv(p_guess)['price'].values
    te_stack_base_l = np.log(np.clip(te_stack_price, 0.01, None)).astype('float32')


is_tr = (df_all['is_train'].values == 1)
is_te = ~is_tr


HAS_PATH = "/kaggle/working/img_has.npy"
assert os.path.exists(HAS_PATH), "img_has.npy not found; run the memmap image step."
has_all = np.load(HAS_PATH)
HAS_TR = has_all[is_tr].astype(bool)
HAS_TE = has_all[is_te].astype(bool)


miss_all = mask_missing_qty.astype(bool)
MISS_TR = miss_all[is_tr]
MISS_TE = miss_all[is_te]

SEG_A_TR = HAS_TR & MISS_TR
SEG_B_TR = ~SEG_A_TR
SEG_A_TE = HAS_TE & MISS_TE
SEG_B_TE = ~SEG_A_TE

y = train['price'].astype(float).values
p_img_tr = np.exp(oof_img_l).clip(0.01)
p_base_tr = np.exp(oof_stack_base_l).clip(0.01)
p_img_te = np.exp(te_img_l).clip(0.01)
p_base_te = np.exp(te_stack_base_l).clip(0.01)

ws = np.linspace(0.0, 1.0, 51)  
best = {}

def best_w_for_mask(mask, name):
    if mask.sum() == 0:
        return 1.0, np.nan  
    y_seg = y[mask]
    pA = p_img_tr[mask]; pB = p_base_tr[mask]
    scores = []
    for w in ws:
        pred = w*pA + (1-w)*pB
        scores.append(smape(y_seg, pred))
    i = int(np.argmin(scores))
    return float(ws[i]), float(scores[i])

wA, segA_cv = best_w_for_mask(SEG_A_TR, "A")
wB, segB_cv = best_w_for_mask(SEG_B_TR, "B")


p_blend_tr = np.empty_like(p_img_tr)
p_blend_tr[SEG_A_TR] = wA*p_img_tr[SEG_A_TR] + (1-wA)*p_base_tr[SEG_A_TR]
p_blend_tr[SEG_B_TR] = wB*p_img_tr[SEG_B_TR] + (1-wB)*p_base_tr[SEG_B_TR]
cv_all = smape(y, p_blend_tr)


p_blend_te = np.empty_like(p_img_te)
p_blend_te[SEG_A_TE] = wA*p_img_te[SEG_A_TE] + (1-wA)*p_base_te[SEG_A_TE]
p_blend_te[SEG_B_TE] = wB*p_img_te[SEG_B_TE] + (1-wB)*p_base_te[SEG_B_TE]


print(f"Segment A (has_img & missing_qty): w={wA:.2f} | seg-CV={segA_cv:.3f}% | size={int(SEG_A_TR.sum())}")
print(f"Segment B (others):                 w={wB:.2f} | seg-CV={segB_cv:.3f}% | size={int(SEG_B_TR.sum())}")
print(f"Segment-aware blended CV SMAPE: {cv_all:.3f}%")

out_dir = "/kaggle/working/out"; os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "test_predictions_segblend_latest.csv")
pd.DataFrame({'sample_id': test['sample_id'], 'price': p_blend_te.astype(float)}).to_csv(out_path, index=False)
print("Saved ->", out_path)



Segment A (has_img & missing_qty): w=0.42 | seg-CV=52.118% | size=27443
Segment B (others):                 w=1.00 | seg-CV=48.516% | size=47557
Segment-aware blended CV SMAPE: 49.834%
Saved -> /kaggle/working/out/test_predictions_segblend_latest.csv


In [ ]:

import numpy as np, pandas as pd, time, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100


for v in ['train','test','FE_TR','FE_TE','EMB_TR','EMB_TE','oof_stack_l','te_stack_l','IMG_TR','IMG_TE','HAS_TR','HAS_TE']:
    assert v in globals(), f"Missing {v}"

X_tr = np.hstack([
    np.asarray(FE_TR, dtype='float32'),
    np.asarray(EMB_TR, dtype='float32'),
    np.asarray(oof_stack_l, dtype='float32').reshape(-1,1),
    np.asarray(IMG_TR, dtype='float32'),
    np.asarray(HAS_TR, dtype='float32').reshape(-1,1),
]).astype('float32')

X_te = np.hstack([
    np.asarray(FE_TE, dtype='float32'),
    np.asarray(EMB_TE, dtype='float32'),
    np.asarray(te_stack_l, dtype='float32').reshape(-1,1),
    np.asarray(IMG_TE, dtype='float32'),
    np.asarray(HAS_TE, dtype='float32').reshape(-1,1),
]).astype('float32')

y = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins = pd.qcut(y, q=20, duplicates='drop').astype(str)

# Best params from your sweep
params_best = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae',
    objective='reg:absoluteerror', max_depth=8, min_child_weight=2, learning_rate=0.03,
)

# Train CV to get OOF + test
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_best_l = np.zeros(len(y_log), dtype='float32')
te_best_l  = np.zeros(X_te.shape[0], dtype='float32')

t0 = time.time()
for f,(tr, va) in enumerate(skf.split(X_tr, bins)):
    m = xgb.XGBRegressor(**params_best)
    m.fit(X_tr[tr], y_log[tr], eval_set=[(X_tr[va], y_log[va])], verbose=False)
    oof_best_l[va] = m.predict(X_tr[va])
    te_best_l += m.predict(X_te) / skf.n_splits
    print(f"[fold {f}] SMAPE: {smape(y[va], np.exp(oof_best_l[va]).clip(0.01)):.3f}%")

cv_best = smape(y, np.exp(oof_best_l).clip(0.01))
print(f"\nPromoted best config CV SMAPE: {cv_best:.3f}% (prev blended: ~49.807%)")

# Save artifacts
import os
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_log': oof_best_l, 'oof_price': np.exp(oof_best_l).clip(0.01)}).to_csv(
    "out/oof_xgb_best_promoted.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'], 'price': np.exp(te_best_l).clip(0.01).astype(float)}).to_csv(
    "out/test_predictions_xgb_best_promoted.csv", index=False)
print("Saved -> out/oof_xgb_best_promoted.csv")
print("Saved -> out/test_predictions_xgb_best_promoted.csv")

# Make these the active “image-augmented” model outputs for later steps
oof_img2_l = oof_best_l.copy()
te_img2_l  = te_best_l.copy()


[fold 0] SMAPE: 49.718%
[fold 1] SMAPE: 49.772%
[fold 2] SMAPE: 50.153%
[fold 3] SMAPE: 49.574%
[fold 4] SMAPE: 49.209%

Promoted best config CV SMAPE: 49.685% (prev blended: ~49.807%)
Saved -> out/oof_xgb_best_promoted.csv
Saved -> out/test_predictions_xgb_best_promoted.csv


In [48]:
# =======================
# Segment-aware blend (PROMOTED model) — with live status & timings
# =======================
import os, time, json, numpy as np, pandas as pd

t0 = time.time()
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    out = np.zeros_like(d, dtype=float)
    m = d != 0
    out[m] = np.abs(y_true[m] - y_pred[m]) / d[m]
    return out.mean() * 100

def load_latest_run_dir():
    ptr = "/kaggle/working/runs/LATEST.txt"
    if os.path.exists(ptr):
        with open(ptr) as f:
            rd = f.read().strip()
        if os.path.isdir(rd):
            return rd
    return None

def load_oof_log_from_csv(path, col='oof_log'):
    df = pd.read_csv(path)
    assert col in df.columns, f"{col} not found in {path}"
    return df[col].values.astype('float32')

def load_test_price_from_csv(path):
    df = pd.read_csv(path)
    assert 'price' in df.columns, f"'price' not in {path}"
    return df['price'].values.astype('float32')

# ---- prerequisites
for n in ['train','test','df_all','mask_missing_qty']:
    assert n in globals(), f"Missing {n}. Re-run earlier setup cells."

log("Starting segment-aware blend (promoted)")

# ---- Load Model A (promoted best) OOF/preds (LOG scale for OOF, price for test)
if 'oof_img2_l' in globals() and 'te_img2_l' in globals():
    log("Using in-memory promoted OOF/test")
    oofA_log = np.asarray(oof_img2_l, dtype='float32')
    teA_log  = np.asarray(te_img2_l,  dtype='float32')  # log-space
else:
    log("Loading promoted OOF/test from disk")
    oof_path = "out/oof_xgb_best_promoted.csv"
    te_path  = "out/test_predictions_xgb_best_promoted.csv"
    assert os.path.exists(oof_path) and os.path.exists(te_path), "Promoted outputs not found; run the promote cell first."
    oofA_log = load_oof_log_from_csv(oof_path, col='oof_log')
    teA_price = load_test_price_from_csv(te_path)
    teA_log = np.log(np.clip(teA_price, 0.01, None))

# ---- Load Model B (base stack-only) OOF/preds
run_dir = load_latest_run_dir()
oofB_log = None
if 'oof_stack_l' in globals():
    oofB_log = np.asarray(oof_stack_l, dtype='float32')
    log("Using in-memory stack-only OOF")
elif run_dir and os.path.exists(os.path.join(run_dir, "oof_stack_only.csv")):
    p = os.path.join(run_dir, "oof_stack_only.csv")
    oofB_log = load_oof_log_from_csv(p, col='oof_log')
    log(f"Loaded base OOF from {p}")
else:
    raise SystemExit("Base stack-only OOF not found in memory or runs/. You can rerun the stack-only OOF cell or point me to its CSV.")

if 'te_stack_l' in globals():
    teB_log = np.asarray(te_stack_l, dtype='float32')
    log("Using in-memory stack-only TEST (log)")
else:
    # fall back to an earlier submission if needed (price -> log)
    guess = "/kaggle/working/out/test_predictions_stack.csv"
    assert os.path.exists(guess), "Stack-only test predictions not found; produce te_stack_l or the CSV."
    teB_price = load_test_price_from_csv(guess)
    teB_log = np.log(np.clip(teB_price, 0.01, None))
    log(f"Loaded base TEST from {guess}")

# ---- Build masks
is_tr = (df_all['is_train'].values == 1)
is_te = ~is_tr

HAS_PATH = "/kaggle/working/img_has.npy"
assert os.path.exists(HAS_PATH), "img_has.npy not found; run the image memmap step earlier."
has_all = np.load(HAS_PATH).astype(bool)
MISS_all = mask_missing_qty.astype(bool)

SEG_A_TR = has_all[is_tr] & MISS_all[is_tr]
SEG_B_TR = ~SEG_A_TR
SEG_A_TE = has_all[is_te] & MISS_all[is_te]
SEG_B_TE = ~SEG_A_TE
log(f"Segments ready | train A={int(SEG_A_TR.sum())}, B={int(SEG_B_TR.sum())} | test A={int(SEG_A_TE.sum())}, B={int(SEG_B_TE.sum())}")

# ---- Convert OOF to price-space
y = train['price'].astype(float).values
pA_tr = np.exp(oofA_log).clip(0.01); pB_tr = np.exp(oofB_log).clip(0.01)
pA_te = np.exp(teA_log).clip(0.01);  pB_te = np.exp(teB_log).clip(0.01)

# ---- Grid search weights with live prints
ws = np.linspace(0.0, 1.0, 51)
def best_w(mask, name):
    if mask.sum() == 0:
        log(f"Segment {name}: empty, defaulting w=1.00")
        return 1.0, np.nan
    y_seg, pA, pB = y[mask], pA_tr[mask], pB_tr[mask]
    best = (None, 1e9)
    t_seg = time.time()
    for i, w in enumerate(ws, 1):
        pred = w*pA + (1-w)*pB
        s = smape(y_seg, pred)
        if s < best[1]:
            best = (float(w), float(s))
        if i % 10 == 0:
            log(f"  {name}: scanned {i}/{len(ws)} weights; best so far w={best[0]:.2f} | SMAPE={best[1]:.3f}%")
    log(f"{name}: done search in +{time.time()-t_seg:.1f}s | best w={best[0]:.2f} | seg-CV={best[1]:.3f}%")
    return best

log("Searching weight for Segment A (has_img & missing_qty)")
wA, segA_cv = best_w(SEG_A_TR, "Segment A")

log("Searching weight for Segment B (others)")
wB, segB_cv = best_w(SEG_B_TR, "Segment B")

# ---- Build blended TRAIN and report overall CV
p_blend_tr = np.empty_like(pA_tr)
p_blend_tr[SEG_A_TR] = wA*pA_tr[SEG_A_TR] + (1-wA)*pB_tr[SEG_A_TR]
p_blend_tr[SEG_B_TR] = wB*pA_tr[SEG_B_TR] + (1-wB)*pB_tr[SEG_B_TR]
cv_all = smape(y, p_blend_tr)
log(f"Overall blended CV SMAPE: {cv_all:.3f}%")

# ---- Apply to TEST and save
p_blend_te = np.empty_like(pA_te)
p_blend_te[SEG_A_TE] = wA*pA_te[SEG_A_TE] + (1-wA)*pB_te[SEG_A_TE]
p_blend_te[SEG_B_TE] = wB*pA_te[SEG_B_TE] + (1-wB)*pB_te[SEG_B_TE]

out_dir = "/kaggle/working/out"; os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "test_predictions_segblend_promoted.csv")
pd.DataFrame({'sample_id': test['sample_id'], 'price': p_blend_te.astype(float)}).to_csv(out_path, index=False)
log(f"Saved submission -> {out_path}")

# ---- Save meta
meta = {
    "wA_hasimg_missingqty": wA, "segA_cv": segA_cv, "segA_size": int(SEG_A_TR.sum()),
    "wB_others": wB, "segB_cv": segB_cv, "segB_size": int(SEG_B_TR.sum()),
    "cv_all": float(cv_all),
    "oofA": "promoted_best", "oofB": "stack_only",
}
with open(os.path.join(out_dir, "segblend_promoted_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
log("Saved segblend metadata JSON")


[07:51:40] Starting segment-aware blend (promoted) | +0.0s
[07:51:40] Using in-memory promoted OOF/test | +0.0s
[07:51:40] Using in-memory stack-only OOF | +0.0s
[07:51:40] Using in-memory stack-only TEST (log) | +0.0s
[07:51:40] Segments ready | train A=27443, B=47557 | test A=27341, B=47659 | +0.0s
[07:51:40] Searching weight for Segment A (has_img & missing_qty) | +0.0s
[07:51:40]   Segment A: scanned 10/51 weights; best so far w=0.18 | SMAPE=52.255% | +0.0s
[07:51:40]   Segment A: scanned 20/51 weights; best so far w=0.38 | SMAPE=51.992% | +0.0s
[07:51:40]   Segment A: scanned 30/51 weights; best so far w=0.58 | SMAPE=51.881% | +0.0s
[07:51:40]   Segment A: scanned 40/51 weights; best so far w=0.64 | SMAPE=51.876% | +0.0s
[07:51:40]   Segment A: scanned 50/51 weights; best so far w=0.64 | SMAPE=51.876% | +0.0s
[07:51:40] Segment A: done search in +0.0s | best w=0.64 | seg-CV=51.876% | +0.0s
[07:51:40] Searching weight for Segment B (others) | +0.0s
[07:51:40]   Segment B: scanned 1

In [53]:
import os, glob, zipfile

print("Top-level /kaggle/input contents:", os.listdir("/kaggle/input"))

def discover_e5_dirs(roots):
    hits = []
    for root in roots:
        for p in glob.glob(os.path.join(root, "*")):
            if os.path.isdir(p):
                # search depth<=3 for model files
                for pat in ("**/model.safetensors", "**/pytorch_model.bin"):
                    for f in glob.glob(os.path.join(p, pat), recursive=True):
                        d = os.path.dirname(f)
                        tok_ok = os.path.exists(os.path.join(d, "tokenizer.json")) or \
                                 os.path.exists(os.path.join(d, "vocab.txt"))
                        if tok_ok:
                            hits.append(d)
            elif p.lower().endswith(".zip"):
                # unzip once into working dir so we can use it
                outdir = "/kaggle/working/_unzipped_" + os.path.splitext(os.path.basename(p))[0]
                if not os.path.exists(outdir):
                    print(f"Unzipping: {p} -> {outdir}")
                    with zipfile.ZipFile(p) as z:
                        z.extractall(outdir)
                # search inside unzipped
                hits += discover_e5_dirs([outdir])
    return sorted(set(hits))

cands = discover_e5_dirs(["/kaggle/input"])
print("E5-like candidate directories (should contain model + tokenizer):")
for i, p in enumerate(cands):
    print(f"  [{i}] {p}")

if cands:
    E5_DIR = cands[0]
    print("Using E5_DIR =", E5_DIR)
else:
    print("\nNo candidates found. Make sure you clicked **Add data** in the right sidebar, "
          "added your E5-large dataset to THIS notebook, or upload a zip of the model.")


Top-level /kaggle/input contents: ['amazonml', 'e5-large-v2']
E5-like candidate directories (should contain model + tokenizer):
  [0] /kaggle/input/e5-large-v2/e5-large-v2
Using E5_DIR = /kaggle/input/e5-large-v2/e5-large-v2


In [56]:
# E5-large embeddings → memmap with resume + optional 2-GPU encoding
import os, time, json, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

assert 'df_all' in globals() and 'catalog_content' in df_all.columns and 'is_train' in df_all.columns

E5_DIR = "/kaggle/input/e5-large-v2/e5-large-v2"
MM_E5  = "/kaggle/working/text_e5_emb_all.f16"
META   = MM_E5 + ".meta.json"   # sidecar for progress

# Offline + tokenizer threads (kept conservative for multi-proc)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def log(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}")

N = len(df_all)
texts = df_all['catalog_content'].fillna('').astype(str).tolist()

# Devices / model
ngpu = torch.cuda.device_count()
targets = [f"cuda:{i}" for i in range(ngpu)] if ngpu else ["cpu"]
model = SentenceTransformer(E5_DIR, device=targets[0] if targets else "cpu")
model.max_seq_length = max(getattr(model, "max_seq_length", 512), 512)
e5_dim = model.get_sentence_embedding_dimension()
log(f"E5 dim={e5_dim} | GPUs: {targets}")

# Create/open memmap
if not os.path.exists(MM_E5):
    log(f"Creating memmap: {MM_E5} for N={N}")
    mm = np.memmap(MM_E5, dtype='float16', mode='w+', shape=(N, e5_dim))
    start = 0
else:
    mm = np.memmap(MM_E5, dtype='float16', mode='r+', shape=(N, e5_dim))
    start = 0
    # Prefer META; fallback to scan
    if os.path.exists(META):
        try:
            meta = json.loads(open(META).read())
            if meta.get("N")==N and meta.get("dim")==e5_dim:
                start = int(meta.get("done", 0))
        except Exception:
            pass
    if start == 0:
        log("Scanning memmap to detect progress...")
        # Any non-zero in a row ⇒ already written (fast with memmap)
        start = int(np.any(mm != 0, axis=1).sum())
    log(f"Resuming from row {start}/{N}")

# Tunables
BATCH = (64 if ngpu >= 2 else (128 if ngpu == 1 else 32))
SHARD = 10_000

def write_meta(done_rows:int):
    with open(META, "w") as f:
        json.dump({"N": N, "dim": e5_dim, "done": int(done_rows)}, f)

# Encode
if ngpu >= 2:
    pool = model.start_multi_process_pool(target_devices=targets[:2])  # use two GPUs
    try:
        for s in range(start, N, SHARD):
            sub = texts[s:s+SHARD]
            # NOTE: encode_multi_process has no 'show_progress_bar' arg in your version
            emb = model.encode_multi_process(
                sub, pool, batch_size=BATCH, normalize_embeddings=True
            )
            mm[s:s+len(sub)] = emb.astype('float16')
            mm.flush(); write_meta(s+len(sub))
            log(f"encoded {min(s+len(sub), N)}/{N} (multi-GPU)")
    finally:
        model.stop_multi_process_pool(pool)
else:
    with torch.no_grad():
        for s in range(start, N, SHARD):
            sub = texts[s:s+SHARD]
            emb = model.encode(
                sub, batch_size=BATCH, convert_to_numpy=True,
                normalize_embeddings=True, show_progress_bar=True
            )
            mm[s:s+len(sub)] = emb.astype('float16')
            mm.flush(); write_meta(s+len(sub))
            log(f"encoded {min(s+len(sub), N)}/{N}")

# Expose arrays for downstream use
is_tr = (df_all['is_train'].values == 1)
E5_TR = np.array(mm[is_tr], dtype='float32')
E5_TE = np.array(mm[~is_tr], dtype='float32')
log(f"E5 ready | E5_TR {E5_TR.shape} | E5_TE {E5_TE.shape}")


[09:01:38] E5 dim=1024 | GPUs: ['cuda:0', 'cuda:1']
[09:01:38] Scanning memmap to detect progress...
[09:01:39] Resuming from row 40000/150000


2025-10-12 09:01:45.592603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760259705.615020    2518 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760259705.621995    2518 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-12 09:01:54.616867: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760259714.638969    2530 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760259714.645826    2530 cuda_blas.cc:1

[09:06:10] encoded 50000/150000 (multi-GPU)
[09:10:35] encoded 60000/150000 (multi-GPU)
[09:15:01] encoded 70000/150000 (multi-GPU)
[09:19:30] encoded 80000/150000 (multi-GPU)
[09:23:56] encoded 90000/150000 (multi-GPU)
[09:28:17] encoded 100000/150000 (multi-GPU)
[09:32:40] encoded 110000/150000 (multi-GPU)
[09:37:05] encoded 120000/150000 (multi-GPU)
[09:41:34] encoded 130000/150000 (multi-GPU)
[09:45:54] encoded 140000/150000 (multi-GPU)
[09:50:17] encoded 150000/150000 (multi-GPU)
[09:50:18] E5 ready | E5_TR (75000, 1024) | E5_TE (75000, 1024)


In [60]:
# =============== E5-fused head with rich live status ===============
import os, sys, gc, time, json, numpy as np, pandas as pd, xgboost as xgb

# optional: free encoder VRAM if it’s still around
try:
    import torch
    try: del e5
    except NameError: pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except Exception:
    pass

from sklearn.model_selection import StratifiedKFold

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def to2(a):
    a = np.asarray(a)
    return (a if a.ndim==2 else a.reshape(-1,1)).astype('float32', copy=False)

# ---- sanity: required arrays must exist
need = ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','E5_TR','E5_TE',
        'oof_stack_l','te_stack_l','IMG_TR','IMG_TE','HAS_TR','HAS_TE']
for n in need: 
    assert n in globals(), f"Missing {n}"

# ---- build matrices
FE_TR_f, FE_TE_f   = to2(FE_TR), to2(FE_TE)
EMB_TR_f, EMB_TE_f = to2(EMB_TR), to2(EMB_TE)     # MiniLM
E5_TR_f,  E5_TE_f  = to2(E5_TR),  to2(E5_TE)      # E5-large
IMG_TR_f, IMG_TE_f = to2(IMG_TR), to2(IMG_TE)     # OpenCLIP (B/32)
HAS_TR_f, HAS_TE_f = to2(HAS_TR), to2(HAS_TE)     # has_img flag
OOF_TR_f, TE_TR_f  = to2(oof_stack_l), to2(te_stack_l)  # prior stack log

X_tr = np.hstack([FE_TR_f, EMB_TR_f, E5_TR_f, OOF_TR_f, IMG_TR_f, HAS_TR_f]).astype('float32')
X_te = np.hstack([FE_TE_f, EMB_TE_f, E5_TE_f, TE_TR_f,  IMG_TE_f, HAS_TE_f]).astype('float32')

y     = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)

# ---- params (same as your best promoted config family)
params = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae',
    objective='reg:absoluteerror', max_depth=8, min_child_weight=2, learning_rate=0.03,
)

# ---- live status helpers
t0 = time.time()
def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg} | +{time.time()-t0:.1f}s")
    sys.stdout.flush()

# GPU/shape summary
try:
    import torch
    if torch.cuda.is_available():
        devs = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        log(f"GPUs: {devs}")
    else:
        log("GPU not detected; running on CPU for XGBoost.")
except Exception:
    pass

def gb(nbytes): return f"{nbytes/1024/1024/1024:.2f} GB"
log(f"E5-fused matrices: X_tr={X_tr.shape} ({gb(X_tr.nbytes)}), X_te={X_te.shape} ({gb(X_te.nbytes)})")

# ---- 5-fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_e5_l = np.zeros_like(y_log, dtype='float32')
te_e5_l  = np.zeros(X_te.shape[0], dtype='float32')

log("Starting CV training...")
for f,(tr, va) in enumerate(skf.split(X_tr, bins)):
    fold_t = time.time()
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr[tr], y_log[tr], eval_set=[(X_tr[va], y_log[va])], verbose=False)
    oof_e5_l[va] = m.predict(X_tr[va])
    te_e5_l += m.predict(X_te) / skf.n_splits

    fold_smape = smape(y[va], np.exp(oof_e5_l[va]).clip(0.01))
    best_iter  = getattr(m, "best_iteration", None)
    log(f"[fold {f}] SMAPE={fold_smape:.3f}% | best_iter={best_iter} | fold_time={time.time()-fold_t:.1f}s")

cv_e5 = smape(y, np.exp(oof_e5_l).clip(0.01))
log(f"E5-FUSED CV SMAPE: {cv_e5:.3f}%")

# ---- save artifacts
os.makedirs("out", exist_ok=True)
pd.DataFrame({
    'sample_id': train['sample_id'],
    'oof_log': oof_e5_l,
    'oof_price': np.exp(oof_e5_l).clip(0.01)
}).to_csv("out/oof_e5_fused.csv", index=False)

pd.DataFrame({
    'sample_id': test['sample_id'],
    'price': np.exp(te_e5_l).clip(0.01).astype(float)
}).to_csv("out/test_predictions_e5_fused.csv", index=False)

meta = {
    "cv_smape": float(cv_e5),
    "X_shapes": {"train": list(X_tr.shape), "test": list(X_te.shape)},
    "params": {k: params[k] for k in ['objective','max_depth','min_child_weight','learning_rate']},
}
with open("out/e5_fused_meta.json","w") as f:
    json.dump(meta, f, indent=2)

log("Saved -> out/oof_e5_fused.csv, out/test_predictions_e5_fused.csv, out/e5_fused_meta.json")

# expose for blending
oof_e5_l = oof_e5_l.copy()
te_e5_l  = te_e5_l.copy()
# ================================================================


[09:59:00] GPUs: ['Tesla T4', 'Tesla T4'] | +0.0s
[09:59:00] E5-fused matrices: X_tr=(75000, 2139) (0.60 GB), X_te=(75000, 2139) (0.60 GB) | +0.0s
[09:59:00] Starting CV training... | +0.0s
[10:01:48] [fold 0] SMAPE=49.824% | best_iter=814 | fold_time=168.0s | +168.1s
[10:05:07] [fold 1] SMAPE=49.643% | best_iter=971 | fold_time=198.7s | +366.8s
[10:08:23] [fold 2] SMAPE=50.153% | best_iter=1031 | fold_time=196.1s | +562.9s
[10:11:38] [fold 3] SMAPE=49.587% | best_iter=964 | fold_time=194.5s | +757.5s
[10:14:23] [fold 4] SMAPE=49.321% | best_iter=841 | fold_time=165.0s | +922.4s
[10:14:23] E5-FUSED CV SMAPE: 49.706% | +922.4s
[10:14:23] Saved -> out/oof_e5_fused.csv, out/test_predictions_e5_fused.csv, out/e5_fused_meta.json | +922.7s


In [61]:
# ============================
# Segment-aware blend: E5-fused (A) vs Promoted Best (B)
# Seg A = has_img & missing_qty ; Seg B = others
# ============================
import os, time, json, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

t0 = time.time()
log = lambda msg: print(f"[{time.strftime('%H:%M:%S')}] {msg} | +{time.time()-t0:.1f}s")

# --- inputs: E5-fused must be in memory from the previous cell
assert 'oof_e5_l' in globals() and 'te_e5_l' in globals(), "Run the E5-fused training cell first."

# Model A (E5-fused)
oofA_log = oof_e5_l.astype('float32')
teA_log  = te_e5_l.astype('float32')

# Model B (promoted best). Use memory if present; else load from files.
if 'oof_img2_l' in globals() and 'te_img2_l' in globals():
    oofB_log = oof_img2_l.astype('float32')
    teB_log  = te_img2_l.astype('float32')
else:
    oofB_log = pd.read_csv("out/oof_xgb_best_promoted.csv")['oof_log'].values.astype('float32')
    teB_price = pd.read_csv("out/test_predictions_xgb_best_promoted.csv")['price'].values
    teB_log   = np.log(np.clip(teB_price, 0.01, None)).astype('float32')

# Masks for segments
assert 'df_all' in globals() and 'mask_missing_qty' in globals(), "Need df_all & mask_missing_qty."
HAS = np.load("/kaggle/working/img_has.npy").astype(bool)
MISS = mask_missing_qty.astype(bool)
is_tr = (df_all['is_train'].values==1)
is_te = ~is_tr

SEG_A_TR = HAS[is_tr] & MISS[is_tr]   # has_img & missing_qty
SEG_B_TR = ~SEG_A_TR
SEG_A_TE = HAS[is_te] & MISS[is_te]
SEG_B_TE = ~SEG_A_TE

log(f"Segments ready | train A={int(SEG_A_TR.sum())}, B={int(SEG_B_TR.sum())} | "
    f"test A={int(SEG_A_TE.sum())}, B={int(SEG_B_TE.sum())}")

# Convert OOF/TEST to price space
y = train['price'].astype(float).values
pA_tr, pB_tr = np.exp(oofA_log).clip(0.01), np.exp(oofB_log).clip(0.01)
pA_te, pB_te = np.exp(teA_log).clip(0.01),  np.exp(teB_log).clip(0.01)

# Grid-search weights per segment
ws = np.linspace(0.0, 1.0, 51)

def search_weight(mask, name):
    if mask.sum()==0:
        log(f"{name}: empty segment; using w=1.00")
        return 1.0, np.nan
    y_seg, a, b = y[mask], pA_tr[mask], pB_tr[mask]
    best_w, best_cv = 0.0, 1e9
    for i,w in enumerate(ws, 1):
        s = smape(y_seg, w*a + (1-w)*b)
        if s < best_cv: best_w, best_cv = float(w), float(s)
        if i % 10 == 0:
            log(f"  {name}: scanned {i}/{len(ws)}; best so far w={best_w:.2f} | seg-CV={best_cv:.3f}%")
    log(f"{name}: best w={best_w:.2f} | seg-CV={best_cv:.3f}% | size={int(mask.sum())}")
    return best_w, best_cv

log("Searching weights...")
wA, segA_cv = search_weight(SEG_A_TR, "Segment A (has_img & missing_qty)")
wB, segB_cv = search_weight(SEG_B_TR, "Segment B (others)")

# Build blended OOF + report overall CV
p_blend_tr = np.empty_like(pA_tr)
p_blend_tr[SEG_A_TR] = wA*pA_tr[SEG_A_TR] + (1-wA)*pB_tr[SEG_A_TR]
p_blend_tr[SEG_B_TR] = wB*pA_tr[SEG_B_TR] + (1-wB)*pB_tr[SEG_B_TR]
cv_all = smape(y, p_blend_tr)
log(f"Overall blended CV SMAPE: {cv_all:.3f}%")

# Apply to test
p_blend_te = np.empty_like(pA_te)
p_blend_te[SEG_A_TE] = wA*pA_te[SEG_A_TE] + (1-wA)*pB_te[SEG_A_TE]
p_blend_te[SEG_B_TE] = wB*pA_te[SEG_B_TE] + (1-wB)*pB_te[SEG_B_TE]

# Save
os.makedirs("out", exist_ok=True)
out_path = "out/test_predictions_segblend_e5_vs_promoted.csv"
pd.DataFrame({'sample_id': test['sample_id'], 'price': p_blend_te.astype(float)}).to_csv(out_path, index=False)

meta = {
    "wA": wA, "segA_cv": float(segA_cv),
    "wB": wB, "segB_cv": float(segB_cv),
    "cv_all": float(cv_all),
    "sizes": {"A_tr": int(SEG_A_TR.sum()), "B_tr": int(SEG_B_TR.sum()),
              "A_te": int(SEG_A_TE.sum()), "B_te": int(SEG_B_TE.sum())}
}
with open("out/segblend_e5_vs_promoted_meta.json","w") as f:
    json.dump(meta, f, indent=2)

log(f"Saved -> {out_path}")


[10:16:44] Segments ready | train A=27443, B=47557 | test A=27341, B=47659 | +0.0s
[10:16:44] Searching weights... | +0.0s
[10:16:44]   Segment A (has_img & missing_qty): scanned 10/51; best so far w=0.18 | seg-CV=51.945% | +0.0s
[10:16:44]   Segment A (has_img & missing_qty): scanned 20/51; best so far w=0.38 | seg-CV=51.835% | +0.0s
[10:16:44]   Segment A (has_img & missing_qty): scanned 30/51; best so far w=0.54 | seg-CV=51.809% | +0.0s
[10:16:44]   Segment A (has_img & missing_qty): scanned 40/51; best so far w=0.54 | seg-CV=51.809% | +0.0s
[10:16:44]   Segment A (has_img & missing_qty): scanned 50/51; best so far w=0.54 | seg-CV=51.809% | +0.0s
[10:16:44] Segment A (has_img & missing_qty): best w=0.54 | seg-CV=51.809% | size=27443 | +0.0s
[10:16:44]   Segment B (others): scanned 10/51; best so far w=0.18 | seg-CV=48.261% | +0.0s
[10:16:44]   Segment B (others): scanned 20/51; best so far w=0.32 | seg-CV=48.258% | +0.1s
[10:16:44]   Segment B (others): scanned 30/51; best so far w=

In [63]:
# ============================
# Epsilon-floor tuning on blended predictions (fast)
# ============================
import os, json, time, numpy as np, pandas as pd

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

# --- Try to reuse in-memory blended preds from your previous cell ---
need_vars = all(v in globals() for v in [
    'train','test','p_blend_tr','p_blend_te'
])
if not need_vars:
    log("In-memory blended arrays not found. Rebuilding from saved meta...")
    # Recreate the blend using saved weights/meta + OOF/Test preds from disk if needed
    assert 'df_all' in globals(), "Need df_all for segment masks."
    y = train['price'].astype(float).values
    HAS = np.load("/kaggle/working/img_has.npy").astype(bool)
    MISS = mask_missing_qty.astype(bool)
    is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
    SEG_A_TR = HAS[is_tr] & MISS[is_tr]; SEG_B_TR = ~SEG_A_TR
    SEG_A_TE = HAS[is_te] & MISS[is_te]; SEG_B_TE = ~SEG_A_TE

    # Load meta (weights) and model outputs
    meta = json.load(open("out/segblend_e5_vs_promoted_meta.json"))
    wA, wB = meta['wA'], meta['wB']
    # E5
    oofA_log = pd.read_csv("out/oof_e5_fused.csv")['oof_log'].values.astype('float32')
    teA_log  = np.log(np.clip(pd.read_csv("out/test_predictions_e5_fused.csv")['price'].values, 0.01, None)).astype('float32')
    # Promoted best
    oofB_log = pd.read_csv("out/oof_xgb_best_promoted.csv")['oof_log'].values.astype('float32')
    teB_log  = np.log(np.clip(pd.read_csv("out/test_predictions_xgb_best_promoted.csv")['price'].values, 0.01, None)).astype('float32')

    pA_tr, pB_tr = np.exp(oofA_log).clip(0.01), np.exp(oofB_log).clip(0.01)
    pA_te, pB_te = np.exp(teA_log).clip(0.01),  np.exp(teB_log).clip(0.01)

    p_blend_tr = np.empty_like(pA_tr)
    p_blend_te = np.empty_like(pA_te)
    p_blend_tr[SEG_A_TR] = wA*pA_tr[SEG_A_TR] + (1-wA)*pB_tr[SEG_A_TR]
    p_blend_tr[SEG_B_TR] = wB*pA_tr[SEG_B_TR] + (1-wB)*pB_tr[SEG_B_TR]
    p_blend_te[SEG_A_TE] = wA*pA_te[SEG_A_TE] + (1-wA)*pB_te[SEG_A_TE]
    p_blend_te[SEG_B_TE] = wB*pA_te[SEG_B_TE] + (1-wB)*pB_te[SEG_B_TE]
else:
    log("Using in-memory blended predictions.")
    y = train['price'].astype(float).values

# --- ε grid (small to moderate floors; adjust if needed)
eps_grid = [0.0005, 0.001, 0.002, 0.005,
            0.010, 0.020, 0.050, 0.100]

log("Scanning epsilon floors...")
best = {"eps": None, "cv": 1e9}
for i,eps in enumerate(eps_grid, 1):
    cv = smape(y, np.maximum(p_blend_tr, eps))
    if cv < best["cv"]:
        best = {"eps": eps, "cv": cv}
    if i % 2 == 0:
        log(f"  scanned {i}/{len(eps_grid)} | best so far eps={best['eps']} CV={best['cv']:.3f}%")

log(f"Best epsilon: {best['eps']} | CV={best['cv']:.3f}%")

# --- apply to TEST and save
p_final_te = np.maximum(p_blend_te, best['eps']).astype(float)
os.makedirs("out", exist_ok=True)
out_eps = f"out/test_predictions_segblend_eps{best['eps']:.4f}.csv"
pd.DataFrame({'sample_id': test['sample_id'], 'price': p_final_te}).to_csv(out_eps, index=False)
log(f"Saved -> {out_eps}")


[10:28:16] Using in-memory blended predictions. | +0.0s
[10:28:16] Scanning epsilon floors... | +0.0s
[10:28:16]   scanned 2/8 | best so far eps=0.0005 CV=49.557% | +0.0s
[10:28:16]   scanned 4/8 | best so far eps=0.0005 CV=49.557% | +0.0s
[10:28:16]   scanned 6/8 | best so far eps=0.0005 CV=49.557% | +0.0s
[10:28:16]   scanned 8/8 | best so far eps=0.0005 CV=49.557% | +0.0s
[10:28:16] Best epsilon: 0.0005 | CV=49.557% | +0.0s
[10:28:16] Saved -> out/test_predictions_segblend_eps0.0005.csv | +0.2s


In [66]:
# ============================
# Build df_all['brand'] if missing, then Brand OOF prior (+freq) and retrain E5-fused head
# ============================
import re, time, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def to2(a):
    a = np.asarray(a)
    return (a if a.ndim==2 else a.reshape(-1,1)).astype('float32', copy=False)

# ---- prerequisites already built in your session
need = ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','E5_TR','E5_TE',
        'oof_stack_l','te_stack_l','IMG_TR','IMG_TE','HAS_TR','HAS_TE']
for n in need: 
    assert n in globals(), f"Missing {n}"

# ---------------------------
# 1) Build df_all['brand'] if missing
# ---------------------------
if 'brand' not in df_all.columns:
    log("brand column missing → extracting from catalog_content...")
    STOP = set("""
        item items product products pack packs case cases pallet bundle set bottle bottles jar can cans pouch bag bags
        count ct ounce ounces oz fl ml g kg lb l litre liter lt cu cup cups piece pieces stick sticks bar bars
        assorted variety classic original extra family value new new! size sizes large small medium mini maxi
        instant decaf decaffeinated sugar-free sugarfree gluten-free glutenfree non-gmo nongmo organic vegan keto
        for with and or the a an of by from
    """.split())

    def extract_brand_one(s: str) -> str:
        s = s or ""
        s_low = s.lower()

        # 1) get a title-like chunk (after "Item Name:" if present)
        m = re.search(r'item\s*name:\s*([^\n\r]+)', s_low)
        title = m.group(1) if m else s_low.split('\n',1)[0]

        # 2) strip after common separators to keep the leading namey bit
        title = re.split(r'\b(bullet point|product description|value:|unit:)\b', title)[0]

        # 3) remove bracket content & punctuation we don't want
        title = re.sub(r'[\(\)\[\]\{\}]', ' ', title)
        title = re.sub(r'[^a-z0-9&\'\-\s\.]', ' ', title)

        # 4) remove obvious size/unit phrases
        title = re.sub(r'\b\d+(\.\d+)?\s*(oz|ounce|ounces|fl\s*oz|ml|l|lt|ltr|liter|litre|g|kg|lb|pound|count|ct)\b', ' ', title)
        title = re.sub(r'\b(pack|case|pallet|set)\s*of\s*\d+\b', ' ', title)
        title = re.sub(r'\b\d+\s*[x×]\s*\d+(\.\d+)?\b', ' ', title)

        # 5) try "by <brand>" and "from <brand>"
        m_by   = re.search(r'\bby\s+([a-z0-9&\'\-]+(?:\s+[a-z0-9&\'\-]+){0,2})', title)
        m_from = re.search(r'\bfrom\s+([a-z0-9&\'\-]+(?:\s+[a-z0-9&\'\-]+){0,2})', title)
        cand = (m_by.group(1) if m_by else (m_from.group(1) if m_from else None))

        def clean_cand(x: str) -> str:
            x = re.sub(r'\s+', ' ', x.strip())
            toks = [t for t in x.split() if t not in STOP and not t.isdigit()]
            if not toks: return ""
            # merge if looks like two-part brand (e.g., "king arthur")
            if len(toks) >= 2 and len(toks[0]) >= 2 and len(toks[1]) >= 2:
                return (toks[0] + " " + toks[1])[:50]
            return toks[0][:50]

        if cand:
            b = clean_cand(cand)
            if b and b not in STOP and b != 'item':
                return b

        # 6) fallback: leading tokens until a stop/number/unit word
        toks = [t for t in title.split() if t and t not in STOP]
        # strip leading numbers/dots
        toks = [t for t in toks if not re.fullmatch(r'\d+(\.\d+)?', t)]
        if not toks:
            return 'other'
        # take first 1–2 alphabetic-ish tokens as brand
        take = []
        for t in toks:
            if re.fullmatch(r"[a-z0-9&'\-]+", t) and t not in STOP:
                take.append(t)
            else:
                break
            if len(take) == 2:
                break
        if not take:
            return 'other'
        b = " ".join(take)
        if b in STOP or b == 'item':
            return 'other'
        return b[:50]

    df_all['brand'] = df_all['catalog_content'].astype(str).map(extract_brand_one)
    cov = (df_all['brand'] != 'other').mean()*100
    log(f"brand extraction done. coverage (brand != 'other'): {cov:.1f}%")
else:
    log("brand column already present. Skipping extraction.")

# ---------------------------
# 2) Brand OOF prior (+freq) and retrain E5-fused head
# ---------------------------
is_tr = (df_all['is_train'].values==1)
brand_tr = df_all.loc[is_tr, 'brand'].fillna('other').astype(str).str.lower().str.strip().str.slice(0,50)
brand_te = df_all.loc[~is_tr, 'brand'].fillna('other').astype(str).str.lower().str.strip().str.slice(0,50)

# shared codes to keep things compact (no leakage; just an index)
all_brand = pd.Categorical(pd.concat([brand_tr, brand_te], axis=0))
b_tr = all_brand.codes[:len(brand_tr)]
b_te = all_brand.codes[len(brand_tr):]

y     = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)

log("Computing OOF brand priors...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
brand_oof = np.zeros_like(y_log, dtype='float32')

for f,(tr_idx, va_idx) in enumerate(skf.split(b_tr, bins)):
    gmed = float(np.median(y_log[tr_idx]))
    df_fold = pd.DataFrame({'b': b_tr[tr_idx], 'y': y_log[tr_idx]})
    med = df_fold.groupby('b')['y'].median()
    cnt = df_fold['b'].value_counts()
    alpha = 20
    prior = {}
    for bid, m in med.items():
        n = int(cnt.get(bid, 0))
        prior[int(bid)] = float((n*m + alpha*gmed) / (n + alpha))
    # fill validation split
    out = np.empty(len(va_idx), dtype='float32')
    for i, bid in enumerate(b_tr[va_idx]):
        out[i] = prior.get(int(bid), gmed)
    brand_oof[va_idx] = out
    log(f"  fold {f}: filled {len(va_idx)}")

# test prior using full train
gmed_full = float(np.median(y_log))
df_full = pd.DataFrame({'b': b_tr, 'y': y_log})
med_full = df_full.groupby('b')['y'].median()
cnt_full = df_full['b'].value_counts()
alpha = 20

brand_te_prior = np.empty(len(b_te), dtype='float32')
for i, bid in enumerate(b_te):
    bid_i = int(bid)
    if bid_i in med_full.index:
        n = int(cnt_full.get(bid_i, 0))
        brand_te_prior[i] = float((n*med_full.loc[bid_i] + alpha*gmed_full) / (n + alpha))
    else:
        brand_te_prior[i] = gmed_full

# brand frequency (stability)
brand_tr_cnt = np.log1p(cnt_full.reindex(b_tr).fillna(0).astype('float32')).values
brand_te_cnt = np.log1p(cnt_full.reindex(b_te).fillna(0).astype('float32')).values

# append to FE
FE_TR = np.hstack([to2(FE_TR), brand_oof.reshape(-1,1), brand_tr_cnt.reshape(-1,1)])
FE_TE = np.hstack([to2(FE_TE), brand_te_prior.reshape(-1,1), brand_te_cnt.reshape(-1,1)])
log(f"New FE shapes -> train={FE_TR.shape}, test={FE_TE.shape}")

# retrain the E5-fused head once to check lift
X_tr = np.hstack([to2(FE_TR), to2(EMB_TR), to2(E5_TR), to2(oof_stack_l), to2(IMG_TR), to2(HAS_TR)])
X_te = np.hstack([to2(FE_TE), to2(EMB_TE), to2(E5_TE), to2(te_stack_l),  to2(IMG_TE), to2(HAS_TE)])
log(f"Matrices ready: X_tr={X_tr.shape}, X_te={X_te.shape}")

params = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae', objective='reg:absoluteerror',
    max_depth=8, min_child_weight=2, learning_rate=0.03,
)

oof_e5brand_l = np.zeros_like(y_log, dtype='float32')
te_e5brand_l  = np.zeros(X_te.shape[0], dtype='float32')

log("Training E5-fused head with brand priors...")
for f,(tr_idx, va_idx) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_tr, bins)):
    m = xgb.XGBRegressor(**params)
    t_fold = time.time()
    m.fit(X_tr[tr_idx], y_log[tr_idx], eval_set=[(X_tr[va_idx], y_log[va_idx])], verbose=False)
    oof_e5brand_l[va_idx] = m.predict(X_tr[va_idx])
    te_e5brand_l += m.predict(X_te)/5
    print(f"  fold {f} SMAPE={smape(y[va_idx], np.exp(oof_e5brand_l[va_idx]).clip(0.01)):.3f}% | {time.time()-t_fold:.1f}s")

cv_brand = smape(y, np.exp(oof_e5brand_l).clip(0.01))
log(f"[E5 + Brand prior] CV SMAPE: {cv_brand:.3f}%")

# expose for the next blending step
oof_e5_l = oof_e5brand_l.copy()
te_e5_l  = te_e5brand_l.copy()


[10:31:59] brand column missing → extracting from catalog_content... | +0.0s
[10:32:04] brand extraction done. coverage (brand != 'other'): 99.2% | +4.9s
[10:32:04] Computing OOF brand priors... | +5.0s
[10:32:04]   fold 0: filled 15000 | +5.2s
[10:32:04]   fold 1: filled 15000 | +5.3s
[10:32:04]   fold 2: filled 15000 | +5.4s
[10:32:04]   fold 3: filled 15000 | +5.5s
[10:32:05]   fold 4: filled 15000 | +5.5s
[10:32:05] New FE shapes -> train=(75000, 219), test=(75000, 219) | +6.3s
[10:32:06] Matrices ready: X_tr=(75000, 2141), X_te=(75000, 2141) | +6.9s
[10:32:06] Training E5-fused head with brand priors... | +6.9s
  fold 0 SMAPE=48.950% | 144.4s
  fold 1 SMAPE=48.720% | 206.0s
  fold 2 SMAPE=49.223% | 150.4s
  fold 3 SMAPE=48.694% | 166.0s
  fold 4 SMAPE=48.545% | 172.0s
[10:46:05] [E5 + Brand prior] CV SMAPE: 48.827% | +845.8s


In [68]:
# ==========================================
# Segment-aware blend (E5+Brand vs Promoted) + ε-floor scan
# Saves: blended CSV + ε-tuned CSV + meta JSON
# ==========================================
import os, json, time, numpy as np, pandas as pd

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# ---- prerequisites
for v in ['train','test','df_all','mask_missing_qty']:
    assert v in globals(), f"Missing {v}"

y = train['price'].astype(float).values

# ---- bring in E5+Brand OOF/TEST (prefer in-memory; else load from disk)
use_mem_e5 = ('oof_e5_l' in globals()) and ('te_e5_l' in globals())
if use_mem_e5:
    log("Using in-memory E5+Brand predictions.")
    oofA_log = np.asarray(oof_e5_l, dtype='float32')
    teA_log  = np.asarray(te_e5_l,  dtype='float32')
else:
    log("Loading E5+Brand predictions from disk.")
    oofA_log = pd.read_csv("out/oof_e5_brand.csv")['oof_log'].values.astype('float32')
    teA      = pd.read_csv("out/test_predictions_e5_brand.csv")['price'].values.astype('float32')
    teA_log  = np.log(np.clip(teA, 0.01, None)).astype('float32')

# ---- bring in promoted-best OOF/TEST
if os.path.exists("out/oof_xgb_best_promoted.csv"):
    oofB_log = pd.read_csv("out/oof_xgb_best_promoted.csv")['oof_log'].values.astype('float32')
    teB      = pd.read_csv("out/test_predictions_xgb_best_promoted.csv")['price'].values.astype('float32')
    teB_log  = np.log(np.clip(teB, 0.01, None)).astype('float32')
else:
    # fallback to previous “promoted” variable names if you kept them in memory
    assert 'oof_img2_l' in globals() and 'te_img2_l' in globals(), "Need promoted-best outputs."
    oofB_log = np.asarray(oof_img2_l, dtype='float32')
    teB_log  = np.asarray(te_img2_l,  dtype='float32')

# convert to price space
pA_tr = np.exp(oofA_log).clip(0.01)
pB_tr = np.exp(oofB_log).clip(0.01)
pA_te = np.exp(teA_log).clip(0.01)
pB_te = np.exp(teB_log).clip(0.01)

# ---- segment masks
is_tr = (df_all['is_train'].values==1)
is_te = ~is_tr
HAS   = np.load("/kaggle/working/img_has.npy").astype(bool) if os.path.exists("/kaggle/working/img_has.npy") else np.zeros(len(df_all), bool)

SEG_A_TR = HAS[is_tr] & mask_missing_qty[is_tr]            # has_img & missing_qty (train)
SEG_B_TR = ~SEG_A_TR
SEG_A_TE = HAS[is_te] & mask_missing_qty[is_te]            # has_img & missing_qty (test)
SEG_B_TE = ~SEG_A_TE

log(f"Segments ready | train A={SEG_A_TR.sum()}, B={SEG_B_TR.sum()} | test A={SEG_A_TE.sum()}, B={SEG_B_TE.sum()}")

# ---- grid search weights for each segment
w_grid = np.linspace(0.0, 1.0, 51)  # 0.00 .. 1.00 step 0.02
bestA = {'w': None, 'cv': 1e9}
bestB = {'w': None, 'cv': 1e9}

log("Searching weights for Segment A...")
for i,w in enumerate(w_grid, 1):
    p = np.empty_like(pA_tr)
    p[SEG_A_TR] = w*pA_tr[SEG_A_TR] + (1-w)*pB_tr[SEG_A_TR]
    p[SEG_B_TR] = pB_tr[SEG_B_TR]   # untouched here
    cv = smape(y, p)
    if cv < bestA['cv']:
        bestA = {'w': float(w), 'cv': float(cv)}
    if i % 10 == 0:
        log(f"  A: scanned {i}/{len(w_grid)}; best w={bestA['w']:.2f} | seg-CV={bestA['cv']:.3f}%")
log(f"Segment A best w={bestA['w']:.2f} | seg-CV={bestA['cv']:.3f}%")

log("Searching weights for Segment B...")
for i,w in enumerate(w_grid, 1):
    p = np.empty_like(pA_tr)
    p[SEG_B_TR] = w*pA_tr[SEG_B_TR] + (1-w)*pB_tr[SEG_B_TR]
    p[SEG_A_TR] = pA_tr[SEG_A_TR]   # untouched here
    cv = smape(y, p)
    if cv < bestB['cv']:
        bestB = {'w': float(w), 'cv': float(cv)}
    if i % 10 == 0:
        log(f"  B: scanned {i}/{len(w_grid)}; best w={bestB['w']:.2f} | seg-CV={bestB['cv']:.3f}%")
log(f"Segment B best w={bestB['w']:.2f} | seg-CV={bestB['cv']:.3f}%")

# ---- build blended OOF/TEST with best weights
p_blend_tr = np.empty_like(pA_tr)
p_blend_te = np.empty_like(pA_te)

p_blend_tr[SEG_A_TR] = bestA['w']*pA_tr[SEG_A_TR] + (1-bestA['w'])*pB_tr[SEG_A_TR]
p_blend_tr[SEG_B_TR] = bestB['w']*pA_tr[SEG_B_TR] + (1-bestB['w'])*pB_tr[SEG_B_TR]

p_blend_te[SEG_A_TE] = bestA['w']*pA_te[SEG_A_TE] + (1-bestA['w'])*pB_te[SEG_A_TE]
p_blend_te[SEG_B_TE] = bestB['w']*pA_te[SEG_B_TE] + (1-bestB['w'])*pB_te[SEG_B_TE]

cv_blend = smape(y, p_blend_tr)
log(f"Overall blended CV SMAPE: {cv_blend:.3f}%")

# ---- save blended submission + meta
os.makedirs("out", exist_ok=True)
sub_path = "out/test_predictions_segblend_e5brand_vs_promoted.csv"
pd.DataFrame({'sample_id': test['sample_id'], 'price': p_blend_te.astype(float)}).to_csv(sub_path, index=False)

meta = {
    "cv_blend": cv_blend,
    "wA_has_img_and_missing_qty": bestA['w'],
    "wB_others": bestB['w'],
    "train_sizes": {"A": int(SEG_A_TR.sum()), "B": int(SEG_B_TR.sum())},
    "test_sizes":  {"A": int(SEG_A_TE.sum()), "B": int(SEG_B_TE.sum())},
    "sources": {
        "e5_brand_oof": use_mem_e5,
        "promoted_best_from_disk": os.path.exists("out/oof_xgb_best_promoted.csv")
    }
}
with open("out/segblend_e5brand_vs_promoted_meta.json","w") as f:
    json.dump(meta, f, indent=2)
log(f"Saved -> {sub_path} and segblend_e5brand_vs_promoted_meta.json")

# ---- ε-floor scan (tiny safeguard for tiny prices)
eps_grid = [0.0005, 0.001, 0.002, 0.005, 0.010, 0.020, 0.050, 0.100]
best_eps = {"eps": None, "cv": 1e9}
for i,eps in enumerate(eps_grid, 1):
    cv = smape(y, np.maximum(p_blend_tr, eps))
    if cv < best_eps["cv"]:
        best_eps = {"eps": float(eps), "cv": float(cv)}
    if i % 2 == 0:
        log(f"  ε-scan {i}/{len(eps_grid)} | best eps={best_eps['eps']} CV={best_eps['cv']:.3f}%")

p_eps_te = np.maximum(p_blend_te, best_eps["eps"]).astype(float)
eps_path = f"out/test_predictions_segblend_e5brand_vs_promoted_eps{best_eps['eps']:.4f}.csv"
pd.DataFrame({'sample_id': test['sample_id'], 'price': p_eps_te}).to_csv(eps_path, index=False)
log(f"Saved -> {eps_path}")


[10:57:12] Using in-memory E5+Brand predictions. | +0.0s
[10:57:12] Segments ready | train A=27443, B=47557 | test A=27341, B=47659 | +0.1s
[10:57:12] Searching weights for Segment A... | +0.1s
[10:57:12]   A: scanned 10/51; best w=0.18 | seg-CV=49.539% | +0.1s
[10:57:12]   A: scanned 20/51; best w=0.38 | seg-CV=49.420% | +0.1s
[10:57:12]   A: scanned 30/51; best w=0.58 | seg-CV=49.342% | +0.2s
[10:57:12]   A: scanned 40/51; best w=0.78 | seg-CV=49.309% | +0.2s
[10:57:12]   A: scanned 50/51; best w=0.82 | seg-CV=49.308% | +0.2s
[10:57:12] Segment A best w=0.82 | seg-CV=49.308% | +0.2s
[10:57:12] Searching weights for Segment B... | +0.2s
[10:57:12]   B: scanned 10/51; best w=0.18 | seg-CV=49.176% | +0.3s
[10:57:12]   B: scanned 20/51; best w=0.38 | seg-CV=49.041% | +0.3s
[10:57:12]   B: scanned 30/51; best w=0.58 | seg-CV=48.938% | +0.4s
[10:57:12]   B: scanned 40/51; best w=0.78 | seg-CV=48.866% | +0.4s
[10:57:12]   B: scanned 50/51; best w=0.98 | seg-CV=48.828% | +0.4s
[10:57:12] Seg

In [69]:
# ===========================================
# Step: E5 KMeans cluster prior (+freq, +min-dist) -> retrain E5-fused head
# ===========================================
import time, json, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.cluster import KMeans

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def to2(a):
    a = np.asarray(a)
    return (a if a.ndim==2 else a.reshape(-1,1)).astype('float32', copy=False)

# ---- prerequisites
need = ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','E5_TR','E5_TE',
        'oof_stack_l','te_stack_l','IMG_TR','IMG_TE','HAS_TR','HAS_TE']
for n in need: assert n in globals(), f"Missing {n}"

y     = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)

# ---- 1) KMeans on E5 (fit on train only to be extra safe)
K = 64
log(f"Fitting KMeans (K={K}) on E5_TR={E5_TR.shape}...")
km = KMeans(n_clusters=K, random_state=42, n_init='auto', max_iter=300, verbose=0)
km.fit(E5_TR)
lab_tr = km.labels_.astype(int)
lab_te = km.predict(E5_TE).astype(int)
log("KMeans done.")

# min distance to assigned centroid (stability signal)
log("Computing min-centroid distances...")
D_tr = km.transform(E5_TR)  # (Ntr, K) distances
D_te = km.transform(E5_TE)
min_tr = D_tr[np.arange(D_tr.shape[0]), lab_tr]
min_te = D_te[np.arange(D_te.shape[0]), lab_te]

# ---- 2) OOF cluster prior (smoothed median log-price) + cluster freq
log("Building OOF cluster priors...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clust_oof = np.zeros_like(y_log, dtype='float32')

for f,(tr_idx, va_idx) in enumerate(skf.split(lab_tr, bins)):
    gmed = float(np.median(y_log[tr_idx]))
    df_fold = pd.DataFrame({'c': lab_tr[tr_idx], 'y': y_log[tr_idx]})
    med = df_fold.groupby('c')['y'].median()
    cnt = df_fold['c'].value_counts()
    alpha = 20
    # smoothed map
    prior = {}
    for cid, m in med.items():
        n = int(cnt.get(cid, 0))
        prior[int(cid)] = float((n*m + alpha*gmed)/(n+alpha))
    out = np.empty(len(va_idx), dtype='float32')
    for i, cid in enumerate(lab_tr[va_idx]):
        out[i] = prior.get(int(cid), gmed)
    clust_oof[va_idx] = out
    log(f"  fold {f}: filled {len(va_idx)}")

# test prior using full train
log("Computing full-train cluster priors for test...")
gmed_full = float(np.median(y_log))
df_full = pd.DataFrame({'c': lab_tr, 'y': y_log})
med_full = df_full.groupby('c')['y'].median()
cnt_full = df_full['c'].value_counts()
alpha = 20

clust_te_prior = np.empty(len(lab_te), dtype='float32')
for i, cid in enumerate(lab_te):
    if cid in med_full.index:
        n = int(cnt_full.get(cid, 0))
        clust_te_prior[i] = float((n*med_full.loc[cid] + alpha*gmed_full)/(n+alpha))
    else:
        clust_te_prior[i] = gmed_full

# cluster frequency feature
clust_tr_cnt = np.log1p(cnt_full.reindex(lab_tr).fillna(0).astype('float32')).values
clust_te_cnt = np.log1p(cnt_full.reindex(lab_te).fillna(0).astype('float32')).values

# ---- 3) Append to FE
FE_TR = np.hstack([to2(FE_TR),
                   clust_oof.reshape(-1,1),
                   clust_tr_cnt.reshape(-1,1),
                   to2(min_tr)])
FE_TE = np.hstack([to2(FE_TE),
                   clust_te_prior.reshape(-1,1),
                   clust_te_cnt.reshape(-1,1),
                   to2(min_te)])
log(f"New FE shapes -> train={FE_TR.shape}, test={FE_TE.shape}")

# ---- 4) Retrain E5-fused head with added cluster features
X_tr = np.hstack([to2(FE_TR), to2(EMB_TR), to2(E5_TR), to2(oof_stack_l), to2(IMG_TR), to2(HAS_TR)])
X_te = np.hstack([to2(FE_TE), to2(EMB_TE), to2(E5_TE), to2(te_stack_l),  to2(IMG_TE), to2(HAS_TE)])
log(f"Matrices ready: X_tr={X_tr.shape}, X_te={X_te.shape}")

params = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae', objective='reg:absoluteerror',
    max_depth=8, min_child_weight=2, learning_rate=0.03,
)

oof_e5clu_l = np.zeros_like(y_log, dtype='float32')
te_e5clu_l  = np.zeros(X_te.shape[0], dtype='float32')

log("Training E5-fused head with cluster priors...")
for f,(tr, va) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X_tr, bins)):
    m = xgb.XGBRegressor(**params)
    t_fold = time.time()
    m.fit(X_tr[tr], y_log[tr], eval_set=[(X_tr[va], y_log[va])], verbose=False)
    oof_e5clu_l[va] = m.predict(X_tr[va])
    te_e5clu_l += m.predict(X_te)/5
    print(f"  fold {f} SMAPE={smape(y[va], np.exp(oof_e5clu_l[va]).clip(0.01)):.3f}% | {time.time()-t_fold:.1f}s")

cv_clu = smape(y, np.exp(oof_e5clu_l).clip(0.01))
log(f"[E5 + Brand + Cluster prior] CV SMAPE: {cv_clu:.3f}%")

# expose for optional blending
oof_e5_l = oof_e5clu_l.copy()
te_e5_l  = te_e5clu_l.copy()

# optional save
import os
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_log': oof_e5clu_l,
              'oof_price': np.exp(oof_e5clu_l).clip(0.01)}).to_csv("out/oof_e5_brand_cluster.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],
              'price': np.exp(te_e5clu_l).clip(0.01).astype(float)}).to_csv("out/test_predictions_e5_brand_cluster.csv", index=False)
with open("out/cluster_prior_meta.json","w") as f:
    json.dump({"K": K, "alpha": 20}, f)
log("Saved -> out/oof_e5_brand_cluster.csv, out/test_predictions_e5_brand_cluster.csv, out/cluster_prior_meta.json")


[11:02:24] Fitting KMeans (K=64) on E5_TR=(75000, 1024)... | +0.0s
[11:02:51] KMeans done. | +26.3s
[11:02:51] Computing min-centroid distances... | +26.3s
[11:02:51] Building OOF cluster priors... | +27.1s
[11:02:51]   fold 0: filled 15000 | +27.1s
[11:02:51]   fold 1: filled 15000 | +27.1s
[11:02:51]   fold 2: filled 15000 | +27.1s
[11:02:51]   fold 3: filled 15000 | +27.1s
[11:02:51]   fold 4: filled 15000 | +27.2s
[11:02:51] Computing full-train cluster priors for test... | +27.2s
[11:02:52] New FE shapes -> train=(75000, 222), test=(75000, 222) | +27.9s
[11:02:53] Matrices ready: X_tr=(75000, 2144), X_te=(75000, 2144) | +28.5s
[11:02:53] Training E5-fused head with cluster priors... | +28.5s
  fold 0 SMAPE=49.005% | 156.9s
  fold 1 SMAPE=48.747% | 171.4s
  fold 2 SMAPE=49.060% | 274.8s
  fold 3 SMAPE=48.473% | 287.5s
  fold 4 SMAPE=48.467% | 145.7s
[11:20:09] [E5 + Brand + Cluster prior] CV SMAPE: 48.750% | +1064.8s
[11:20:09] Saved -> out/oof_e5_brand_cluster.csv, out/test_predic

Saving the progress:

In [70]:
# === CONSOLIDATE & SNAPSHOT ARTIFACTS ===
import os, json, time, glob, shutil, hashlib, numpy as np, pandas as pd

t0=time.time()
log=lambda m: print(f"[save] {m} (+{time.time()-t0:.1f}s)")
os.makedirs("out", exist_ok=True)
os.makedirs("runs", exist_ok=True)

def md5sum(path, block=1<<20):
    h=hashlib.md5()
    with open(path,'rb') as f:
        while True:
            b=f.read(block)
            if not b: break
            h.update(b)
    return h.hexdigest()

# 0) Try to persist any in-memory OOF/test arrays so blends are reproducible
def save_if(name, arr, path):
    if name in globals():
        np.save(path, globals()[name])
        log(f"saved {name} -> {path}")

save_if('oof_stack_l', 'oof_stack_l.npy', 'out/oof_stack_l.npy')
save_if('te_stack_l',  'te_stack_l.npy',  'out/te_stack_l.npy')
save_if('oof_img2_l',  'oof_img2_l.npy',  'out/oof_img2_l.npy')
save_if('te_img2_l',   'te_img2_l.npy',   'out/te_img2_l.npy')
save_if('oof_e5_l',    'oof_e5_l.npy',    'out/oof_e5_l.npy')
save_if('te_e5_l',     'te_e5_l.npy',     'out/te_e5_l.npy')

# Also persist brand column (cheap and useful)
if 'df_all' in globals() and 'brand' in df_all.columns:
    df_all[['sample_id','brand']].to_csv('out/brand_map.csv', index=False)
    log("saved out/brand_map.csv")

# 1) Record key files you want to carry over (only those that exist)
must_keep = [
    # embeddings / memmaps
    "/kaggle/working/img_emb_all.f16",
    "/kaggle/working/img_has.npy",
    "/kaggle/working/text_e5_emb_all.f16",
    # predictions / OOFs (many already exist in out/)
    "out/oof_xgb_best_promoted.csv",
    "out/test_predictions_xgb_best_promoted.csv",
    "out/oof_e5_fused.csv",
    "out/test_predictions_e5_fused.csv",
    "out/oof_e5_brand.csv",
    "out/test_predictions_e5_brand.csv",
    "out/oof_e5_brand_cluster.csv",
    "out/test_predictions_e5_brand_cluster.csv",
    "out/test_predictions_segblend_e5brand_vs_promoted.csv",
    "out/segblend_e5brand_vs_promoted_meta.json",
    # your earlier snapshots
    "runs/manifest.json",
    "runs/fe_columns.csv",
    "runs/feature_meta.json",
    "runs/oof_summary.json",
    "runs/xgb_params.json",
]

# Include MiniLM shards if you still rely on them
if os.path.exists("/kaggle/working/text_emb_shards"):
    # don’t copy, just ensure it stays in working; we’ll index it in manifest
    pass

# 2) Minimal environment/specs for reproducibility
!python -V > out/env_python.txt
!pip freeze > out/requirements.txt 2>/dev/null

# 3) Build a manifest with sizes and md5
manifest = {
    "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    "note": "Artifacts for resume; add this notebook's Output as a dataset next session.",
    "files": [],
}
for p in must_keep:
    if os.path.exists(p):
        s = os.path.getsize(p)
        try: h = md5sum(p)
        except Exception: h = None
        manifest["files"].append({"path": p, "size": s, "md5": h})

# add optional dirs (we don’t copy; just record)
for d in ["/kaggle/working/text_emb_shards", "/kaggle/working/runs", "/kaggle/working/out"]:
    if os.path.exists(d):
        total = 0
        for root,_,files in os.walk(d):
            for f in files:
                fp = os.path.join(root,f)
                try: total += os.path.getsize(fp)
                except: pass
        manifest.setdefault("dirs",[]).append({"path": d, "approx_size": total})

# record dataset-level info to recover memmap dims on resume
meta = {}
try:
    N = len(df_all)
    meta["N_rows_all"] = N
except Exception:
    meta["N_rows_all"] = None

def infer_dim(path):
    try:
        sz = os.path.getsize(path)
        N = meta["N_rows_all"]
        if N and sz % (2*N) == 0:  # float16
            return (sz // 2) // N
    except: pass
    return None

for mm in ["/kaggle/working/img_emb_all.f16", "/kaggle/working/text_e5_emb_all.f16"]:
    if os.path.exists(mm):
        manifest.setdefault("memmaps",[]).append({
            "path": mm,
            "dtype": "float16",
            "rows": meta["N_rows_all"],
            "dim": infer_dim(mm)
        })

manifest["meta"] = meta
with open("out/persist_manifest.json","w") as f:
    json.dump(manifest, f, indent=2)
log("wrote out/persist_manifest.json")

print("\nAll set. Now click **Save Version** (Include output). Next session, add these Notebook Output files as input.")


[save] saved oof_stack_l -> out/oof_stack_l.npy (+0.0s)
[save] saved te_stack_l -> out/te_stack_l.npy (+0.0s)
[save] saved oof_img2_l -> out/oof_img2_l.npy (+0.0s)
[save] saved te_img2_l -> out/te_img2_l.npy (+0.0s)
[save] saved oof_e5_l -> out/oof_e5_l.npy (+0.0s)
[save] saved te_e5_l -> out/te_e5_l.npy (+0.0s)
[save] saved out/brand_map.csv (+0.2s)
[save] wrote out/persist_manifest.json (+5.4s)

All set. Now click **Save Version** (Include output). Next session, add these Notebook Output files as input.


In [73]:
# ===========================================
# Segment-aware blend: (E5+Brand+Cluster) vs (Promoted Best)
# ===========================================
import time, json, numpy as np, pandas as pd

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# --- prerequisites
assert all(v in globals() for v in ['train','test','df_all']), "Need train/test/df_all."
y = train['price'].astype(float).values

# segment masks: A = (has_img & missing_qty), B = others
assert 'mask_missing_qty' in globals(), "Need mask_missing_qty from FE step."
HAS = np.load("/kaggle/working/img_has.npy").astype(bool)
is_tr = (df_all['is_train'].values==1); is_te = ~is_tr
SEG_A_TR = HAS[is_tr] & mask_missing_qty[is_tr]
SEG_B_TR = ~SEG_A_TR
SEG_A_TE = HAS[is_te] & mask_missing_qty[is_te]
SEG_B_TE = ~SEG_A_TE
log(f"Segments ready | train A={SEG_A_TR.sum()}, B={SEG_B_TR.sum()} | test A={SEG_A_TE.sum()}, B={SEG_B_TE.sum()}")

# --- Stream A: E5 + Brand + Cluster (use in-memory if present; else load)
if all(v in globals() for v in ['oof_e5_l','te_e5_l']):
    log("Using in-memory E5+Brand+Cluster OOF/TEST.")
    pA_tr = np.exp(oof_e5_l).clip(0.01)
    pA_te = np.exp(te_e5_l).clip(0.01)
else:
    log("Loading E5+Brand+Cluster OOF/TEST from disk ...")
    oofA = pd.read_csv("out/oof_e5_brand_cluster.csv")['oof_log'].values.astype('float32')
    teA  = np.log(np.clip(pd.read_csv("out/test_predictions_e5_brand_cluster.csv")['price'].values, 0.01, None)).astype('float32')
    pA_tr = np.exp(oofA).clip(0.01); pA_te = np.exp(teA).clip(0.01)

# --- Stream B: promoted best XGB
oofB = pd.read_csv("out/oof_xgb_best_promoted.csv")['oof_log'].values.astype('float32')
teB  = np.log(np.clip(pd.read_csv("out/test_predictions_xgb_best_promoted.csv")['price'].values, 0.01, None)).astype('float32')
pB_tr = np.exp(oofB).clip(0.01); pB_te = np.exp(teB).clip(0.01)

# --- grid search weights for each segment
grid = np.linspace(0.0, 1.0, 51)
bestA = {'w': 0.0, 'cv': 1e9}
bestB = {'w': 1.0, 'cv': 1e9}
log("Searching weights...")

for i, w in enumerate(grid, 1):
    if SEG_A_TR.any():
        cvA = smape(y[SEG_A_TR], w*pA_tr[SEG_A_TR] + (1-w)*pB_tr[SEG_A_TR])
        if cvA < bestA['cv']:
            bestA = {'w': float(w), 'cv': float(cvA)}
    if SEG_B_TR.any():
        cvB = smape(y[SEG_B_TR], w*pA_tr[SEG_B_TR] + (1-w)*pB_tr[SEG_B_TR])
        if cvB < bestB['cv']:
            bestB = {'w': float(w), 'cv': float(cvB)}
    if i % 10 == 0:
        log(f"  scanned {i}/{len(grid)} | best A w={bestA['w']:.2f} cv={bestA['cv']:.3f}% | best B w={bestB['w']:.2f} cv={bestB['cv']:.3f}%")

log(f"Segment A best w={bestA['w']:.2f} | seg-CV={bestA['cv']:.3f}%")
log(f"Segment B best w={bestB['w']:.2f} | seg-CV={bestB['cv']:.3f}%")

# --- compose blended OOF/TEST
p_blend_tr = np.empty_like(pA_tr); p_blend_te = np.empty_like(pA_te)
p_blend_tr[SEG_A_TR] = bestA['w']*pA_tr[SEG_A_TR] + (1-bestA['w'])*pB_tr[SEG_A_TR]
p_blend_tr[SEG_B_TR] = bestB['w']*pA_tr[SEG_B_TR] + (1-bestB['w'])*pB_tr[SEG_B_TR]
p_blend_te[SEG_A_TE] = bestA['w']*pA_te[SEG_A_TE] + (1-bestA['w'])*pB_te[SEG_A_TE]
p_blend_te[SEG_B_TE] = bestB['w']*pA_te[SEG_B_TE] + (1-bestB['w'])*pB_te[SEG_B_TE]

cv_blend = smape(y, p_blend_tr)
log(f"Overall blended CV SMAPE: {cv_blend:.3f}%")

# --- optional epsilon floor scan
eps_grid = [0.0005, 0.001, 0.002, 0.005, 0.010, 0.020, 0.050, 0.100]
best_eps = {'eps': None, 'cv': 1e9}
for i, eps in enumerate(eps_grid, 1):
    cv = smape(y, np.maximum(p_blend_tr, eps))
    if cv < best_eps['cv']:
        best_eps = {'eps': float(eps), 'cv': float(cv)}
    if i % 2 == 0:
        log(f"  ε-scan {i}/{len(eps_grid)} | best eps={best_eps['eps']} cv={best_eps['cv']:.3f}%")

log(f"Epsilon best: eps={best_eps['eps']} | CV={best_eps['cv']:.3f}%")

# --- save
import os
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': test['sample_id'], 'price': np.maximum(p_blend_te, best_eps['eps']).astype(float)}).to_csv(
    f"out/test_predictions_48.729{str(best_eps['eps']).replace('.','p')}.csv", index=False)

with open("out/segblend_e5brandcluster_vs_promoted_meta.json","w") as f:
    json.dump({'wA': bestA['w'], 'wB': bestB['w'], 'cvA': bestA['cv'], 'cvB': bestB['cv'], 'cv_overall': float(cv_blend),
               'best_eps': best_eps}, f)

log("Saved -> out/test_predictions_segblend_e5brandcluster_vs_promoted.csv")
log("Saved -> out/test_predictions_segblend_e5brandcluster_eps*.csv")
log("Saved -> out/segblend_e5brandcluster_vs_promoted_meta.json")


[11:37:18] Segments ready | train A=27443, B=47557 | test A=27341, B=47659 | +0.0s
[11:37:18] Using in-memory E5+Brand+Cluster OOF/TEST. | +0.0s
[11:37:18] Searching weights... | +0.1s
[11:37:18]   scanned 10/51 | best A w=0.18 cv=51.712% | best B w=0.18 cv=48.032% | +0.1s
[11:37:18]   scanned 20/51 | best A w=0.38 cv=51.378% | best B w=0.38 cv=47.800% | +0.1s
[11:37:18]   scanned 30/51 | best A w=0.58 cv=51.163% | best B w=0.58 cv=47.615% | +0.2s
[11:37:18]   scanned 40/51 | best A w=0.78 cv=51.073% | best B w=0.78 cv=47.477% | +0.2s
[11:37:18]   scanned 50/51 | best A w=0.82 cv=51.071% | best B w=0.98 cv=47.384% | +0.3s
[11:37:18] Segment A best w=0.82 | seg-CV=51.071% | +0.3s
[11:37:18] Segment B best w=1.00 | seg-CV=47.378% | +0.3s
[11:37:18] Overall blended CV SMAPE: 48.729% | +0.3s
[11:37:18]   ε-scan 2/8 | best eps=0.0005 cv=48.729% | +0.3s
[11:37:18]   ε-scan 4/8 | best eps=0.0005 cv=48.729% | +0.3s
[11:37:18]   ε-scan 6/8 | best eps=0.0005 cv=48.729% | +0.3s
[11:37:18]   ε-sca

In [76]:
# ===========================================
# CatBoost fusion head (log-price) with multi-GPU + blend with E5+Brand+Cluster
# ===========================================
import os, time, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold

# ---- utilities
def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def to2(a):
    a = np.asarray(a)
    return (a if a.ndim==2 else a.reshape(-1,1)).astype('float32', copy=False)

# ---- check we have matrices & targets in memory
need = ['train','test','FE_TR','FE_TE','EMB_TR','EMB_TE','E5_TR','E5_TE','oof_stack_l','te_stack_l','IMG_TR','IMG_TE','HAS_TR','HAS_TE']
for n in need:
    assert n in globals(), f"Missing {n}"

# fused matrices (same recipe you used for the E5 head)
X_tr = np.hstack([to2(FE_TR), to2(EMB_TR), to2(E5_TR), to2(oof_stack_l), to2(IMG_TR), to2(HAS_TR)])
X_te = np.hstack([to2(FE_TE), to2(EMB_TE), to2(E5_TE), to2(te_stack_l),  to2(IMG_TE), to2(HAS_TE)])
y     = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)

print(f"[{time.strftime('%H:%M:%S')}] CatBoost matrices: X_tr={X_tr.shape}, X_te={X_te.shape} (~{(X_tr.nbytes+X_te.nbytes)/1e9:.2f} GB)")

# ---- CatBoost: proper multi-GPU detection & params
from catboost import CatBoostRegressor, Pool
try:
    from catboost.utils import get_gpu_device_count
    gpu_count = int(get_gpu_device_count() or 0)
except Exception:
    # fallback: trust CUDA visibility
    import torch
    gpu_count = 2 if (torch.cuda.is_available() and os.environ.get('CUDA_VISIBLE_DEVICES','') not in ('','-1') and len(os.environ.get('CUDA_VISIBLE_DEVICES','0').split(','))>1) else (1 if torch.cuda.is_available() else 0)

task_type = "GPU" if gpu_count > 0 else "CPU"
devices   = ",".join(map(str, range(gpu_count))) if gpu_count > 1 else ("0" if gpu_count == 1 else None)
print(f"[{time.strftime('%H:%M:%S')}] CatBoost task_type={task_type} | GPUs detected={gpu_count} | devices={devices or '-'}")

params = dict(
    loss_function="MAE",         # training on log(price); MAE/Huber correlate better with SMAPE
    eval_metric="MAE",
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=3.0,
    random_seed=42,
    iterations=10000,
    od_type="Iter",
    od_wait=200,                 # early stopping patience
    verbose=False,
    allow_writing_files=False,   # keep it clean in Kaggle
)
if task_type == "GPU":
    params["task_type"] = "GPU"
    if devices:
        params["devices"] = devices
else:
    params["task_type"] = "CPU"
    params["thread_count"] = -1

# ---- CV train + predict with live logs
oof_cat_l = np.zeros(len(y_log), dtype='float32')
te_cat_l  = np.zeros(X_te.shape[0], dtype='float32')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
t0 = time.time()
print(f"[{time.strftime('%H:%M:%S')}] Training CatBoost folds...")
for f,(tr, va) in enumerate(skf.split(X_tr, bins)):
    t_fold = time.time()
    model = CatBoostRegressor(**params)
    model.fit(Pool(X_tr[tr], y_log[tr]), eval_set=Pool(X_tr[va], y_log[va]))
    oof_cat_l[va] = model.predict(X_tr[va]).astype('float32')
    te_cat_l     += model.predict(X_te).astype('float32') / skf.n_splits
    print(f"  [fold {f}] SMAPE={smape(y[va], np.exp(oof_cat_l[va]).clip(0.01)):.3f}% | time={time.time()-t_fold:.1f}s")

cv_cat = smape(y, np.exp(oof_cat_l).clip(0.01))
print(f"[{time.strftime('%H:%M:%S')}] CatBoost CV SMAPE: {cv_cat:.3f}% | total={time.time()-t0:.1f}s")

# ---- Blend CatBoost with current best (E5+Brand+Cluster)
# Use in-memory E5+Brand+Cluster if available; else load from disk.
if not all(v in globals() for v in ['oof_e5_l','te_e5_l']):
    oof_e5_l = pd.read_csv("out/oof_e5_brand_cluster.csv")['oof_log'].values.astype('float32')
    te_e5_l  = np.log(np.clip(pd.read_csv("out/test_predictions_e5_brand_cluster.csv")['price'].values, 0.01, None)).astype('float32')

pA_tr = np.exp(oof_e5_l).clip(0.01)      # E5+brand+cluster (price)
pA_te = np.exp(te_e5_l).clip(0.01)
pC_tr = np.exp(oof_cat_l).clip(0.01)     # CatBoost (price)
pC_te = np.exp(te_cat_l).clip(0.01)

best = {'w':0.5, 'cv':1e9}
for i,w in enumerate(np.linspace(0.0, 1.0, 51), 1):
    cv = smape(y, w*pA_tr + (1-w)*pC_tr)
    if cv < best['cv']:
        best = {'w':float(w), 'cv':float(cv)}
    if i % 10 == 0:
        print(f"  [blend] scanned {i}/51 | best w={best['w']:.2f} CV={best['cv']:.3f}%")

p_blend_tr = best['w']*pA_tr + (1-best['w'])*pC_tr
p_blend_te = best['w']*pA_te + (1-best['w'])*pC_te
cv_blend   = smape(y, p_blend_tr)
print(f"[blend] Best w={best['w']:.2f} | CV={cv_blend:.3f}%")

# ---- Save artifacts
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_log': oof_cat_l,
              'oof_price': pC_tr}).to_csv("out/oof_catboost_lprice.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'], 'price': pC_te}).to_csv("out/test_predictions_catboost_lprice.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'], 'price': p_blend_te}).to_csv("out/test_predictions_blend_e5brandcluster_catboost.csv", index=False)
print("Saved -> out/oof_catboost_lprice.csv")
print("Saved -> out/test_predictions_catboost_lprice.csv")
print("Saved -> out/test_predictions_blend_e5brandcluster_catboost.csv")


[11:45:29] CatBoost matrices: X_tr=(75000, 2144), X_te=(75000, 2144) (~1.29 GB)
[11:45:29] CatBoost task_type=GPU | GPUs detected=2 | devices=0,1
[11:45:29] Training CatBoost folds...


Default metric period is 5 because MAE is/are not implemented for GPU


  [fold 0] SMAPE=49.100% | time=241.1s


Default metric period is 5 because MAE is/are not implemented for GPU


  [fold 1] SMAPE=49.075% | time=215.2s


...

  [fold 2] SMAPE=49.472% | time=174.2s


Default metric period is 5 because MAE is/are not implemented for GPU


  [fold 3] SMAPE=48.997% | time=172.6s


Default metric period is 5 because MAE is/are not implemented for GPU


  [fold 4] SMAPE=48.626% | time=209.0s
[12:02:21] CatBoost CV SMAPE: 49.054% | total=1012.1s
  [blend] scanned 10/51 | best w=0.18 CV=48.907%
  [blend] scanned 20/51 | best w=0.38 CV=48.789%
  [blend] scanned 30/51 | best w=0.58 CV=48.724%
  [blend] scanned 40/51 | best w=0.74 CV=48.709%
  [blend] scanned 50/51 | best w=0.74 CV=48.709%
[blend] Best w=0.74 | CV=48.709%
Saved -> out/oof_catboost_lprice.csv
Saved -> out/test_predictions_catboost_lprice.csv
Saved -> out/test_predictions_blend_e5brandcluster_catboost.csv


In [77]:
# ===========================================
# Segment-aware blend: (E5+Brand+Cluster) vs (CatBoost)
# ===========================================
import time, json, numpy as np, pandas as pd

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# --- prerequisites
assert all(v in globals() for v in ['train','test','df_all','mask_missing_qty']), "Need train/test/df_all/mask_missing_qty."
y = train['price'].astype(float).values
HAS = np.load("/kaggle/working/img_has.npy").astype(bool)
is_tr = (df_all['is_train'].values==1); is_te = ~is_tr

SEG_A_TR = HAS[is_tr] & mask_missing_qty[is_tr]  # has_img & missing_qty
SEG_B_TR = ~SEG_A_TR
SEG_A_TE = HAS[is_te] & mask_missing_qty[is_te]
SEG_B_TE = ~SEG_A_TE
log(f"Segments ready | train A={SEG_A_TR.sum()}, B={SEG_B_TR.sum()} | test A={SEG_A_TE.sum()}, B={SEG_B_TE.sum()}")

# --- E5+Brand+Cluster
if not all(v in globals() for v in ['oof_e5_l','te_e5_l']):
    log("Loading E5+Brand+Cluster OOF/TEST from disk ...")
    oof_e5_l = pd.read_csv("out/oof_e5_brand_cluster.csv")['oof_log'].values.astype('float32')
    te_e5_l  = np.log(np.clip(pd.read_csv("out/test_predictions_e5_brand_cluster.csv")['price'].values, 0.01, None)).astype('float32')
pA_tr = np.exp(oof_e5_l).clip(0.01)
pA_te = np.exp(te_e5_l).clip(0.01)

# --- CatBoost
oof_cat_l = pd.read_csv("out/oof_catboost_lprice.csv")['oof_log'].values.astype('float32')
te_cat    = np.log(np.clip(pd.read_csv("out/test_predictions_catboost_lprice.csv")['price'].values, 0.01, None)).astype('float32')
pC_tr = np.exp(oof_cat_l).clip(0.01)
pC_te = np.exp(te_cat).clip(0.01)

# --- search per-segment weights w in [0..1] :  w * E5 + (1-w) * CatBoost
grid = np.linspace(0.0, 1.0, 51)
bestA = {'w': 1.0, 'cv': 1e9}
bestB = {'w': 1.0, 'cv': 1e9}
log("Searching weights...")

for i, w in enumerate(grid, 1):
    if SEG_A_TR.any():
        cvA = smape(y[SEG_A_TR], w*pA_tr[SEG_A_TR] + (1-w)*pC_tr[SEG_A_TR])
        if cvA < bestA['cv']: bestA = {'w': float(w), 'cv': float(cvA)}
    if SEG_B_TR.any():
        cvB = smape(y[SEG_B_TR], w*pA_tr[SEG_B_TR] + (1-w)*pC_tr[SEG_B_TR])
        if cvB < bestB['cv']: bestB = {'w': float(w), 'cv': float(cvB)}
    if i % 10 == 0:
        log(f"  scanned {i}/{len(grid)} | bestA w={bestA['w']:.2f} cv={bestA['cv']:.3f}% | bestB w={bestB['w']:.2f} cv={bestB['cv']:.3f}%")

log(f"Segment A best w={bestA['w']:.2f} | seg-CV={bestA['cv']:.3f}%")
log(f"Segment B best w={bestB['w']:.2f} | seg-CV={bestB['cv']:.3f}%")

# --- compose blended OOF/TEST
p_bl_tr = np.empty_like(pA_tr); p_bl_te = np.empty_like(pA_te)
p_bl_tr[SEG_A_TR] = bestA['w']*pA_tr[SEG_A_TR] + (1-bestA['w'])*pC_tr[SEG_A_TR]
p_bl_tr[SEG_B_TR] = bestB['w']*pA_tr[SEG_B_TR] + (1-bestB['w'])*pC_tr[SEG_B_TR]
p_bl_te[SEG_A_TE] = bestA['w']*pA_te[SEG_A_TE] + (1-bestA['w'])*pC_te[SEG_A_TE]
p_bl_te[SEG_B_TE] = bestB['w']*pA_te[SEG_B_TE] + (1-bestB['w'])*pC_te[SEG_B_TE]

cv_bl = smape(y, p_bl_tr)
log(f"Segment-aware CatBoost blend CV SMAPE: {cv_bl:.3f}%")

# --- optional epsilon floor
eps_grid = [0.0005, 0.001, 0.002, 0.005, 0.010]
best_eps = {'eps': 0.0005, 'cv': smape(y, np.maximum(p_bl_tr, 0.0005))}
for eps in eps_grid[1:]:
    cv = smape(y, np.maximum(p_bl_tr, eps))
    if cv < best_eps['cv']: best_eps = {'eps': float(eps), 'cv': float(cv)}
log(f"Epsilon best: eps={best_eps['eps']} | CV={best_eps['cv']:.3f}%")


[12:06:23] Segments ready | train A=27443, B=47557 | test A=27341, B=47659 | +0.0s
[12:06:23] Searching weights... | +0.1s
[12:06:23]   scanned 10/51 | bestA w=0.18 cv=50.990% | bestB w=0.18 cv=47.705% | +0.1s
[12:06:23]   scanned 20/51 | bestA w=0.38 cv=50.895% | bestB w=0.38 cv=47.574% | +0.1s
[12:06:23]   scanned 30/51 | bestA w=0.50 cv=50.880% | bestB w=0.58 cv=47.476% | +0.2s
[12:06:23]   scanned 40/51 | bestA w=0.50 cv=50.880% | bestB w=0.78 cv=47.412% | +0.2s
[12:06:23]   scanned 50/51 | bestA w=0.50 cv=50.880% | bestB w=0.98 cv=47.379% | +0.3s
[12:06:23] Segment A best w=0.50 | seg-CV=50.880% | +0.3s
[12:06:23] Segment B best w=1.00 | seg-CV=47.378% | +0.3s
[12:06:23] Segment-aware CatBoost blend CV SMAPE: 48.659% | +0.3s
[12:06:23] Epsilon best: eps=0.0005 | CV=48.659% | +0.3s


In [79]:

# --- save
os.makedirs("out", exist_ok=True)

pd.DataFrame({'sample_id': test['sample_id'], 'price': np.maximum(p_bl_te, best_eps['eps']).astype(float)}).to_csv(
    f"out/test_predictions_48.659{str(best_eps['eps']).replace('.','p')}.csv", index=False)
with open("out/segblend_e5brandcluster_cat_meta.json","w") as f:
    json.dump({'wA': bestA['w'], 'wB': bestB['w'], 'cvA': bestA['cv'], 'cvB': bestB['cv'],
               'cv_overall': float(cv_bl), 'eps': best_eps}, f)
log("Saved -> out/test_predictions_segblend_e5brandcluster_cat*.csv and meta.json")


[12:09:20] Saved -> out/test_predictions_segblend_e5brandcluster_cat*.csv and meta.json | +177.3s


In [80]:
# ===========================================
# E5-fused head with monotone constraints on quantity features
# ===========================================
import os, time, json, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

t0 = time.time()
log = lambda m: print(f"[{time.strftime('%H:%M:%S')}] {m} | +{time.time()-t0:.1f}s")

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

def to2(a):
    a = np.asarray(a)
    return (a if a.ndim==2 else a.reshape(-1,1)).astype('float32', copy=False)

# ---- prerequisites in memory
need = ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','E5_TR','E5_TE',
        'oof_stack_l','te_stack_l','IMG_TR','IMG_TE','HAS_TR','HAS_TE']
for n in need: 
    assert n in globals(), f"Missing {n}"

# ---- get FE column names (from memory or runs/fe_columns.csv)
fe_cols = None
if 'FE_COLS' in globals():
    fe_cols = list(FE_COLS)
elif os.path.exists("runs/fe_columns.csv"):
    fe_cols = pd.read_csv("runs/fe_columns.csv")['col'].tolist()
else:
    # last resort: make placeholder names
    fe_cols = [f'fe_{i}' for i in range(to2(FE_TR).shape[1])]
log(f"FE columns: {len(fe_cols)}")

# ---- build monotone constraints vector for XGBoost
# positive-up features we expect price to increase with if all else equal
POS_FEATS = {'total_mass_g','total_vol_ml','eff_units','pack_count','case_count'}
name_to_idx = {c:i for i,c in enumerate(fe_cols)}
mono_fe = [0]*len(fe_cols)
for k in POS_FEATS:
    if k in name_to_idx:
        mono_fe[name_to_idx[k]] = 1

# full feature order: [FE, EMB, E5, oof_stack, IMG, HAS]
mono = []
mono += mono_fe
mono += [0]*to2(EMB_TR).shape[1]
mono += [0]*to2(E5_TR).shape[1]
mono += [0]  # oof_stack_l
mono += [0]*to2(IMG_TR).shape[1]
mono += [0]  # HAS_TR

# XGBoost expects "(a,b,c,...)"
mono_str = "(" + ",".join(map(str, mono)) + ")"
log(f"Monotone vector length={len(mono)}; pos_count={sum(mono)}")

# ---- build matrices
X_tr = np.hstack([to2(FE_TR), to2(EMB_TR), to2(E5_TR), to2(oof_stack_l), to2(IMG_TR), to2(HAS_TR)])
X_te = np.hstack([to2(FE_TE), to2(EMB_TE), to2(E5_TE), to2(te_stack_l),  to2(IMG_TE), to2(HAS_TE)])
log(f"Matrices ready: X_tr={X_tr.shape}, X_te={X_te.shape}")

y     = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)

params = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae',
    objective='reg:absoluteerror', max_depth=8, min_child_weight=2, learning_rate=0.03,
    monotone_constraints=mono_str,
)

oof_mono_l = np.zeros_like(y_log, dtype='float32')
te_mono_l  = np.zeros(X_te.shape[0], dtype='float32')

log("Training monotone-constrained E5-fused head...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for f,(tr, va) in enumerate(skf.split(X_tr, bins)):
    t_fold = time.time()
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr[tr], y_log[tr], eval_set=[(X_tr[va], y_log[va])], verbose=False)
    oof_mono_l[va] = m.predict(X_tr[va])
    te_mono_l += m.predict(X_te)/skf.n_splits
    print(f"  [fold {f}] SMAPE={smape(y[va], np.exp(oof_mono_l[va]).clip(0.01)):.3f}% | {time.time()-t_fold:.1f}s")

cv_mono = smape(y, np.exp(oof_mono_l).clip(0.01))
log(f"[E5-fused + Monotone constraints] CV SMAPE: {cv_mono:.3f}%")

# expose + save
oof_e5_l = oof_mono_l.copy()
te_e5_l  = te_mono_l.copy()

os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_log': oof_mono_l,
              'oof_price': np.exp(oof_mono_l).clip(0.01)}).to_csv("out/oof_e5_mono.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],
              'price': np.exp(te_mono_l).clip(0.01).astype(float)}).to_csv("out/test_predictions_e5_mono.csv", index=False)
with open("out/e5_mono_meta.json","w") as f:
    json.dump({'pos_feats_used': [c for c in POS_FEATS if c in name_to_idx],
               'cv_smape': float(cv_mono)}, f)

log("Saved -> out/oof_e5_mono.csv, out/test_predictions_e5_mono.csv, out/e5_mono_meta.json")


[12:12:25] FE columns: 222 | +0.0s
[12:12:25] Monotone vector length=2144; pos_count=0 | +0.0s
[12:12:26] Matrices ready: X_tr=(75000, 2144), X_te=(75000, 2144) | +0.5s
[12:12:26] Training monotone-constrained E5-fused head... | +0.5s
  [fold 0] SMAPE=48.937% | 185.3s
  [fold 1] SMAPE=48.668% | 155.6s
  [fold 2] SMAPE=49.145% | 128.5s
  [fold 3] SMAPE=48.575% | 189.7s
  [fold 4] SMAPE=48.543% | 172.1s
[12:26:17] [E5-fused + Monotone constraints] CV SMAPE: 48.773% | +831.7s
[12:26:17] Saved -> out/oof_e5_mono.csv, out/test_predictions_e5_mono.csv, out/e5_mono_meta.json | +832.0s


In [9]:
import numpy as np, pandas as pd

# --- 1a. scan FE column names for suspicious tokens
sus_tokens = ("price","mrp","cost","rs","₹","dollar","usd")
fe_cols = (list(FE_COLS) if 'FE_COLS' in globals()
           else (pd.read_csv("runs/fe_columns.csv")['col'].tolist()
                 if os.path.exists("runs/fe_columns.csv")
                 else [f'fe_{i}' for i in range(np.asarray(FE_TR).shape[1])]))
bad = [c for c in fe_cols if any(t in c.lower() for t in sus_tokens)]
print("Suspicious FE columns:", bad or "none")

# --- 1b. fold stability from your latest OOF (use your current best stream)
def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    m = d!=0
    out = np.zeros_like(d, dtype=float); out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

y = train['price'].astype(float).values
# choose the stream you consider "best" OOF in LOG space:
oof_log_best = (oof_e5_l if 'oof_e5_l' in globals() else 
                pd.read_csv("out/oof_e5_brand_cluster.csv")['oof_log'].values)
oof_price = np.exp(oof_log_best).clip(0.01)

# if you saved folds.csv earlier, use it; else recompute approximate bins
if os.path.exists("runs/folds.csv"):
    folds = pd.read_csv("runs/folds.csv")['fold'].values
else:
    folds = pd.qcut(y, 5, labels=False, duplicates='drop')  # proxy

fold_scores = []
for f in np.unique(folds):
    idx = (folds==f)
    fold_scores.append(smape(y[idx], oof_price[idx]))
print("Fold SMAPEs:", [round(s,3) for s in fold_scores], "| mean=", round(np.mean(fold_scores),3), "std=", round(np.std(fold_scores),3))

# --- 1c. brand frequency buckets
assert 'brand' in df_all.columns, "No 'brand' column. Run the brand extraction you used earlier."
is_tr = (df_all['is_train'].values==1)
tr_brand = df_all.loc[is_tr, 'brand'].fillna('other')
freq = tr_brand.value_counts()
freq_bin = tr_brand.map(lambda b: ('1',) if freq[b]==1 else
                                 ('2-5',) if 2<=freq[b]<=5 else
                                 ('6-20',) if 6<=freq[b]<=20 else
                                 ('21+',))
tmp = pd.DataFrame({'y':y, 'pred':oof_price, 'bin':freq_bin})
print(tmp.groupby('bin').apply(lambda d: smape(d['y'].values, d['pred'].values)).round(3))

NameError: name 'FE_TR' is not defined

In [83]:
# Validate the segment-aware blend weights via pseudo-nested CV (no retrain)
import numpy as np, pandas as pd

def nested_weight_cv(y, oofA_log, oofB_log, seg_mask, grid=np.linspace(0,1,51), folds=None):
    """Return SMAPE using per-fold held-out weights."""
    oofA = np.exp(oofA_log).clip(0.01); oofB = np.exp(oofB_log).clip(0.01)
    if folds is None:
        folds = pd.qcut(y, 5, labels=False, duplicates='drop')  # proxy if no saved folds
    pred = np.zeros_like(y, dtype=float)
    for f in np.unique(folds):
        tr = folds!=f; va = folds==f
        best_w, best_cv = 0.5, 1e9
        for w in grid:
            cv = smape(y[tr & seg_mask], w*oofA[tr & seg_mask] + (1-w)*oofB[tr & seg_mask])
            if cv < best_cv: best_cv, best_w = cv, w
        pred[va & seg_mask] = best_w*oofA[va & seg_mask] + (1-best_w)*oofB[va & seg_mask]
    return smape(y[seg_mask], pred[seg_mask]), pred

# pick two streams: A = E5+brand+cluster, B = CatBoost (both LOG space)
oofA_log = (oof_e5_l if 'oof_e5_l' in globals()
            else pd.read_csv("out/oof_e5_brand_cluster.csv")['oof_log'].values.astype('float32'))
oofB_log = pd.read_csv("out/oof_catboost_lprice.csv")['oof_log'].values.astype('float32')

HAS = np.load("/kaggle/working/img_has.npy").astype(bool)
is_tr = (df_all['is_train'].values==1)
segA = HAS[is_tr] & mask_missing_qty[is_tr]   # your “has_img & missing_qty” slice
segB = ~segA

# overall nested
folds = pd.read_csv("runs/folds.csv")['fold'].values if os.path.exists("runs/folds.csv") else None
cvA, _ = nested_weight_cv(y, oofA_log, oofB_log, segA, folds=folds)
cvB, _ = nested_weight_cv(y, oofA_log, oofB_log, segB, folds=folds)

# compose overall
# For the other segment while validating one, just copy the better single-stream:
pred_all = np.zeros_like(y, dtype=float)
_, predA = nested_weight_cv(y, oofA_log, oofB_log, segA, folds=folds)
_, predB = nested_weight_cv(y, oofA_log, oofB_log, segB, folds=folds)
pred_all[segA] = predA[segA]; pred_all[segB] = predB[segB]
print("Nested blend CV (segA):", round(cvA,3), "| (segB):", round(cvB,3), "| overall:", round(smape(y, pred_all),3))

Nested blend CV (segA): 51.584 | (segB): 47.75 | overall: 49.153


In [84]:
# ================================
# OOF audit: alignment + fold SMAPEs for seg-aware E5+Brand+Cluster ⟷ CatBoost
# ================================
import os, json, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

# --- prerequisites
assert all(v in globals() for v in ['train','test','df_all','mask_missing_qty']), "Need train/test/df_all/mask_missing_qty"
HAS = np.load("/kaggle/working/img_has.npy").astype(bool)
is_tr = (df_all['is_train'].values==1)
y = train['price'].astype(float).values
sid_tr = train['sample_id'].values

# --- load OOF parts (log space) and ensure alignment by sample_id
oofA = pd.read_csv("out/oof_e5_brand_cluster.csv")      # columns: sample_id, oof_log, oof_price
oofC = pd.read_csv("out/oof_catboost_lprice.csv")       # columns: sample_id, oof_log, oof_price

# Align by sample_id to avoid any index drift
oof_join = (pd.DataFrame({'sample_id': sid_tr})
            .merge(oofA[['sample_id','oof_log']].rename(columns={'oof_log':'log_A'}), on='sample_id', how='left')
            .merge(oofC[['sample_id','oof_log']].rename(columns={'oof_log':'log_C'}), on='sample_id', how='left'))
assert not oof_join[['log_A','log_C']].isna().any().any(), "Missing OOF rows after join; check earlier artifacts."

pA_tr = np.exp(oof_join['log_A'].values).clip(0.01)
pC_tr = np.exp(oof_join['log_C'].values).clip(0.01)

# --- segment masks on train
SEG_A_TR = HAS[is_tr] & mask_missing_qty[is_tr]
SEG_B_TR = ~SEG_A_TR

# --- read segment blend weights used previously
meta_path = "out/segblend_e5brandcluster_cat_meta.json"
assert os.path.exists(meta_path), "Missing segblend_e5brandcluster_cat_meta.json (run the seg-aware CatBoost blend cell first)."
meta = json.load(open(meta_path))
wA, wB = meta['wA'], meta['wB']
print(f"Using weights: SegmentA w={wA:.2f}, SegmentB w={wB:.2f}")

# --- compose blended OOF and compute overall CV
p_blend_tr = np.empty_like(pA_tr)
p_blend_tr[SEG_A_TR] = wA*pA_tr[SEG_A_TR] + (1-wA)*pC_tr[SEG_A_TR]
p_blend_tr[SEG_B_TR] = wB*pA_tr[SEG_B_TR] + (1-wB)*pC_tr[SEG_B_TR]
cv_all = smape(y, p_blend_tr)
print(f"Recomputed overall blended CV: {cv_all:.3f}%")

# ---- fold-wise SMAPE using the same fold scheme as modeling
# Prefer saved folds if you have them; else reconstruct with same seed/binning
folds_csv = None
for cand in ["runs/folds.csv", "/kaggle/working/runs/folds.csv"]:
    if os.path.exists(cand):
        folds_csv = cand; break

if folds_csv:
    folds_df = pd.read_csv(folds_csv)
    assert 'fold' in folds_df.columns and 'sample_id' in folds_df.columns
    folds_df = pd.DataFrame({'sample_id': sid_tr}).merge(folds_df[['sample_id','fold']], on='sample_id', how='left')
    assert not folds_df['fold'].isna().any(), "Some train rows missing fold id."
    fold_idx = folds_df['fold'].astype(int).values
    n_folds = folds_df['fold'].nunique()
else:
    # reconstruct
    bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)
    skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_idx = np.zeros(len(y), dtype=int)
    for f,(_, va) in enumerate(skf.split(np.zeros_like(y), bins)):
        fold_idx[va] = f
    n_folds = 5

# compute per-fold SMAPE + seg breakdown
s_fold = []
for f in range(n_folds):
    m = (fold_idx == f)
    sm = smape(y[m], p_blend_tr[m])
    s_fold.append(sm)
    if SEG_A_TR[m].any():
        print(f"  fold {f} | A={SEG_A_TR[m].sum()} B={(~SEG_A_TR[m]).sum()} | SMAPE={sm:.3f}% "
              f"| segA={smape(y[m & SEG_A_TR], p_blend_tr[m & SEG_A_TR]):.3f}% segB={smape(y[m & ~SEG_A_TR], p_blend_tr[m & ~SEG_A_TR]):.3f}%")
    else:
        print(f"  fold {f} | A=0 B={m.sum()} | SMAPE={sm:.3f}%")

print("Fold SMAPEs:", [round(v,3) for v in s_fold], "| mean=", round(float(np.mean(s_fold)),3), "std=", round(float(np.std(s_fold)),3))

# ---- error by quantity bucket (if available)
bucket = np.full(len(y), "unknown", dtype=object)
if 'eff_units' in globals() and isinstance(eff_units := (FE_TR[:, list(pd.read_csv("runs/fe_columns.csv")['col']).index('eff_units')] if os.path.exists("runs/fe_columns.csv") and 'FE_TR' in globals() else None), np.ndarray):
    # If you have eff_units as a column, you can plug it here instead—this is placeholder
    pass

# Safer: price deciles view (always available)
dec = pd.qcut(y, 10, duplicates='drop')
tmp = pd.DataFrame({'y': y, 'pred': p_blend_tr, 'decile': dec})
print("\nPer-price-decile SMAPE:")
print(tmp.groupby('decile').apply(lambda d: smape(d['y'].values, d['pred'].values)).round(3))


Using weights: SegmentA w=0.50, SegmentB w=1.00
Recomputed overall blended CV: 48.659%
  fold 0 | A=5497 B=9503 | SMAPE=48.889% | segA=51.040% segB=47.646%
  fold 1 | A=5550 B=9450 | SMAPE=48.678% | segA=50.597% segB=47.551%
  fold 2 | A=5390 B=9610 | SMAPE=48.948% | segA=51.516% segB=47.508%
  fold 3 | A=5484 B=9516 | SMAPE=48.415% | segA=50.786% segB=47.049%
  fold 4 | A=5522 B=9478 | SMAPE=48.365% | segA=50.477% segB=47.134%
Fold SMAPEs: [48.889, 48.678, 48.948, 48.415, 48.365] | mean= 48.659 std= 0.238

Per-price-decile SMAPE:
decile
(0.129, 3.565]      83.854
(3.565, 5.6]        55.027
(5.6, 7.99]         42.293
(7.99, 10.78]       34.002
(10.78, 14.0]       31.283
(14.0, 18.45]       32.054
(18.45, 24.69]      36.308
(24.69, 33.891]     44.366
(33.891, 52.301]    56.354
(52.301, 2796.0]    70.887
dtype: float64


/tmp/ipykernel_227/4012095028.py:98: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(tmp.groupby('decile').apply(lambda d: smape(d['y'].values, d['pred'].values)).round(3))
/tmp/ipykernel_227/4012095028.py:98: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(tmp.groupby('decile').apply(lambda d: smape(d['y'].values, d['pred'].values)).round(3))


In [5]:
# ==== QUICK SAVE (run now) ====
import os, shutil, tarfile, json, numpy as np, pandas as pd, time

os.makedirs("/kaggle/working/checkpoint", exist_ok=True)

# 1) Save small-but-important arrays (so you don't need to recompute FE)
to_save = {}
if 'FE_TR' in globals(): to_save['fe_tr.npy'] = np.asarray(FE_TR, dtype='float32')
if 'FE_TE' in globals(): to_save['fe_te.npy'] = np.asarray(FE_TE, dtype='float32')
if 'oof_stack_l' in globals(): to_save['oof_stack_l.npy'] = np.asarray(oof_stack_l, dtype='float32')
if 'te_stack_l'  in globals(): to_save['te_stack_l.npy']  = np.asarray(te_stack_l,  dtype='float32')

# If you have the latest promoted/e5 oof logs, save them too (optional)
if 'oof_img2_l' in globals(): to_save['oof_img2_l.npy'] = np.asarray(oof_img2_l, dtype='float32')
if 'oof_e5_l'   in globals(): to_save['oof_e5_l.npy']   = np.asarray(oof_e5_l,   dtype='float32')

for name, arr in to_save.items():
    np.save(f"/kaggle/working/{name}", arr)

# 2) Copy directories/files we must persist
KEEP_PATHS = [
    "/kaggle/working/out",                     # submissions, oof csvs, metas
    "/kaggle/working/runs",                    # manifests, folds, params
    "/kaggle/working/text_emb_shards",         # MiniLM shards (384-d)
    "/kaggle/working/text_e5_emb_all.f16",     # E5 memmap (if you created it)
    "/kaggle/working/img_emb_all.f16",         # OpenCLIP B/32 memmap
    "/kaggle/working/img_has.npy",             # B/32 coverage flags
    "/kaggle/working/img_emb_L14_all.f16",     # OpenCLIP L/14 memmap (partial is fine)
    "/kaggle/working/img_has_L14.npy",         # L/14 coverage flags
    "/kaggle/working/fe_columns.csv",          # your FE column list
    "/kaggle/working/feature_meta.json",       # your FE metadata (if any)
    "/kaggle/working/oof_stack_l.npy",
    "/kaggle/working/te_stack_l.npy",
    "/kaggle/working/oof_img2_l.npy",
    "/kaggle/working/oof_e5_l.npy",
    "/kaggle/working/fe_tr.npy",
    "/kaggle/working/fe_te.npy",
]

CP_DIR = "/kaggle/working/checkpoint"
for p in KEEP_PATHS:
    if not os.path.exists(p):
        continue
    dst = os.path.join(CP_DIR, os.path.basename(p.rstrip("/")))
    if os.path.isdir(p):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(p, dst)
    else:
        shutil.copy2(p, dst)

# 3) Freeze environment (handy if you switch runtimes)
with open("/kaggle/working/checkpoint/ENV_INFO.txt", "w") as f:
    f.write(time.strftime("%Y-%m-%d %H:%M:%S") + "\n")

# 4) Pack into a single tar.gz (shows up under "Output" tab)
tar_path = "/kaggle/working/checkpoint_bundle.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(CP_DIR, arcname="checkpoint")
print("Packed ->", tar_path)


Packed -> /kaggle/working/checkpoint_bundle.tar.gz


In [88]:
# ============== Dual-GPU ViT-L/14 encoder (THREADS, resumable) ==============
import os, io, time, gc, requests, numpy as np, pandas as pd, torch, open_clip
from PIL import Image, ImageFile
from concurrent.futures import ThreadPoolExecutor, as_completed
ImageFile.LOAD_TRUNCATED_IMAGES = True
requests.packages.urllib3.disable_warnings()

assert 'df_all' in globals(), "Need df_all"
N = len(df_all)
img_urls = df_all['image_link'].fillna('')

MM_L14_PATH  = "/kaggle/working/img_emb_L14_all.f16"
HAS_L14_PATH = "/kaggle/working/img_has_L14.npy"

# Ensure memmap exists and we know L14 dim
if not os.path.exists(MM_L14_PATH):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    mdl, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='laion2b_s32b_b82k', device=dev)
    mdl.eval()
    with torch.no_grad():
        L14_DIM = mdl.encode_image(torch.zeros(1,3,224,224, device=dev)).shape[1]
    del mdl; gc.collect()
    mm = np.memmap(MM_L14_PATH, dtype='float16', mode='w+', shape=(N, L14_DIM))
    del mm
else:
    sz = os.path.getsize(MM_L14_PATH)
    L14_DIM = (sz // 2) // N

HAS_L14 = np.load(HAS_L14_PATH) if os.path.exists(HAS_L14_PATH) else np.zeros(N, dtype=np.uint8)

# remaining targets (you can prioritize Segment-A if you want)
remaining = np.where(HAS_L14 == 0)[0]
print(f"[main] L14 dim={L14_DIM} | remaining rows: {len(remaining)}")

# split into two disjoint shards
shard0, shard1 = np.array_split(remaining, 2)

def encode_shard(device_id: int, idxs: np.ndarray, batch: int = 96, timeout=6):
    """Thread target: each thread loads its own model on its own GPU."""
    if len(idxs) == 0:
        print(f"[gpu{device_id}] nothing to do"); return 0
    device = f"cuda:{device_id}" if torch.cuda.is_available() else "cpu"
    model, _, preprocess = open_clip.create_model_and_transforms('ViT-L-14', pretrained='laion2b_s32b_b82k', device=device)
    model.eval()
    mm = np.memmap(MM_L14_PATH, dtype='float16', mode='r+', shape=(N, L14_DIM))
    sess = requests.Session()
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Kaggle/Images/1.0)"}
    B = batch if "cuda" in device else 24
    done = 0; t0 = time.time()
    for i in range(0, len(idxs), B):
        chunk = idxs[i:i+B]
        ims, keep = [], []
        for idx in chunk:
            url = img_urls.iloc[idx]
            if not isinstance(url, str) or not url.strip():
                continue
            try:
                r = sess.get(url, headers=headers, timeout=timeout, verify=False)
                if r.status_code == 200 and r.content:
                    im = Image.open(io.BytesIO(r.content)).convert('RGB')
                    ims.append(preprocess(im)); keep.append(idx)
            except Exception:
                pass
        if not ims:
            continue
        with torch.no_grad():
            t = torch.stack(ims).to(device)
            f = model.encode_image(t)
            f = torch.nn.functional.normalize(f, dim=1).float().cpu().numpy().astype('float16')
        for j, idx in enumerate(keep):
            mm[idx, :L14_DIM] = f[j]
            HAS_L14[idx] = 1
        done += len(keep)
        if (i//B) % 8 == 0:
            mm.flush(); np.save(HAS_L14_PATH, HAS_L14)
            print(f"[gpu{device_id}] encoded {i+len(chunk)}/{len(idxs)} | new={done} | {((time.time()-t0)/60):.1f}m", flush=True)
    mm.flush(); np.save(HAS_L14_PATH, HAS_L14)
    print(f"[gpu{device_id}] finished shard | new={done} | {(time.time()-t0)/60:.1f}m", flush=True)
    return done

# launch 2 threads (one per GPU)
futs = []
with ThreadPoolExecutor(max_workers=2) as ex:
    futs.append(ex.submit(encode_shard, 0, shard0))
    futs.append(ex.submit(encode_shard, 1, shard1))
    total = 0
    for f in as_completed(futs):
        total += f.result()
print(f"[main] total newly encoded: {total} | total has_l14={int(HAS_L14.sum())}/{N}")


[main] L14 dim=768 | remaining rows: 133584
[gpu0] encoded 96/66792 | new=96 | 0.2m
[gpu1] encoded 96/66792 | new=96 | 0.2m
[gpu0] encoded 864/66792 | new=864 | 1.6m
[gpu1] encoded 864/66792 | new=864 | 1.8m
[gpu0] encoded 1632/66792 | new=1632 | 3.2m
[gpu1] encoded 1632/66792 | new=1632 | 3.3m
[gpu0] encoded 2400/66792 | new=2400 | 4.7m
[gpu1] encoded 2400/66792 | new=2400 | 4.9m
[gpu0] encoded 3168/66792 | new=3168 | 6.3m
[gpu1] encoded 3168/66792 | new=3168 | 6.4m
[gpu0] encoded 3936/66792 | new=3936 | 7.9m
[gpu1] encoded 3936/66792 | new=3936 | 8.0m
[gpu1] encoded 4704/66792 | new=4704 | 9.5m
[gpu0] encoded 4704/66792 | new=4704 | 9.5m
[gpu1] encoded 5472/66792 | new=5472 | 11.1m
[gpu0] encoded 5472/66792 | new=5472 | 11.1m
[gpu1] encoded 6240/66792 | new=6240 | 12.6m
[gpu0] encoded 6240/66792 | new=6240 | 12.7m
[gpu1] encoded 7008/66792 | new=7008 | 14.1m
[gpu0] encoded 7008/66792 | new=7008 | 14.2m
[gpu1] encoded 7776/66792 | new=7776 | 15.7m
[gpu0] encoded 7776/66792 | new=7776 

In [2]:
# ==== QUICK SAVE (run now) ====
import os, shutil, tarfile, json, numpy as np, pandas as pd, time

os.makedirs("/kaggle/working/checkpoint", exist_ok=True)

# 1) Save small-but-important arrays (so you don't need to recompute FE)
to_save = {}
if 'FE_TR' in globals(): to_save['fe_tr.npy'] = np.asarray(FE_TR, dtype='float32')
if 'FE_TE' in globals(): to_save['fe_te.npy'] = np.asarray(FE_TE, dtype='float32')
if 'oof_stack_l' in globals(): to_save['oof_stack_l.npy'] = np.asarray(oof_stack_l, dtype='float32')
if 'te_stack_l'  in globals(): to_save['te_stack_l.npy']  = np.asarray(te_stack_l,  dtype='float32')

# If you have the latest promoted/e5 oof logs, save them too (optional)
if 'oof_img2_l' in globals(): to_save['oof_img2_l.npy'] = np.asarray(oof_img2_l, dtype='float32')
if 'oof_e5_l'   in globals(): to_save['oof_e5_l.npy']   = np.asarray(oof_e5_l,   dtype='float32')

for name, arr in to_save.items():
    np.save(f"/kaggle/working/{name}", arr)

# 2) Copy directories/files we must persist
KEEP_PATHS = [
    "/kaggle/working/out",                     # submissions, oof csvs, metas
    "/kaggle/working/runs",                    # manifests, folds, params
    "/kaggle/working/text_emb_shards",         # MiniLM shards (384-d)
    "/kaggle/working/text_e5_emb_all.f16",     # E5 memmap (if you created it)
    "/kaggle/working/img_emb_all.f16",         # OpenCLIP B/32 memmap
    "/kaggle/working/img_has.npy",             # B/32 coverage flags
    "/kaggle/working/img_emb_L14_all.f16",     # OpenCLIP L/14 memmap (partial is fine)
    "/kaggle/working/img_has_L14.npy",         # L/14 coverage flags
    "/kaggle/working/fe_columns.csv",          # your FE column list
    "/kaggle/working/feature_meta.json",       # your FE metadata (if any)
    "/kaggle/working/oof_stack_l.npy",
    "/kaggle/working/te_stack_l.npy",
    "/kaggle/working/oof_img2_l.npy",
    "/kaggle/working/oof_e5_l.npy",
    "/kaggle/working/fe_tr.npy",
    "/kaggle/working/fe_te.npy",
]

CP_DIR = "/kaggle/working/checkpoint"
for p in KEEP_PATHS:
    if not os.path.exists(p):
        continue
    dst = os.path.join(CP_DIR, os.path.basename(p.rstrip("/")))
    if os.path.isdir(p):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(p, dst)
    else:
        shutil.copy2(p, dst)

# 3) Freeze environment (handy if you switch runtimes)
with open("/kaggle/working/checkpoint/ENV_INFO.txt", "w") as f:
    f.write(time.strftime("%Y-%m-%d %H:%M:%S") + "\n")

# 4) Pack into a single tar.gz (shows up under "Output" tab)
tar_path = "/kaggle/working/checkpoint_bundle.tar1.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(CP_DIR, arcname="checkpoint")
print("Packed ->", tar_path)


Packed -> /kaggle/working/checkpoint_bundle.tar1.gz


In [8]:
# ==== Verify L/14 coverage + build X_tr/X_te with new features ====
import os, numpy as np, pandas as pd

# Required (should already be in memory from earlier steps in this session)
need = ['train','test','df_all','FE_TR','FE_TE','EMB_TR','EMB_TE','oof_stack_l','te_stack_l',
        'IMG_TR','IMG_TE','HAS_TR']  # IMG_TR/IMG_TE is B/32; HAS_TR is B/32 flag
for n in need: 
    assert n in globals(), f"Missing {n}. (Restore first if you are in a new session.)"

# Optional E5
if 'E5_TR' not in globals():
    E5_TR = np.zeros((len(train),0), dtype='float32')
    E5_TE = np.zeros((len(test), 0), dtype='float32')

# Load L/14 memmap + flags
MM_L14_PATH  = "/kaggle/working/img_emb_L14_all.f16"
HAS_L14_PATH = "/kaggle/working/img_has_L14.npy"
assert os.path.exists(MM_L14_PATH) and os.path.exists(HAS_L14_PATH), "L/14 memmap/flags not found yet."

N = len(df_all)
dim_L14 = (os.path.getsize(MM_L14_PATH)//2) // N  # float16 -> 2 bytes
mm_L14  = np.memmap(MM_L14_PATH, dtype='float16', mode='r', shape=(N, dim_L14))
has_L14_all = np.load(HAS_L14_PATH).astype(np.uint8)
is_tr = (df_all['is_train'].values==1)

IMG_L14_TR = np.array(mm_L14[is_tr],  dtype='float32')
IMG_L14_TE = np.array(mm_L14[~is_tr], dtype='float32')
HAS_L14_TR = has_L14_all[is_tr].reshape(-1,1).astype('float32')
HAS_L14_TE = has_L14_all[~is_tr].reshape(-1,1).astype('float32')

# zero out missing rows for L/14 (safety)
IMG_L14_TR *= HAS_L14_TR
IMG_L14_TE *= HAS_L14_TE

print(f"L/14 dim={dim_L14} | coverage train={HAS_L14_TR.mean()*100:.1f}% | test={HAS_L14_TE.mean()*100:.1f}%")

# Compose final matrices (B/32 already in IMG_TR/IMG_TE; HAS_TR is for B/32)
def to2(a): 
    a = np.asarray(a); 
    return a if a.ndim==2 else a.reshape(-1,1)

X_tr = np.hstack([
    to2(FE_TR).astype('float32'),
    to2(EMB_TR).astype('float32'),
    to2(E5_TR).astype('float32'),
    to2(oof_stack_l).astype('float32'),
    to2(IMG_TR).astype('float32'),      # B/32
    to2(IMG_L14_TR).astype('float32'),  # L/14
    to2(HAS_TR).astype('float32'),      # B/32 flag
    to2(HAS_L14_TR).astype('float32'),  # L/14 flag
])

X_te = np.hstack([
    to2(FE_TE).astype('float32'),
    to2(EMB_TE).astype('float32'),
    to2(E5_TE).astype('float32'),
    to2(te_stack_l).astype('float32'),
    to2(IMG_TE).astype('float32'),
    to2(IMG_L14_TE).astype('float32'),
    to2(HAS_TE).astype('float32'),
    to2(HAS_L14_TE).astype('float32'),
])

print("Matrices ready:", X_tr.shape, X_te.shape)


AssertionError: Missing train. (Restore first if you are in a new session.)

In [7]:
# ==== Refit promoted XGB head with L/14 features (prints fold SMAPE) ====
import time, numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import StratifiedKFold

def smape(y_true, y_pred):
    d = (np.abs(y_true)+np.abs(y_pred))/2.0
    out = np.zeros_like(d, dtype=float); m = d!=0
    out[m] = np.abs(y_true[m]-y_pred[m])/d[m]
    return out.mean()*100

y     = train['price'].astype(float).values
y_log = np.log(np.clip(y, 0.01, None))
bins  = pd.qcut(y, q=20, duplicates='drop').astype(str)

params = dict(
    n_estimators=4000, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', device='cuda', random_state=42, early_stopping_rounds=200,
    eval_metric='mae', objective='reg:absoluteerror',
    max_depth=8, min_child_weight=2, learning_rate=0.03,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_l = np.zeros_like(y_log, dtype='float32')
te_l  = np.zeros(X_te.shape[0], dtype='float32')

t0 = time.time()
for f,(tr,va) in enumerate(skf.split(X_tr, bins)):
    t_fold = time.time()
    m = xgb.XGBRegressor(**params)
    m.fit(X_tr[tr], y_log[tr], eval_set=[(X_tr[va], y_log[va])], verbose=False)
    oof_l[va] = m.predict(X_tr[va])
    te_l     += m.predict(X_te) / skf.n_splits
    print(f"[fold {f}] SMAPE={smape(y[va], np.exp(oof_l[va]).clip(0.01)):.3f}% | fold_time={time.time()-t_fold:.1f}s")

cv = smape(y, np.exp(oof_l).clip(0.01))
print(f"[L14-refit] CV SMAPE: {cv:.3f}% | total={(time.time()-t0)/60:.1f}m")

import os
os.makedirs("out", exist_ok=True)
pd.DataFrame({'sample_id': train['sample_id'], 'oof_log': oof_l,
              'oof_price': np.exp(oof_l).clip(0.01)}).to_csv("out/oof_xgb_l14_refit.csv", index=False)
pd.DataFrame({'sample_id': test['sample_id'],
              'price': np.exp(te_l).clip(0.01).astype(float)}).to_csv("out/test_predictions_xgb_l14_refit.csv", index=False)
print("Saved -> out/oof_xgb_l14_refit.csv")
print("Saved -> out/test_predictions_xgb_l14_refit.csv")


NameError: name 'train' is not defined